# Reproducible Hybrid SR Experiments
**Purpose:** load canonical dataset → build groups by correlation → run importance-guided hybrid SR sweep → evaluate vs baselines → save results.

In [12]:
# Cell 1 — imports and base paths
import os, json, time, math, pathlib
from pathlib import Path
import numpy as np
import pandas as pd

# plotting (optional)
import matplotlib.pyplot as plt

# main output root for all dataset results
ROOT = Path("artifacts") / "hybrid_experiments_summary"
ROOT.mkdir(parents=True, exist_ok=True)

# make sure notebooks helpers (your earlier functions) are in scope:
# - load_canonical_Xy(ds_dir)
# - fit_one_sr_on_subset(Xtr, ytr, Xte, subset_idx, ...)  # your SR wrapper
# - eval_baselines(Xtr, Xte, ytr, yte)
# - sample_one_subset(groups, importances, alpha, pick_one_per_group=True)
# If any are not defined already in this fresh notebook, paste their definitions into a helper cell (see below).
print("ROOT:", ROOT)

ROOT: artifacts/hybrid_experiments_summary


In [13]:
# Hard reset of dataset variables for clean reload
for var in ["Xtr","Xte","ytr","yte",
            "Xtr_local","Xte_local","ytr_local","yte_local"]:
    if var in globals():
        del globals()[var]

print("Cleared old dataset variables.")

Cleared old dataset variables.


In [14]:
# === CELL 2: Robust canonical dataset loader ===
import pandas as pd
from pathlib import Path

def load_canonical_Xy(ds_dir: Path):
    """
    Load canonical dataset from artifacts/canonical/<dataset>.

    Expected structure:
        X_train.parquet
        X_test.parquet
        y_train.pkl
        y_test.pkl
        meta.json

    Returns:
        Xtr, ytr, meta, used_files
    """

    ds_dir = Path(ds_dir)

    if not ds_dir.exists():
        raise FileNotFoundError(f"{ds_dir} does not exist.")

    used_files = []

    # --- Preferred format: explicit train/test ---
    X_train_path = ds_dir / "X_train.parquet"
    y_train_path = ds_dir / "y_train.pkl"
    X_test_path  = ds_dir / "X_test.parquet"
    y_test_path  = ds_dir / "y_test.pkl"

    if X_train_path.exists() and y_train_path.exists():
        Xtr = pd.read_parquet(X_train_path)
        ytr = pd.read_pickle(y_train_path).squeeze()

        used_files.extend(["X_train.parquet", "y_train.pkl"])

        meta = {
            "n_samples": int(Xtr.shape[0]),
            "n_features": int(Xtr.shape[1]),
        }

        return Xtr, ytr, meta, used_files

    # --- Fallback: generic X/y files ---
    X_path = ds_dir / "X.parquet"
    y_path = ds_dir / "y.pkl"

    if X_path.exists() and y_path.exists():
        Xtr = pd.read_parquet(X_path)
        ytr = pd.read_pickle(y_path).squeeze()

        used_files.extend(["X.parquet", "y.pkl"])

        meta = {
            "n_samples": int(Xtr.shape[0]),
            "n_features": int(Xtr.shape[1]),
        }

        return Xtr, ytr, meta, used_files

    # --- Last fallback: combined file (split last column as y) ---
    for f in ds_dir.iterdir():
        if f.suffix in (".parquet", ".csv", ".dat", ".pkl"):
            try:
                if f.suffix == ".parquet":
                    df = pd.read_parquet(f)
                elif f.suffix == ".pkl":
                    df = pd.read_pickle(f)
                else:
                    df = pd.read_csv(f)

                if isinstance(df, pd.DataFrame) and df.shape[1] >= 2:
                    Xtr = df.iloc[:, :-1]
                    ytr = df.iloc[:, -1]
                    used_files.append(f.name)
                    meta = {
                        "n_samples": int(Xtr.shape[0]),
                        "n_features": int(Xtr.shape[1]),
                    }
                    return Xtr, ytr.values, meta, used_files
            except Exception:
                continue

    raise FileNotFoundError(
        f"Could not locate canonical dataset format in {ds_dir}.\n"
        "Expected X_train.parquet/y_train.pkl or similar."
    )

print("Canonical loader ready.")

Canonical loader ready.


In [15]:
# Diagnostic + safe loader: make sure Xtr_local / Xte_local / ytr_local / yte_local exist
from pathlib import Path
import pandas as pd
import numpy as np

ds_name = "openml_586"   # << change this to the dataset folder you want to load
ds_dir = Path("artifacts") / "canonical" / ds_name

print("Canonical folder:", ds_dir, "exists?", ds_dir.exists())

# show what notebook variables exist now
vars_to_check = ["Xtr","Xte","ytr","yte","Xtr_local","Xte_local","ytr_local","yte_local"]
for v in vars_to_check:
    print(v, "in globals():", v in globals(), " type:",
          type(globals().get(v)).__name__ if v in globals() else None)

# If loader function is not in scope, fail early with helpful message
if "load_canonical_Xy" not in globals():
    raise RuntimeError("load_canonical_Xy(...) is not defined in this notebook. Run the cell that defines it (Cell 2).")

# If canonical variables are missing, call loader
need_load = not ("Xtr_local" in globals() or "Xtr" in globals())
if need_load:
    print("Calling load_canonical_Xy(...) to populate canonical variables for", ds_name)
    try:
        Xtr, ytr, meta, used = load_canonical_Xy(ds_dir)
        # If loader returned X,Y directly (some versions), wrap into names
        # ensure we have Xtr and Xte variables
        try:
            Xtr_local = pd.DataFrame(Xtr)  # coerce
        except Exception:
            Xtr_local = Xtr
        # try to load explicit X_test / y_test if available
        Xte_local = None
        yte_local = None
        try:
            if (ds_dir / "X_test.parquet").exists():
                Xte_local = pd.read_parquet(ds_dir / "X_test.parquet")
            if (ds_dir / "y_test.pkl").exists():
                yte_local = pd.read_pickle(ds_dir / "y_test.pkl").squeeze()
        except Exception as e:
            print("  warning: couldn't load X_test/y_test from canonical folder:", e)

        # If loader didn't return test split, try to split
        if Xte_local is None or yte_local is None:
            from sklearn.model_selection import train_test_split
            Xtr_local, Xte_local, ytr_arr, yte_arr = train_test_split(
                Xtr_local, ytr, test_size=0.2, random_state=42
            )
            ytr_local = np.asarray(pd.to_numeric(ytr_arr, errors="coerce")).ravel()
            yte_local = np.asarray(pd.to_numeric(yte_arr, errors="coerce")).ravel()
        else:
            ytr_local = np.asarray(pd.to_numeric(ytr, errors="coerce")).ravel()
            yte_local = np.asarray(pd.to_numeric(yte_local, errors="coerce")).ravel()

        # final canonical names (replace/overwrite)
        Xtr_local = pd.DataFrame(Xtr_local).reset_index(drop=True)
        Xte_local = pd.DataFrame(Xte_local).reset_index(drop=True)
        print("Loaded canonical dataset:", ds_name)
    except FileNotFoundError as e:
        raise FileNotFoundError(f"Could not load dataset folder {ds_dir!s}. Check that artifacts/canonical/{ds_name} exists and contains X_train.parquet etc.\nOriginal error: {e}")

else:
    # use the existing variables (coerce into canonical names)
    print("Canonical variables already present; coercing into standard names.")
    if "Xtr_local" in globals():
        Xtr_local = pd.DataFrame(globals()["Xtr_local"])
    elif "Xtr" in globals():
        Xtr_local = pd.DataFrame(globals()["Xtr"])
    else:
        raise RuntimeError("No Xtr found after check.")

    if "Xte_local" in globals():
        Xte_local = pd.DataFrame(globals()["Xte_local"])
    elif "Xte" in globals():
        Xte_local = pd.DataFrame(globals()["Xte"])
    else:
        # try to load a test split from canonical folder
        if (ds_dir / "X_test.parquet").exists():
            Xte_local = pd.read_parquet(ds_dir / "X_test.parquet")
        else:
            # not present: create a train/test split
            from sklearn.model_selection import train_test_split
            Xtr_local, Xte_local, ytr_arr, yte_arr = train_test_split(
                Xtr_local, globals().get("ytr", globals().get("ytr_local")), test_size=0.2, random_state=42
            )
            ytr_local = np.asarray(pd.to_numeric(ytr_arr, errors="coerce")).ravel()
            yte_local = np.asarray(pd.to_numeric(yte_arr, errors="coerce")).ravel()

    # coerce y names
    if "ytr_local" in globals():
        ytr_local = np.asarray(pd.to_numeric(globals()["ytr_local"], errors="coerce")).ravel()
    elif "ytr" in globals():
        ytr_local = np.asarray(pd.to_numeric(globals()["ytr"], errors="coerce")).ravel()
    else:
        # attempt to read canonical y_train
        if (ds_dir / "y_train.pkl").exists():
            ytr_local = np.asarray(pd.to_numeric(pd.read_pickle(ds_dir/"y_train.pkl").squeeze(), errors="coerce")).ravel()

# Final quick sanity prints
print("Xtr_local shape:", Xtr_local.shape, "Xte_local shape:", Xte_local.shape)
print("ytr_local shape:", getattr(ytr_local, "shape", None), "yte_local shape:", getattr(yte_local, "shape", None))
print("Sample X columns:", list(Xtr_local.columns[:10]))

Canonical folder: artifacts/canonical/openml_586 exists? True
Xtr in globals(): False  type: None
Xte in globals(): False  type: None
ytr in globals(): False  type: None
yte in globals(): False  type: None
Xtr_local in globals(): False  type: None
Xte_local in globals(): False  type: None
ytr_local in globals(): False  type: None
yte_local in globals(): False  type: None
Calling load_canonical_Xy(...) to populate canonical variables for openml_586
Loaded canonical dataset: openml_586
Xtr_local shape: (800, 25) Xte_local shape: (200, 25)
ytr_local shape: (800,) yte_local shape: (200,)
Sample X columns: ['oz1', 'oz2', 'oz3', 'oz4', 'oz5', 'oz6', 'oz7', 'oz8', 'oz9', 'oz10']


In [16]:
# === CELL 3: Impute + Coerce numeric types (defensive) ===
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer

# Expect dataset loader to have set: Xtr, Xte, ytr, yte or returned as Xtr_local etc
# We'll create canonical notebook locals: Xtr_local, Xte_local, ytr_local, yte_local
try:
    Xtr  # check
except NameError:
    # If loader returned Xtr_local already, coerce names
    Xtr = globals().get("Xtr_local", globals().get("Xtr_local", None))

# Use the variables returned by your loader (adapt names if needed)
# prefer Xtr/Xte/ytr/yte returned by load_canonical_Xy
if "Xtr" not in globals() and "Xtr_local" in globals():
    Xtr = Xtr_local
if "Xte" not in globals() and "Xte_local" in globals():
    Xte = Xte_local
if "ytr" not in globals() and "ytr_local" in globals():
    ytr = ytr_local
if "yte" not in globals() and "yte_local" in globals():
    yte = yte_local

# make copies and impute if necessary
imp = SimpleImputer(strategy="median")
if (Xtr.isna().sum().sum() > 0) or (Xte.isna().sum().sum() > 0):
    Xtr_imp = pd.DataFrame(imp.fit_transform(Xtr), columns=Xtr.columns, index=Xtr.index)
    Xte_imp = pd.DataFrame(imp.transform(Xte), columns=Xte.columns, index=Xte.index)
else:
    Xtr_imp = Xtr.copy()
    Xte_imp = Xte.copy()

# coerce numeric columns defensively
for df in (Xtr_imp, Xte_imp):
    for c in df.columns:
        if not pd.api.types.is_numeric_dtype(df[c]):
            df[c] = pd.to_numeric(df[c], errors="coerce")

# final fill of any remaining NaNs (should be minimal)
Xtr_imp = Xtr_imp.fillna(0.0)
Xte_imp = Xte_imp.fillna(0.0)

# numeric y coercion
ytr_arr = np.asarray(pd.to_numeric(ytr, errors="coerce")).ravel()
yte_arr = np.asarray(pd.to_numeric(yte, errors="coerce")).ravel()

# notebook-local canonical names used by later cells
Xtr_local, Xte_local, ytr_local, yte_local = Xtr_imp, Xte_imp, ytr_arr, yte_arr

print("Pre-impute NaNs:", Xtr.isna().sum().sum(), Xte.isna().sum().sum())
print("After impute:", Xtr_local.shape, Xte_local.shape)

Pre-impute NaNs: 0 0
After impute: (800, 25) (200, 25)


In [17]:
# === CELL 4: Build groups & importances (safe fallback using RandomForest) ===
import numpy as np
from sklearn.ensemble import RandomForestRegressor

# infer number of features
n_feat = Xtr_local.shape[1]
print("Building groups/importances for dataset with", n_feat, "features ->", list(range(n_feat)))

# fit small RF fallback to get an importance ranking (safe, fast)
rf = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1)
rf.fit(Xtr_local, ytr_local)
imp = rf.feature_importances_

# groups -- default one-feature-per-group (you can replace with domain groups later)
groups_list = [[i] for i in range(n_feat)]

# importance array in column order (float)
imp_arr = np.array(imp, dtype=float)

# defensive length check (pad/trim)
if imp_arr.shape[0] != n_feat:
    tmp = np.zeros(n_feat, dtype=float)
    tmp[:min(len(imp_arr), n_feat)] = imp_arr[:min(len(imp_arr), n_feat)]
    imp_arr = tmp

print("Prepared imp_arr shape:", imp_arr.shape)
print("Prepared groups_list length:", len(groups_list))
print("Sample groups (first 6):", groups_list[:6])

# show top-k importances for sanity
order = np.argsort(-imp_arr)
print("Top 8 importances (index -> score):")
for i in order[:8]:
    print(i, Xtr_local.columns[i], imp_arr[i])

Building groups/importances for dataset with 25 features -> [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]
Prepared imp_arr shape: (25,)
Prepared groups_list length: 25
Sample groups (first 6): [[0], [1], [2], [3], [4], [5]]
Top 8 importances (index -> score):
1 oz2 0.3511124311378813
0 oz1 0.25689273058857925
3 oz4 0.18265058353862476
4 oz5 0.1473180736366364
2 oz3 0.03310101310004447
17 oz18 0.0033836705170254768
18 oz19 0.0017945214405673416
16 oz17 0.0016765568069636583


In [18]:
# ====== Helper functions cell ======
# CELL: sampling + SR-fit helpers (robust / matching original hybrid notebook)
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

def _to_imp_arr(importances, n_features=None, feature_names=None):
    """Normalize many possible importance representations to 1D numpy array in column order."""
    if importances is None:
        return np.zeros(int(n_features) if n_features is not None else 0, dtype=float)

    if isinstance(importances, dict):
        if feature_names is not None:
            arr = np.zeros(len(feature_names), dtype=float)
            for k, v in importances.items():
                try:
                    idx = int(k)
                except Exception:
                    if str(k) in feature_names:
                        idx = feature_names.index(str(k))
                    else:
                        continue
                if 0 <= idx < len(arr):
                    arr[idx] = float(v)
            if n_features is not None and len(arr) != n_features:
                tmp = np.zeros(int(n_features), dtype=float)
                tmp[:min(len(arr), n_features)] = arr[:min(len(arr), n_features)]
                return tmp
            return arr
        else:
            try:
                arr_dict = {int(k): float(v) for k, v in importances.items()}
                maxidx = max(arr_dict.keys()) if arr_dict else -1
                arr = np.zeros(maxidx + 1, dtype=float)
                for k, v in arr_dict.items(): arr[k] = v
                if n_features is not None and arr.shape[0] != n_features:
                    tmp = np.zeros(int(n_features), dtype=float)
                    tmp[:min(len(arr), n_features)] = arr[:min(len(arr), n_features)]
                    return tmp
                return arr
            except Exception:
                vals = list(importances.values())
                return np.array([float(v) for v in vals], dtype=float)

    try:
        arr = np.asarray(importances, dtype=float)
        if n_features is not None and arr.shape[0] != n_features:
            tmp = np.zeros(int(n_features), dtype=float)
            tmp[:min(len(arr), n_features)] = arr[:min(len(arr), n_features)]
            return tmp
        return arr
    except Exception:
        return np.zeros(int(n_features) if n_features is not None else 0, dtype=float)


def sample_one_subset(groups, importances, alpha=1.0, pick_one_per_group=True, rng=None):
    """
    Sample a subset of feature indices guided by importances.
      - groups: list-of-lists of feature indices (or flat list of ints -> one-per-group)
      - importances: dict or array-like mapping features -> score OR list/array of scores in column order.
      - alpha: power to emphasize importance (p_i prop to (imp_i+eps)**alpha)
      - pick_one_per_group: if True pick exactly one index from each non-empty group.
      - rng: optional numpy.random.Generator or seed (int).
    Returns: list of selected feature indices (ints)
    """
    # RNG normalization
    if rng is None:
        rng = np.random.default_rng()
    elif isinstance(rng, (int, np.integer)):
        rng = np.random.default_rng(int(rng))
    elif isinstance(rng, np.random.Generator):
        pass
    else:
        rng = np.random.default_rng()

    # coerce groups -> list-of-lists
    if isinstance(groups, dict):
        groups_list = [list(v) for k, v in groups.items()]
    elif isinstance(groups, (list, tuple)):
        if len(groups) == 0:
            groups_list = []
        else:
            first = groups[0]
            if isinstance(first, (list, tuple, np.ndarray)):
                groups_list = [list(g) for g in groups]
            else:
                groups_list = [[int(g)] for g in groups]
    else:
        raise TypeError("Unsupported groups type for sample_one_subset")

    # determine n_features (if possible)
    n_feat = None
    if isinstance(importances, (list, tuple, np.ndarray)):
        n_feat = len(importances)
    else:
        try:
            n_feat = max([max(g) for g in groups_list if g]) + 1
        except Exception:
            n_feat = None

    imp_arr = _to_imp_arr(importances, n_features=n_feat)
    eps = 1e-12
    probs = (imp_arr + eps) ** float(alpha)
    if probs.sum() == 0:
        probs = np.ones_like(probs, dtype=float)

    chosen = []
    if pick_one_per_group:
        for group in groups_list:
            if not group: continue
            valid = [int(i) for i in group if 0 <= int(i) < len(probs)]
            if len(valid) == 0: continue
            subp = probs[valid].astype(float)
            if subp.sum() <= 0:
                subp = np.ones_like(subp, dtype=float)
            p = subp / subp.sum()
            pick = int(rng.choice(valid, p=p))
            chosen.append(pick)
    else:
        all_idxs = sorted({int(i) for grp in groups_list for i in grp})
        if len(all_idxs) == 0:
            return []
        subp = probs[all_idxs].astype(float)
        if subp.sum() <= 0:
            subp = np.ones_like(subp, dtype=float)
        p = subp / subp.sum()
        pick = int(rng.choice(all_idxs, p=p))
        chosen = [pick]

    return chosen


def fit_one_sr_on_subset(Xtr, ytr, Xte, subset_idx,
                         function_set=('add','sub','mul','div'),
                         parsimony=0.01,
                         generations=50,
                         population_size=500,
                         random_state=None,
                         verbose=0):
    """
    Fit one SymbolicRegressor on the chosen subset of columns.
    Returns: (estimator, train_rmse, test_rmse)
    - Defensive about Xtr/Xte being DataFrame or ndarray.
    """
    try:
        from gplearn.genetic import SymbolicRegressor
    except Exception as e:
        raise RuntimeError("gplearn.SymbolicRegressor not available. Install gplearn or provide an alternative.") from e

    # prepare subset arrays
    if isinstance(Xtr, pd.DataFrame):
        Xtr_sub = Xtr.iloc[:, subset_idx].values
    else:
        Xtr_sub = np.asarray(Xtr)[:, subset_idx]

    if isinstance(Xte, pd.DataFrame):
        Xte_sub = Xte.iloc[:, subset_idx].values
    else:
        Xte_sub = np.asarray(Xte)[:, subset_idx]

    est = SymbolicRegressor(
        function_set=list(function_set),
        generations=int(generations),
        population_size=int(population_size),
        parsimony_coefficient=float(parsimony),
        random_state=int(random_state) if random_state is not None else None,
        n_jobs=1,
        verbose=verbose
    )

    est.fit(Xtr_sub, np.ravel(ytr))
    ytr_pred = est.predict(Xtr_sub)
    yte_pred = est.predict(Xte_sub)

    def _rmse(a,b):
        return float(np.sqrt(mean_squared_error(a, b)))

    tr_rmse = _rmse(ytr, ytr_pred)
    te_rmse = _rmse(yte, yte_pred)

    return est, tr_rmse, te_rmse

def compute_rae(y_true, y_pred):
    """
    Compute Relative Absolute Error (raw) = sum(|y - yhat|) / sum(|y - y_mean|).
    Also return a flipped score (1 - raw) so higher is better.
    Defensive: if denominator is zero, return raw=inf and flipped= -inf (handled as nan by caller).
    """
    import numpy as _np
    y_true = _np.asarray(y_true).ravel()
    y_pred = _np.asarray(y_pred).ravel()
    if y_true.shape[0] == 0:
        return float("nan"), float("nan")
    denom = _np.sum(_np.abs(y_true - _np.mean(y_true)))
    num = _np.sum(_np.abs(y_true - y_pred))
    if denom == 0:
        rae_raw = float("inf") if num != 0 else 0.0
    else:
        rae_raw = float(num) / float(denom)
    # flipped so higher is better (paper wording uses "higher RAE indicates higher quality")
    rae_score = None
    try:
        rae_score = 1.0 - rae_raw
    except Exception:
        rae_score = float("nan")
    return rae_raw, rae_score


def eval_baselines(Xtr, Xte, ytr, yte):
    """
    Evaluate baseline models on (Xtr,ytr) -> (Xte,yte).
    Returns dict with RMSE and RAE (raw and flipped 'score') for each model:
      {"LR_RMSE":.., "LR_RAE":.., "LR_RAE_score":.., ...}
    Falls back to sklearn's GradientBoostingRegressor if xgboost not available.
    """
    import numpy as np
    from sklearn.metrics import mean_squared_error
    from sklearn.linear_model import LinearRegression, Ridge

    # xgboost fallback handling
    use_xgb = True
    try:
        from xgboost import XGBRegressor
    except Exception:
        from sklearn.ensemble import GradientBoostingRegressor as XGBRegressor  # fallback
        use_xgb = False

    # Models (stable defaults)
    lr = LinearRegression()
    ridge = Ridge(alpha=1.0, random_state=42)
    xgb = XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        random_state=42,
        n_jobs=-1,
        verbosity=0 if use_xgb else 0
    )

    # Coerce inputs to numeric numpy arrays
    Xtr_a = np.asarray(Xtr)
    Xte_a = np.asarray(Xte)
    ytr_a = np.asarray(ytr).ravel()
    yte_a = np.asarray(yte).ravel()

    # Basic sanity checks
    if Xtr_a.ndim != 2 or Xte_a.ndim != 2:
        raise ValueError("Xtr/Xte must be 2D array-like")
    if ytr_a.ndim != 1 or yte_a.ndim != 1:
        raise ValueError("ytr/yte must be 1D array-like")
    if Xtr_a.shape[0] != ytr_a.shape[0]:
        raise ValueError(f"Train rows mismatch: Xtr {Xtr_a.shape[0]} vs ytr {ytr_a.shape[0]}")
    if Xte_a.shape[0] != yte_a.shape[0]:
        raise ValueError(f"Test rows mismatch: Xte {Xte_a.shape[0]} vs yte {yte_a.shape[0]}")

    def fit_and_metrics(model, Xtr_loc, ytr_loc, Xte_loc, yte_loc):
        model.fit(Xtr_loc, ytr_loc)
        preds = np.asarray(model.predict(Xte_loc)).ravel()
        if preds.shape[0] != yte_loc.shape[0]:
            raise RuntimeError("Prediction length does not match yte length")
        rmse = float(np.sqrt(mean_squared_error(yte_loc, preds)))
        rae_raw, rae_score = compute_rae(yte_loc, preds)
        return rmse, rae_raw, rae_score

    results = {}
    lr_rmse, lr_rae_raw, lr_rae_score = fit_and_metrics(lr, Xtr_a, ytr_a, Xte_a, yte_a)
    results["LR_RMSE"] = lr_rmse
    results["LR_RAE"] = lr_rae_raw
    results["LR_RAE_score"] = lr_rae_score

    ridge_rmse, ridge_rae_raw, ridge_rae_score = fit_and_metrics(ridge, Xtr_a, ytr_a, Xte_a, yte_a)
    results["Ridge_RMSE"] = ridge_rmse
    results["Ridge_RAE"] = ridge_rae_raw
    results["Ridge_RAE_score"] = ridge_rae_score

    try:
        xgb_rmse, xgb_rae_raw, xgb_rae_score = fit_and_metrics(xgb, Xtr_a, ytr_a, Xte_a, yte_a)
        results["XGB_RMSE"] = xgb_rmse
        results["XGB_RAE"] = xgb_rae_raw
        results["XGB_RAE_score"] = xgb_rae_score
    except Exception:
        # keep RMSE as NaN like before but also include RAE fields as NaN
        results["XGB_RMSE"] = float("nan")
        results["XGB_RAE"] = float("nan")
        results["XGB_RAE_score"] = float("nan")

    return results
    
print("Helper functions defined: sample_one_subset, fit_one_sr_on_subset, eval_baselines")

Helper functions defined: sample_one_subset, fit_one_sr_on_subset, eval_baselines


In [19]:
# CELL: make_correlated_groups helper (fixed to match original hybrid semantics)
import numpy as np
import pandas as pd

# clustering
try:
    from sklearn.cluster import AgglomerativeClustering
except Exception:
    AgglomerativeClustering = None

from sklearn.ensemble import RandomForestRegressor

def make_correlated_groups(X, importances=None, n_groups=None, threshold=None,
                           prefer_names=False, random_state=42, y_for_imp=None):
    """
    Build correlated feature groups (list-of-lists of feature indices) using
    agglomerative clustering on distance = 1 - |corr|, and return an importance
    array aligned to column order.

    Behavior:
    - If `importances` is provided (dict or array-like) it is normalized and used.
    - Else, if `y_for_imp` is provided (or a common global target exists), prefer
      a supervised importance:
        * try XGBoost feature_importances_ if xgboost is installed,
        * else RandomForestRegressor fit on (X, y_for_imp).
    - Else fall back to variance-based proxy (normalized variance).
    - Agglomerative clustering uses precomputed distance `1 - |corr|` and
      returns list-of-lists of integer column indices.
    Returns: (groups_list, imp_arr)
    """
    # coerce to DataFrame for consistent column names & correlation
    if not isinstance(X, pd.DataFrame):
        X_df = pd.DataFrame(X)
    else:
        X_df = X.copy()
    n_feat = X_df.shape[1]
    colnames = list(X_df.columns)

    # Resolve supervised target if not explicitly passed
    if y_for_imp is None:
        # try common global names as best-effort (do NOT fail if absent)
        # Avoid using `or` with numpy arrays (truthiness ambiguous).
        # Pick first available global target variable if present.
        for _k in ("ytr_local", "ytr", "y_train"):
            _val = globals().get(_k, None)
            if _val is not None:
                y_for_imp = _val
                break
        else:
            y_for_imp = None

    # --- build importance array (length n_feat) ---
    def _as_array_and_pad(arr_like):
        arr = np.asarray(arr_like, dtype=float)
        if arr.shape[0] != n_feat:
            tmp = np.zeros(n_feat, dtype=float)
            tmp[:min(len(arr), n_feat)] = arr[:min(len(arr), n_feat)]
            return tmp
        return arr

    if importances is None:
        # Attempt supervised importance if we have a target
        imp_arr = None
        if y_for_imp is not None:
            # try XGBoost first (closer to original notebook), else RF
            try:
                from xgboost import XGBRegressor
                xgb = XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.05,
                                   subsample=0.9, colsample_bytree=0.9,
                                   random_state=int(random_state), n_jobs=-1, verbosity=0)
                xgb.fit(X_df, np.asarray(y_for_imp).ravel())
                imp_arr = np.asarray(xgb.feature_importances_, dtype=float)
            except Exception:
                try:
                    rf = RandomForestRegressor(n_estimators=200, max_depth=6,
                                               random_state=int(random_state), n_jobs=-1)
                    rf.fit(X_df, np.asarray(y_for_imp).ravel())
                    imp_arr = np.asarray(rf.feature_importances_, dtype=float)
                except Exception:
                    imp_arr = None

        # If supervised importance not available, use variance proxy
        if imp_arr is None:
            var = X_df.var(axis=0).values.astype(float)
            if np.all(var == 0) or np.isnan(var).all():
                imp_arr = np.ones(n_feat, dtype=float) / float(n_feat)
            else:
                imp_arr = var / var.sum()
    else:
        # importances provided: accept dict(name->score), dict(idx->score), or array-like
        if isinstance(importances, dict):
            imp_arr = np.zeros(n_feat, dtype=float)
            for k, v in importances.items():
                try:
                    idx = int(k)
                except Exception:
                    if k in colnames:
                        idx = colnames.index(k)
                    else:
                        continue
                if 0 <= idx < n_feat:
                    imp_arr[idx] = float(v)
        else:
            imp_arr = _as_array_and_pad(importances)

    # Defensive normalization & NaN/inf handling
    imp_arr = np.nan_to_num(imp_arr, nan=0.0, posinf=0.0, neginf=0.0)
    if imp_arr.sum() <= 0:
        imp_arr = np.ones(n_feat, dtype=float) / float(n_feat)
    else:
        imp_arr = imp_arr.astype(float)
        imp_arr = imp_arr / imp_arr.sum()

    # --- grouping by (1 - |corr|) distance and agglomerative clustering ---
    if n_feat <= 1:
        return [[i] for i in range(n_feat)], imp_arr

    # compute absolute correlation matrix, fill NaNs with zeros
    corr = X_df.corr().abs().fillna(0.0)
    dist = 1.0 - corr.values
    # choose number of clusters if not provided (sqrt heuristic)
    if n_groups is None:
        n_groups = int(max(2, min(n_feat, round(np.sqrt(n_feat)))))

    if AgglomerativeClustering is None:
        groups_list = [[i] for i in range(n_feat)]
    else:
        # sklearn older versions expect affinity='precomputed', newer want metric='precomputed'
        try:
            cl = AgglomerativeClustering(n_clusters=n_groups, affinity='precomputed', linkage='average')
        except TypeError:
            cl = AgglomerativeClustering(n_clusters=n_groups, metric='precomputed', linkage='average')
        labels = cl.fit_predict(dist)
        groups_list = []
        for lab in range(int(labels.max()) + 1):
            members = [int(i) for i, val in enumerate(labels) if val == lab]
            if members:
                groups_list.append(members)

    # final defensive: ensure every index included
    all_idx = set(sum(groups_list, [])) if groups_list else set()
    missing = [i for i in range(n_feat) if i not in all_idx]
    for i in missing:
        groups_list.append([i])

    # Sort groups for reproducibility: original hybrid used size-based sorting for determinism.
    groups_list = sorted(groups_list, key=len)

    # Print short summary (helpful during experiments)
    print(f"make_correlated_groups: n_features={n_feat}, n_groups_requested={n_groups}, produced_groups={len(groups_list)}")
    print("Sample groups (first 6):", groups_list[:6])
    topk = np.argsort(-imp_arr)[:min(10, n_feat)]
    print("Top features by imp (idx -> name -> score):")
    for idx in topk:
        name = colnames[idx] if prefer_names else str(idx)
        print(f" {idx} {name} {imp_arr[idx]:.4f}")

    return groups_list, imp_arr

# Alias for convenience
def make_groups_from_X(X, n_groups=None, **kwargs):
    return make_correlated_groups(X, n_groups=n_groups, **kwargs)

print("Grouping helper defined. Example usage:")
print("  groups_list, imp_arr = make_groups_from_X(Xtr_local, n_groups=6)")

Grouping helper defined. Example usage:
  groups_list, imp_arr = make_groups_from_X(Xtr_local, n_groups=6)


## California Housing Testing

In [59]:
# California Housing
# prepare canonical/california_housing (fixed: drop any target column in X)
from pathlib import Path
import json, pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

ROOT_CANON = Path("artifacts") / "canonical"
ROOT_CANON.mkdir(parents=True, exist_ok=True)

ds_name = "california_housing"
out_dir = ROOT_CANON / ds_name
if out_dir.exists():
    # remove old broken files so we recreate cleanly
    for p in out_dir.iterdir():
        p.unlink()
else:
    out_dir.mkdir(parents=True)

print("Downloading California Housing dataset...")
data = fetch_california_housing(as_frame=True)

# Use data.data (features only). If data.frame was used, ensure target column is dropped.
X = data.data.copy()
y = data.target.copy()

# defensive: if target-like column present in X, drop it
if "MedHouseVal" in X.columns:
    print("Dropping 'MedHouseVal' column from X (was present).")
    X = X.drop(columns=["MedHouseVal"])

# deterministic split (80/20)
RND = 42
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=RND)

# Standardize ONLY X
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test),  columns=X_test.columns)

# write canonical files
X_train_scaled.to_parquet(out_dir / "X_train.parquet", index=False)
X_test_scaled.to_parquet(out_dir / "X_test.parquet", index=False)
pd.to_pickle(y_train.reset_index(drop=True), out_dir / "y_train.pkl")
pd.to_pickle(y_test.reset_index(drop=True),  out_dir / "y_test.pkl")

meta = {
    "n_samples_train": int(X_train.shape[0]),
    "n_samples_test": int(X_test.shape[0]),
    "n_features": int(X_train.shape[1]),
    "feature_names": list(X_train.columns)
}
with open(out_dir / "meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Saved canonical dataset to:", out_dir)
print("Train shape:", X_train_scaled.shape, "Test shape:", X_test_scaled.shape)

Saved canonical dataset to: artifacts/canonical/california_housing
Train shape: (16512, 8) Test shape: (4128, 8)


In [60]:
import pandas as pd
Xtr = pd.read_parquet("artifacts/canonical/california_housing/X_train.parquet")
ytr = pd.read_pickle("artifacts/canonical/california_housing/y_train.pkl")
print("X columns:", Xtr.columns.tolist())
print("X shape:", Xtr.shape)
print("y stats:", pd.Series(ytr).describe())

X columns: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
X shape: (16512, 8)
y stats: count    16512.000000
mean         2.071947
std          1.156226
min          0.149990
25%          1.198000
50%          1.798500
75%          2.651250
max          5.000010
Name: MedHouseVal, dtype: float64


In [ ]:
# CELL: run_hybrid_for_one_dataset (california_housing test) - NEW CONFIG + RAE
import time, json, shutil, traceback
from pathlib import Path
import numpy as np
import pandas as pd

# USER CONFIG (change per dataset)
ds_name = "california_housing"   # << changed to California Housing canonical folder name
N_RUNS = 40                       # full sweep
ALPHA = 2.0                      # importance exponent
BASE_SEED = 42
ROOT = Path("artifacts") / "hybrid_experiments_summary"
ROOT.mkdir(parents=True, exist_ok=True)

# degenerate threshold: reject SR features with tiny train variance
VAR_THRESH = 1e-3

print("Hybrid run config:", {"dataset": ds_name, "N_RUNS": N_RUNS, "ALPHA": ALPHA, "BASE_SEED": BASE_SEED, "VAR_THRESH": VAR_THRESH})

# output guard (skip if already evaluated)
eval_json = ROOT / f"{ds_name}_evaluation_summary.json"
eval_csv = ROOT / f"{ds_name}_evaluation_summary.csv"
if eval_json.exists() or eval_csv.exists():
    print(f"Skipping {ds_name}: evaluation already exists at {eval_json} / {eval_csv}")
else:
    start_ts = time.time()

    # --- load canonical dataset (uses your load_canonical_Xy) ---
    ds_dir = Path("artifacts") / "canonical" / ds_name
    if "load_canonical_Xy" not in globals():
        raise RuntimeError("load_canonical_Xy(...) not found — define it before running this cell.")

    X_all, y_all, meta, used_files = load_canonical_Xy(ds_dir)
    # try explicit test split if present
    Xte = None; yte = None
    try:
        if (ds_dir / "X_test.parquet").exists():
            Xte = pd.read_parquet(ds_dir / "X_test.parquet")
        if (ds_dir / "y_test.pkl").exists():
            yte = pd.read_pickle(ds_dir / "y_test.pkl").squeeze()
    except Exception as e:
        print("Warning loading explicit test split (ignored):", e)

    # if no explicit test, do deterministic 80/20 split
    if Xte is None or yte is None:
        from sklearn.model_selection import train_test_split
        Xtr_df, Xte_df, ytr_ser, yte_ser = train_test_split(
            pd.DataFrame(X_all), y_all, test_size=0.2, random_state=42
        )
        ytr = np.asarray(pd.to_numeric(ytr_ser, errors="coerce")).ravel()
        yte = np.asarray(pd.to_numeric(yte_ser, errors="coerce")).ravel()
    else:
        Xtr_df = pd.DataFrame(X_all)
        Xte_df = pd.DataFrame(Xte)
        ytr = np.asarray(pd.to_numeric(y_all, errors="coerce")).ravel()
        yte = np.asarray(pd.to_numeric(yte, errors="coerce")).ravel()

    # canonical local copies
    Xtr_local = Xtr_df.reset_index(drop=True)
    Xte_local = Xte_df.reset_index(drop=True)
    ytr_local = ytr
    yte_local = yte

    print("Loaded dataset:", ds_name, "| train:", Xtr_local.shape, "| test:", Xte_local.shape, "| used files:", used_files)

    # --- impute + coerce numeric defensively ---
    from sklearn.impute import SimpleImputer
    imp = SimpleImputer(strategy="median")
    if (Xtr_local.isna().sum().sum() > 0) or (Xte_local.isna().sum().sum() > 0):
        Xtr_imp = pd.DataFrame(imp.fit_transform(Xtr_local), columns=Xtr_local.columns, index=Xtr_local.index)
        Xte_imp = pd.DataFrame(imp.transform(Xte_local), columns=Xte_local.columns, index=Xte_local.index)
    else:
        Xtr_imp = Xtr_local.copy(); Xte_imp = Xte_local.copy()

    # coerce non-numeric to numeric (defensive)
    for df in (Xtr_imp, Xte_imp):
        for c in df.columns:
            if not pd.api.types.is_numeric_dtype(df[c]):
                df[c] = pd.to_numeric(df[c], errors="coerce")
    Xtr_imp = Xtr_imp.fillna(0.0); Xte_imp = Xte_imp.fillna(0.0)

    ytr_arr = np.asarray(pd.to_numeric(ytr_local, errors="coerce")).ravel()
    yte_arr = np.asarray(pd.to_numeric(yte_local, errors="coerce")).ravel()

    Xtr_local = Xtr_imp.reset_index(drop=True)
    Xte_local = Xte_imp.reset_index(drop=True)
    ytr_local = ytr_arr
    yte_local = yte_arr

    print("After impute shapes:", Xtr_local.shape, Xte_local.shape)

    # --- build groups & importances using make_correlated_groups (robust handling) ---
    n_feat = Xtr_local.shape[1]
    groups_list = None
    imp_arr = None

    def _coerce_groups_to_index_lists(groups_in, Xdf):
        if groups_in is None:
            return None
        out = []
        if isinstance(groups_in, dict):
            vals = list(groups_in.values())
        else:
            vals = list(groups_in)
        for g in vals:
            if isinstance(g, (list, tuple, np.ndarray)):
                idxs = []
                for el in g:
                    if isinstance(el, str):
                        try:
                            idxs.append(int(list(Xdf.columns).index(el)))
                        except Exception:
                            raise ValueError(f"Cannot map group element '{el}' -> not a column name")
                    else:
                        idxs.append(int(el))
                out.append(list(idxs))
            else:
                if isinstance(g, str):
                    idx = int(list(Xdf.columns).index(g))
                    out.append([idx])
                else:
                    out.append([int(g)])
        return out

    if "make_correlated_groups" in globals():
        try:
            mg_out = make_correlated_groups(Xtr_local, n_groups=None, random_state=BASE_SEED)
            groups_raw = None
            imp_from_groups = None

            if mg_out is None:
                groups_raw = None
                imp_from_groups = None
            elif isinstance(mg_out, tuple) or isinstance(mg_out, list):
                if len(mg_out) == 0:
                    groups_raw = None
                elif len(mg_out) == 1:
                    first = mg_out[0]
                    if isinstance(first, (list, tuple, np.ndarray)):
                        groups_raw = first
                    else:
                        try:
                            arr = np.asarray(first, dtype=float)
                            if arr.shape[0] == n_feat:
                                imp_from_groups = arr
                            else:
                                groups_raw = first
                        except Exception:
                            groups_raw = first
                else:
                    groups_raw = mg_out[0]
                    cand_imp = mg_out[1]
                    try:
                        arr = np.asarray(cand_imp, dtype=float)
                        if arr.shape[0] >= 1:
                            imp_from_groups = arr
                    except Exception:
                        imp_from_groups = None
            elif isinstance(mg_out, dict):
                if "groups" in mg_out and "importance" in mg_out:
                    groups_raw = mg_out["groups"]
                    imp_from_groups = np.asarray(mg_out["importance"], dtype=float)
                else:
                    groups_raw = list(mg_out.values())
            elif isinstance(mg_out, np.ndarray):
                imp_from_groups = np.asarray(mg_out, dtype=float)
            else:
                groups_raw = mg_out

            if groups_raw is not None:
                try:
                    groups_list = _coerce_groups_to_index_lists(groups_raw, Xtr_local)
                    groups_list = [[int(j) for j in g if 0 <= int(j) < n_feat] for g in groups_list if len(g) > 0]
                    if len(groups_list) == 0:
                        groups_list = None
                    else:
                        print(f"Correlated groups produced: {len(groups_list)} groups")
                except Exception as e:
                    print("Could not coerce correlated groups (exception):", e)
                    groups_list = None

            if imp_from_groups is not None:
                try:
                    tmp_imp = np.asarray(imp_from_groups, dtype=float)
                    if tmp_imp.shape[0] == n_feat:
                        imp_arr = tmp_imp.copy()
                        print("Using importance array returned by make_correlated_groups.")
                    else:
                        imp_arr = tmp_imp
                        print("Imported importance array (length mismatch will be padded/trimmed).")
                except Exception:
                    imp_arr = None

        except Exception as e:
            print("make_correlated_groups failed (will fallback to RF one-feature-per-group). Exception:")
            traceback.print_exc()
            groups_list = None
    else:
        print("make_correlated_groups not found; will fallback to RF/one-feature groups.")

    # retry with explicit heuristic if needed
    if groups_list is None and "make_correlated_groups" in globals():
        try:
            print("Retrying make_correlated_groups with explicit n_groups heuristic...")
            mg_retry = make_correlated_groups(Xtr_local, n_groups=max(2, int(round(np.sqrt(n_feat)))), random_state=BASE_SEED)
            if isinstance(mg_retry, tuple) and len(mg_retry) >= 1:
                groups_raw = mg_retry[0]
                imp_from_groups = mg_retry[1] if len(mg_retry) > 1 else None
            else:
                groups_raw = mg_retry
                imp_from_groups = None
            if groups_raw is not None:
                groups_list = _coerce_groups_to_index_lists(groups_raw, Xtr_local)
                groups_list = [[int(j) for j in g if 0 <= int(j) < n_feat] for g in groups_list if len(g) > 0]
                if len(groups_list) == 0:
                    groups_list = None
                else:
                    print(f"Retry succeeded: {len(groups_list)} groups")
            if imp_from_groups is not None and imp_arr is None:
                try:
                    tmp = np.asarray(imp_from_groups, dtype=float)
                    imp_arr = tmp
                except Exception:
                    pass
        except Exception as e:
            print("Retry attempt failed (ignored):", e)

    # compute RF importances as fallback / canonical importance
    from sklearn.ensemble import RandomForestRegressor
    rf = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=BASE_SEED, n_jobs=-1)
    try:
        rf.fit(Xtr_local, ytr_local)
        rf_imp = np.asarray(rf.feature_importances_, dtype=float)
    except Exception as e:
        print("RF importance failed:", e)
        rf_imp = np.zeros(n_feat, dtype=float)

    if groups_list is None:
        groups_list = [[i] for i in range(n_feat)]
        print("Fallback groups: one-feature-per-group (count = {})".format(len(groups_list)))

    # finalize imp_arr (prefer helper-provided, else RF)
    if imp_arr is None:
        imp_arr = rf_imp.copy()
    else:
        tmp = np.zeros(n_feat, dtype=float)
        tmp[:min(len(imp_arr), n_feat)] = imp_arr[:min(len(imp_arr), n_feat)]
        imp_arr = tmp

    # defensive pad/trim and normalization
    if imp_arr.shape[0] != n_feat:
        tmp = np.zeros(n_feat, dtype=float)
        tmp[:min(len(imp_arr), n_feat)] = imp_arr[:min(len(imp_arr), n_feat)]
        imp_arr = tmp
    imp_arr = np.nan_to_num(imp_arr, nan=0.0, posinf=0.0, neginf=0.0)
    if imp_arr.sum() <= 0:
        imp_arr = np.ones(n_feat, dtype=float) / float(n_feat)
    else:
        imp_arr = imp_arr.astype(float) / float(np.sum(imp_arr))

    print("Prepared groups_list length:", len(groups_list), "imp_arr shape:", imp_arr.shape)
    order = np.argsort(-imp_arr)[:min(8, n_feat)]
    print("Top importances (idx -> name -> score):")
    for i in order:
        print(i, Xtr_local.columns[i], f"{imp_arr[i]:.4f}")

    # --- ensure required helpers exist ---
    for helper in ("sample_one_subset", "fit_one_sr_on_subset", "eval_baselines"):
        if helper not in globals():
            raise RuntimeError(f"Helper '{helper}' missing — paste its definition before running this cell.")

    # --- SWEEP: importance-guided sampling + SR training (NEW CONFIG) ---
    best_te = float("inf")
    best_est = None
    best_subset = None
    rows = []

    print("Starting hybrid sweep: N_RUNS=", N_RUNS)
    for i in range(1, N_RUNS+1):
        run_seed = int(BASE_SEED + i)
        try:
            subset_idx = sample_one_subset(groups_list, imp_arr, alpha=ALPHA, pick_one_per_group=True,
                                           rng=np.random.default_rng(run_seed))
        except TypeError:
            subset_idx = sample_one_subset(groups_list, imp_arr, alpha=ALPHA, pick_one_per_group=True)

        # defensive: ensure subset_idx is list of ints
        try:
            subset_idx = [int(x) for x in list(subset_idx)]
        except Exception:
            subset_idx = list(subset_idx) if hasattr(subset_idx, "__iter__") else []

        # skip invalid empty subset
        if len(subset_idx) == 0:
            print(f"[{i}] sampled empty subset — skipping")
            rows.append({"run": i, "seed": run_seed, "subset_idx": subset_idx, "tr_rmse": None, "te_rmse": None, "expr": None, "degenerate": True})
            continue

        try:
            # NEW SR config: expanded function_set (trig), larger population, more generations
            est, tr_rmse, te_rmse = fit_one_sr_on_subset(
                Xtr_local, ytr_local, Xte_local, subset_idx,
                function_set=('add','sub','mul','div','sin','cos'),
                parsimony=0.001,
                generations=100,
                population_size=1500,
                random_state=run_seed,
                verbose=0
            )
        except Exception as e:
            print(f"[{i}] SR training failed: {e}")
            rows.append({"run": i, "seed": run_seed, "subset_idx": subset_idx, "tr_rmse": None, "te_rmse": None, "expr": None, "error": str(e), "degenerate": True})
            continue

        expr = None
        try:
            if hasattr(est, "program_"):
                expr = str(est.program_)
            elif hasattr(est, "program"):
                expr = str(est.program)
            else:
                expr = repr(est)
        except Exception:
            expr = repr(est)

        # compute SR feature predictions & check degeneracy (tiny train variance)
        try:
            # select subset columns for predict (works for df or ndarray)
            try:
                Xtr_sub = Xtr_local.iloc[:, subset_idx]
                Xte_sub = Xte_local.iloc[:, subset_idx]
            except Exception:
                Xtr_sub = Xtr_local.loc[:, subset_idx]
                Xte_sub = Xte_local.loc[:, subset_idx]

            sr_tr_feature = est.predict(Xtr_sub)
            sr_te_feature = est.predict(Xte_sub)
            std_tr = float(np.nanstd(sr_tr_feature))
            degenerate = (not np.isfinite(std_tr)) or (std_tr < VAR_THRESH)
            # compute SR RAE if helper available
            sr_tr_rae = sr_tr_rae_score = sr_te_rae = sr_te_rae_score = None
            if "compute_rae" in globals():
                try:
                    sr_tr_rae, sr_tr_rae_score = compute_rae(ytr_local, sr_tr_feature)
                    sr_te_rae, sr_te_rae_score = compute_rae(yte_local, sr_te_feature)
                except Exception:
                    sr_tr_rae = sr_tr_rae_score = sr_te_rae = sr_te_rae_score = None

            if degenerate:
                print(f"[{i}] Degenerate SR run (std_tr={std_tr:.6g}) — skipping for best-model selection. expr={expr}")
            else:
                # only consider for best if non-degenerate and te_rmse valid
                if te_rmse is not None and te_rmse < best_te:
                    best_te = te_rmse
                    best_est = est
                    best_subset = subset_idx

            rows.append({"run": i, "seed": run_seed, "subset_idx": subset_idx,
                         "tr_rmse": float(tr_rmse) if tr_rmse is not None else None,
                         "te_rmse": float(te_rmse) if te_rmse is not None else None,
                         "expr": expr, "degenerate": bool(degenerate), "sr_tr_std": std_tr,
                         "sr_tr_rae": sr_tr_rae, "sr_tr_rae_score": sr_tr_rae_score,
                         "sr_te_rae": sr_te_rae, "sr_te_rae_score": sr_te_rae_score})
        except Exception as e:
            print(f"[{i}] Prediction or degeneracy check failed: {e}")
            rows.append({"run": i, "seed": run_seed, "subset_idx": subset_idx, "tr_rmse": float(tr_rmse) if tr_rmse is not None else None,
                         "te_rmse": float(te_rmse) if te_rmse is not None else None, "expr": expr, "degenerate": True, "error": str(e)})
            continue

        if (i % 10) == 0:
            print(f"[{i}/{N_RUNS}] checkpoint best_te: {best_te:.6f}")

    # Save raw sweep rows
    outdir = ROOT
    outdir.mkdir(parents=True, exist_ok=True)
    partial_csv = outdir / f"{ds_name}_sweep_partial.csv"
    pd.DataFrame(rows).to_csv(partial_csv, index=False)
    print("Saved partial sweep results to:", partial_csv)

    # --- finalize: if best found, augment and evaluate baselines (with RAE) ---
    summary = {
        "dataset": ds_name,
        "n_samples_train": int(Xtr_local.shape[0]),
        "n_features": int(Xtr_local.shape[1]),
        "best_subset": best_subset,
        "best_te": float(best_te) if best_te != float("inf") else None,
        "timestamp": time.time(),
    }

    if best_est is not None and best_subset is not None:
        try:
            try:
                Xtr_sub = Xtr_local.iloc[:, best_subset]
                Xte_sub = Xte_local.iloc[:, best_subset]
            except Exception:
                Xtr_sub = Xtr_local.loc[:, best_subset]
                Xte_sub = Xte_local.loc[:, best_subset]

            sr_tr_feature = best_est.predict(Xtr_sub)
            sr_te_feature = best_est.predict(Xte_sub)

            # final degenerate guard (do not augment if degenerate)
            if np.nanstd(sr_tr_feature) < VAR_THRESH or not np.isfinite(np.nanstd(sr_tr_feature)):
                print("Warning: Best SR feature has near-zero variance on train — marking as degenerate; summary saved but augmentation skipped.")
                summary.update({"note": "best_SR_feature_degenerate"})
            else:
                Xtr_aug = Xtr_local.copy(); Xte_aug = Xte_local.copy()
                Xtr_aug["SR_1"] = sr_tr_feature
                Xte_aug["SR_1"] = sr_te_feature

                # eval baselines returns RMSE + RAE fields (if helpers updated)
                baseline_metrics = eval_baselines(Xtr_local, Xte_local, ytr_local, yte_local)
                augmented_metrics = eval_baselines(Xtr_aug, Xte_aug, ytr_local, yte_local)

                # dRMSE (difference augmented - baseline) for all RMSE keys
                dRMSE = {k: augmented_metrics[k] - baseline_metrics[k] for k in baseline_metrics.keys() if k.endswith("_RMSE")}
                # dRAE_raw and dRAE_score (aug - base) for RAE keys
                dRAE_raw = {k: augmented_metrics[k] - baseline_metrics[k] for k in baseline_metrics.keys() if k.endswith("_RAE")}
                dRAE_score = {k: augmented_metrics[k] - baseline_metrics[k] for k in baseline_metrics.keys() if k.endswith("_RAE_score")}

                summary.update({
                    "baseline": baseline_metrics,
                    "augmented": augmented_metrics,
                    "dRMSE": dRMSE,
                    "dRAE_raw": dRAE_raw,
                    "dRAE_score": dRAE_score
                })

                print("Baseline:", baseline_metrics)
                print("Augmented:", augmented_metrics)
                print("ΔRMSE:", dRMSE)
                print("ΔRAE_raw:", dRAE_raw)
                print("ΔRAE_score:", dRAE_score)

            # persist summary
            with open(eval_json, "w") as f:
                json.dump(summary, f, indent=2)
            pd.DataFrame([summary]).to_csv(eval_csv, index=False)

            print("Saved evaluation summary JSON:", eval_json)
            print("Saved evaluation summary CSV:", eval_csv)

        except Exception as e:
            print("Error during final augmentation/evaluation:", e)
            traceback.print_exc()
    else:
        print("No successful non-degenerate SR model found during sweep for", ds_name)
        # still save summary skeleton
        with open(eval_json, "w") as f:
            json.dump(summary, f, indent=2)
        pd.DataFrame([summary]).to_csv(eval_csv, index=False)

    print("Elapsed (s):", time.time() - start_ts)

Hybrid run config: {'dataset': 'california_housing', 'N_RUNS': 40, 'ALPHA': 2.0, 'BASE_SEED': 42, 'VAR_THRESH': 0.001}
Loaded dataset: california_housing | train: (16512, 8) | test: (4128, 8) | used files: ['X_train.parquet', 'y_train.pkl']
After impute shapes: (16512, 8) (4128, 8)
make_correlated_groups: n_features=8, n_groups_requested=3, produced_groups=3
Sample groups (first 6): [[5], [6, 7], [0, 1, 2, 3, 4]]
Top features by imp (idx -> name -> score):
 0 0 0.4869
 5 5 0.1476
 2 2 0.1007
 7 7 0.0783
 6 6 0.0728
 1 1 0.0688
 3 3 0.0323
 4 4 0.0126
Correlated groups produced: 3 groups
Using importance array returned by make_correlated_groups.


/home/rshayan1/.local/lib/python3.12/site-packages/sklearn/cluster/_agglomerative.py:983: FutureWarning: Attribute `affinity` was deprecated in version 1.2 and will be removed in 1.4. Use `metric` instead
  warnings.warn(


Prepared groups_list length: 3 imp_arr shape: (8,)
Top importances (idx -> name -> score):
0 MedInc 0.4869
5 AveOccup 0.1476
2 AveRooms 0.1007
7 Longitude 0.0783
6 Latitude 0.0728
1 HouseAge 0.0688
3 AveBedrms 0.0323
4 Population 0.0126
Starting hybrid sweep: N_RUNS= 40


In [63]:
# Small check: print best expr and (if possible) recreate SR feature stats
import pandas as pd, numpy as np, json
from pathlib import Path

ROOT = Path("artifacts") / "hybrid_experiments_summary"
ds = "california_housing"
partial = ROOT / f"{ds}_sweep_partial.csv"
eval_json = ROOT / f"{ds}_evaluation_summary.json"

if not partial.exists():
    raise FileNotFoundError(partial)

rows = pd.read_csv(partial)
# pick best by minimum te_rmse (ignore NaNs)
rows_valid = rows[rows["te_rmse"].notna()]
if len(rows_valid) == 0:
    print("No successful runs recorded in partial CSV.")
else:
    best_row = rows_valid.loc[rows_valid["te_rmse"].astype(float).idxmin()]
    print("Best row (from partial csv):")
    print(best_row[["run","seed","subset_idx","tr_rmse","te_rmse","expr"]].to_dict())
    # print saved final summary if present
    if eval_json.exists():
        print("\nEvaluation summary (final):")
        print(json.loads(open(eval_json).read()))
    # try to re-create SR predictions if helpers and data are available
    try:
        # requires these to exist in the notebook: load_canonical_Xy, fit_one_sr_on_subset
        from pathlib import Path
        ds_dir = Path("artifacts") / "canonical" / ds
        X_all, y_all, meta, used = load_canonical_Xy(ds_dir)
        # load the explicit test split if present, else do the same 80/20 split
        import pandas as pd
        if (ds_dir / "X_test.parquet").exists():
            Xtr = pd.read_parquet(ds_dir / "X_train.parquet")
            Xte = pd.read_parquet(ds_dir / "X_test.parquet")
            ytr = pd.read_pickle(ds_dir / "y_train.pkl").squeeze()
            yte = pd.read_pickle(ds_dir / "y_test.pkl").squeeze()
        else:
            from sklearn.model_selection import train_test_split
            Xtr, Xte, ytr, yte = train_test_split(pd.DataFrame(X_all), y_all, test_size=0.2, random_state=42)
        # parse subset_idx
        subset = best_row["subset_idx"]
        # subset may be saved as a string like "[0, 2]" -> eval safely
        if isinstance(subset, str):
            try:
                subset_idx = eval(subset)
            except Exception:
                # fallback: try to parse numbers
                subset_idx = [int(x) for x in subset.strip("[]").split(",") if x.strip()!='']
        else:
            subset_idx = list(subset)
        # re-run SR fit with same seed to get predictions
        seed = int(best_row["seed"]) if not np.isnan(best_row["seed"]) else 42
        print("\nRe-fitting SR on the best subset to get feature stats (may take a short while)...")
        est, tr_rmse, te_rmse = fit_one_sr_on_subset(
            pd.DataFrame(Xtr).reset_index(drop=True), 
            np.asarray(ytr).ravel(),
            pd.DataFrame(Xte).reset_index(drop=True),
            subset_idx,
            function_set=('add','sub','mul','div'),
            parsimony=0.005, generations=50, population_size=1000, random_state=seed, verbose=0
        )
        # predictions
        try:
            sr_tr = est.predict(pd.DataFrame(Xtr).iloc[:, subset_idx])
            sr_te = est.predict(pd.DataFrame(Xte).iloc[:, subset_idx])
        except Exception:
            sr_tr = est.predict(pd.DataFrame(Xtr)[Xtr.columns[subset_idx]])
            sr_te = est.predict(pd.DataFrame(Xte)[Xte.columns[subset_idx]])
        print("\nSR feature (train) stats:")
        print(pd.Series(sr_tr).describe())
        print("\nSR feature (test) stats:")
        print(pd.Series(sr_te).describe())
        print("\nRecreated expression (est object):")
        if hasattr(est, "program_"):
            print(est.program_)
        elif hasattr(est, "program"):
            print(est.program)
        else:
            print(repr(est))
    except Exception as e:
        print("Could not recreate SR estimator/predictions here (missing helper or data).")
        print("But best expression string (from file) is shown above.")
        print("Error:", e)

Best row (from partial csv):
{'run': 31, 'seed': 73, 'subset_idx': '[5, 6, 0]', 'tr_rmse': 0.779032781163286, 'te_rmse': 0.7771745738689393, 'expr': "SymbolicRegressor(function_set=['add', 'sub', 'mul', 'div', 'sin', 'cos'],\n                  generations=100, population_size=1500, random_state=73)"}

Evaluation summary (final):
{'dataset': 'california_housing', 'n_samples_train': 16512, 'n_features': 8, 'best_subset': [5, 6, 0], 'best_te': 0.7771745738689393, 'timestamp': 1773808797.0417545, 'baseline': {'LR_RMSE': 0.7455813830127763, 'LR_RAE': 0.5906596232557056, 'LR_RAE_score': 0.40934037674429435, 'Ridge_RMSE': 0.7455567442814779, 'Ridge_RAE': 0.5906518568183589, 'Ridge_RAE_score': 0.40934814318164114, 'XGB_RMSE': 0.4926498991162629, 'XGB_RAE': 0.3692162776562004, 'XGB_RAE_score': 0.6307837223437995}, 'augmented': {'LR_RMSE': 0.6992729947400603, 'LR_RAE': 0.5549584257828756, 'LR_RAE_score': 0.4450415742171244, 'Ridge_RMSE': 0.6992665908360037, 'Ridge_RAE': 0.5549546354062348, 'Ridg

## Boston Housing Testing

In [10]:
# PREPARE canonical/boston_housing (correct + controlled standardization)

from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

ROOT_CANON = Path("artifacts") / "canonical"
ROOT_CANON.mkdir(parents=True, exist_ok=True)

ds_name = "boston_housing"
out_dir = ROOT_CANON / ds_name

# reset folder
if out_dir.exists():
    for p in out_dir.iterdir():
        try:
            p.unlink()
        except Exception:
            pass
else:
    out_dir.mkdir(parents=True, exist_ok=True)

# === LOAD DATA ===
data_path = Path("/home/rshayan1/Datasets/hou_all.csv")
print("Using dataset:", data_path)

df = pd.read_csv(data_path)

# clean column names
df.columns = [str(c).strip().lower() for c in df.columns]

# drop unnamed / empty columns
df = df.loc[:, [c for c in df.columns if c and not c.startswith("unnamed")]]

print("Columns:", list(df.columns))

# === TARGET ===
if "medv" not in df.columns:
    raise RuntimeError(f"Expected 'medv' column as target. Found columns: {list(df.columns)}")

y = df["medv"].copy()

# === DROP NON-FEATURE / EXTRA COLUMNS ===
drop_cols = [c for c in ["medv"] if c in df.columns]
X = df.drop(columns=drop_cols)

# === DEFINE CONTINUOUS FEATURES (ALL FEATURES GET STANDARDIZED HERE) ===
continuous_cols = list(X.columns)

# === SPLIT ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# === STANDARDIZE ALL FEATURES ===
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

if len(continuous_cols) > 0:
    X_train_scaled[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
    X_test_scaled[continuous_cols] = scaler.transform(X_test[continuous_cols])

# === SAVE CANONICAL FILES ===
X_train_scaled.reset_index(drop=True).to_parquet(out_dir / "X_train.parquet", index=False)
X_test_scaled.reset_index(drop=True).to_parquet(out_dir / "X_test.parquet", index=False)

pd.to_pickle(y_train.reset_index(drop=True), out_dir / "y_train.pkl")
pd.to_pickle(y_test.reset_index(drop=True), out_dir / "y_test.pkl")

# meta
meta = {
    "n_samples_train": int(X_train.shape[0]),
    "n_samples_test": int(X_test.shape[0]),
    "n_features": int(X_train.shape[1]),
    "feature_names": list(X_train.columns)
}
with open(out_dir / "meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Saved canonical dataset to:", out_dir)
print("Train shape:", X_train_scaled.shape, "Test shape:", X_test_scaled.shape)

Using dataset: /home/rshayan1/Datasets/hou_all.csv
Columns: ['crim', 'zn', 'indus', 'chas', 'nox', 'rm', 'age', 'dis', 'rad', 'tax', 'ptratio', 'black', 'lstat', 'medv']
Saved canonical dataset to: artifacts/canonical/boston_housing
Train shape: (404, 13) Test shape: (102, 13)


In [11]:
# CELL: run_hybrid_for_one_dataset (boston_housing - NEW CONFIG)
import time, json, traceback
from pathlib import Path
import numpy as np
import pandas as pd

# === USER CONFIG ===
ds_name = "boston_housing"
N_RUNS = 40
ALPHA = 2.0
BASE_SEED = 42
VAR_THRESH = 1e-3
ROOT = Path("artifacts") / "hybrid_experiments_summary"
ROOT.mkdir(parents=True, exist_ok=True)

print("Hybrid run config:", {
    "dataset": ds_name,
    "N_RUNS": N_RUNS,
    "ALPHA": ALPHA,
    "BASE_SEED": BASE_SEED,
    "VAR_THRESH": VAR_THRESH
})

# output guard
eval_json = ROOT / f"{ds_name}_evaluation_summary.json"
eval_csv = ROOT / f"{ds_name}_evaluation_summary.csv"
if eval_json.exists() or eval_csv.exists():
    print(f"Skipping {ds_name}: evaluation already exists at {eval_json} / {eval_csv}")
else:
    start_ts = time.time()

    # --- load canonical dataset ---
    ds_dir = Path("artifacts") / "canonical" / ds_name
    if "load_canonical_Xy" not in globals():
        raise RuntimeError("load_canonical_Xy(...) not found — define it before running this cell.")

    X_all, y_all, meta, used_files = load_canonical_Xy(ds_dir)

    # try explicit test split if present
    Xte = None
    yte = None
    try:
        if (ds_dir / "X_test.parquet").exists():
            Xte = pd.read_parquet(ds_dir / "X_test.parquet")
        if (ds_dir / "y_test.pkl").exists():
            yte = pd.read_pickle(ds_dir / "y_test.pkl").squeeze()
    except Exception as e:
        print("Warning loading explicit test split (ignored):", e)

    # if no explicit test, do deterministic 80/20 split
    if Xte is None or yte is None:
        from sklearn.model_selection import train_test_split
        Xtr_df, Xte_df, ytr_ser, yte_ser = train_test_split(
            pd.DataFrame(X_all), y_all, test_size=0.2, random_state=42
        )
        ytr = np.asarray(pd.to_numeric(ytr_ser, errors="coerce")).ravel()
        yte = np.asarray(pd.to_numeric(yte_ser, errors="coerce")).ravel()
    else:
        Xtr_df = pd.DataFrame(X_all)
        Xte_df = pd.DataFrame(Xte)
        ytr = np.asarray(pd.to_numeric(y_all, errors="coerce")).ravel()
        yte = np.asarray(pd.to_numeric(yte, errors="coerce")).ravel()

    Xtr_local = Xtr_df.reset_index(drop=True)
    Xte_local = Xte_df.reset_index(drop=True)
    ytr_local = ytr
    yte_local = yte

    print("Loaded dataset:", ds_name, "| train:", Xtr_local.shape, "| test:", Xte_local.shape, "| used files:", used_files)

    # --- impute + coerce numeric defensively ---
    from sklearn.impute import SimpleImputer
    imp = SimpleImputer(strategy="median")
    if (Xtr_local.isna().sum().sum() > 0) or (Xte_local.isna().sum().sum() > 0):
        Xtr_imp = pd.DataFrame(imp.fit_transform(Xtr_local), columns=Xtr_local.columns, index=Xtr_local.index)
        Xte_imp = pd.DataFrame(imp.transform(Xte_local), columns=Xte_local.columns, index=Xte_local.index)
    else:
        Xtr_imp = Xtr_local.copy()
        Xte_imp = Xte_local.copy()

    for df in (Xtr_imp, Xte_imp):
        for c in df.columns:
            if not pd.api.types.is_numeric_dtype(df[c]):
                df[c] = pd.to_numeric(df[c], errors="coerce")
    Xtr_imp = Xtr_imp.fillna(0.0)
    Xte_imp = Xte_imp.fillna(0.0)

    Xtr_local = Xtr_imp.reset_index(drop=True)
    Xte_local = Xte_imp.reset_index(drop=True)
    ytr_local = np.asarray(pd.to_numeric(ytr_local, errors="coerce")).ravel()
    yte_local = np.asarray(pd.to_numeric(yte_local, errors="coerce")).ravel()

    print("After impute shapes:", Xtr_local.shape, Xte_local.shape)

    # --- build groups & importances ---
    from sklearn.ensemble import RandomForestRegressor
    n_feat = Xtr_local.shape[1]
    groups_list = None
    imp_arr = None

    def _coerce_groups_to_index_lists(groups_in, Xdf):
        if groups_in is None:
            return None
        out = []
        if isinstance(groups_in, dict):
            vals = list(groups_in.values())
        else:
            vals = list(groups_in)
        for g in vals:
            if isinstance(g, (list, tuple, np.ndarray)):
                idxs = []
                for el in g:
                    if isinstance(el, str):
                        try:
                            idxs.append(int(list(Xdf.columns).index(el)))
                        except Exception:
                            raise ValueError(f"Cannot map group element '{el}' -> not a column name")
                    else:
                        idxs.append(int(el))
                out.append(list(idxs))
            else:
                if isinstance(g, str):
                    idx = int(list(Xdf.columns).index(g))
                    out.append([idx])
                else:
                    out.append([int(g)])
        return out

    if "make_correlated_groups" in globals():
        try:
            mg_out = make_correlated_groups(Xtr_local, n_groups=None, random_state=BASE_SEED)
            groups_raw = None
            imp_from_groups = None

            if mg_out is None:
                groups_raw = None
            elif isinstance(mg_out, (tuple, list)):
                if len(mg_out) == 0:
                    groups_raw = None
                elif len(mg_out) == 1:
                    first = mg_out[0]
                    if isinstance(first, (list, tuple, np.ndarray)):
                        groups_raw = first
                    else:
                        try:
                            arr = np.asarray(first, dtype=float)
                            if arr.shape[0] == n_feat:
                                imp_from_groups = arr
                            else:
                                groups_raw = first
                        except Exception:
                            groups_raw = first
                else:
                    groups_raw = mg_out[0]
                    try:
                        imp_from_groups = np.asarray(mg_out[1], dtype=float)
                    except Exception:
                        imp_from_groups = None
            elif isinstance(mg_out, dict):
                if "groups" in mg_out and "importance" in mg_out:
                    groups_raw = mg_out["groups"]
                    imp_from_groups = np.asarray(mg_out["importance"], dtype=float)
                else:
                    groups_raw = list(mg_out.values())
            elif isinstance(mg_out, np.ndarray):
                imp_from_groups = np.asarray(mg_out, dtype=float)
            else:
                groups_raw = mg_out

            if groups_raw is not None:
                try:
                    groups_list = _coerce_groups_to_index_lists(groups_raw, Xtr_local)
                    groups_list = [[int(j) for j in g if 0 <= int(j) < n_feat] for g in groups_list if len(g) > 0]
                    if len(groups_list) == 0:
                        groups_list = None
                    else:
                        print(f"Correlated groups produced: {len(groups_list)} groups")
                except Exception as e:
                    print("Could not coerce correlated groups (exception):", e)
                    groups_list = None

            if imp_from_groups is not None:
                try:
                    tmp_imp = np.asarray(imp_from_groups, dtype=float)
                    if tmp_imp.shape[0] == n_feat:
                        imp_arr = tmp_imp.copy()
                        print("Using importance array returned by make_correlated_groups.")
                    else:
                        imp_arr = tmp_imp
                        print("Imported importance array (length mismatch will be padded/trimmed).")
                except Exception:
                    imp_arr = None

        except Exception as e:
            print("make_correlated_groups failed (will fallback to RF one-feature-per-group). Exception:")
            traceback.print_exc()
            groups_list = None
    else:
        print("make_correlated_groups not found; will fallback to RF/one-feature groups.")

    if groups_list is None and "make_correlated_groups" in globals():
        try:
            print("Retrying make_correlated_groups with explicit n_groups heuristic...")
            mg_retry = make_correlated_groups(
                Xtr_local,
                n_groups=max(2, int(round(np.sqrt(n_feat)))),
                random_state=BASE_SEED
            )
            if isinstance(mg_retry, (tuple, list)) and len(mg_retry) >= 1:
                groups_raw = mg_retry[0]
                imp_from_groups = mg_retry[1] if len(mg_retry) > 1 else None
            else:
                groups_raw = mg_retry
                imp_from_groups = None

            if groups_raw is not None:
                groups_list = _coerce_groups_to_index_lists(groups_raw, Xtr_local)
                groups_list = [[int(j) for j in g if 0 <= int(j) < n_feat] for g in groups_list if len(g) > 0]
                if len(groups_list) == 0:
                    groups_list = None
                else:
                    print(f"Retry succeeded: {len(groups_list)} groups")

            if imp_from_groups is not None and imp_arr is None:
                try:
                    imp_arr = np.asarray(imp_from_groups, dtype=float)
                except Exception:
                    pass
        except Exception as e:
            print("Retry attempt failed (ignored):", e)

    rf = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=BASE_SEED, n_jobs=-1)
    try:
        rf.fit(Xtr_local, ytr_local)
        rf_imp = np.asarray(rf.feature_importances_, dtype=float)
    except Exception as e:
        print("RF importance failed:", e)
        rf_imp = np.zeros(n_feat, dtype=float)

    if groups_list is None:
        groups_list = [[i] for i in range(n_feat)]
        print("Fallback groups: one-feature-per-group (count = {})".format(len(groups_list)))

    if imp_arr is None:
        imp_arr = rf_imp.copy()
    else:
        tmp = np.zeros(n_feat, dtype=float)
        tmp[:min(len(imp_arr), n_feat)] = imp_arr[:min(len(imp_arr), n_feat)]
        imp_arr = tmp

    if imp_arr.shape[0] != n_feat:
        tmp = np.zeros(n_feat, dtype=float)
        tmp[:min(len(imp_arr), n_feat)] = imp_arr[:min(len(imp_arr), n_feat)]
        imp_arr = tmp

    imp_arr = np.nan_to_num(imp_arr, nan=0.0, posinf=0.0, neginf=0.0)
    if imp_arr.sum() <= 0:
        imp_arr = np.ones(n_feat, dtype=float) / float(n_feat)
    else:
        imp_arr = imp_arr.astype(float) / float(np.sum(imp_arr))

    print("Prepared groups_list length:", len(groups_list), "imp_arr shape:", imp_arr.shape)
    order = np.argsort(-imp_arr)[:min(8, n_feat)]
    print("Top importances (idx -> name -> score):")
    for i in order:
        print(i, Xtr_local.columns[i], f"{imp_arr[i]:.4f}")

    # --- ensure required helpers exist ---
    for helper in ("sample_one_subset", "fit_one_sr_on_subset", "eval_baselines"):
        if helper not in globals():
            raise RuntimeError(f"Helper '{helper}' missing — paste its definition before running this cell.")

    def get_program_str(est):
        for attr in ("program", "program_", "_program"):
            prog = getattr(est, attr, None)
            if prog is not None:
                try:
                    return str(prog)
                except Exception:
                    return repr(prog)
        return None

    # --- SWEEP ---
    best_te = float("inf")
    best_est = None
    best_subset = None
    rows = []

    print("Starting hybrid sweep: N_RUNS=", N_RUNS)
    for i in range(1, N_RUNS + 1):
        run_seed = int(BASE_SEED + i)

        try:
            subset_idx = sample_one_subset(
                groups_list,
                imp_arr,
                alpha=ALPHA,
                pick_one_per_group=True,
                rng=np.random.default_rng(run_seed)
            )
        except TypeError:
            subset_idx = sample_one_subset(
                groups_list,
                imp_arr,
                alpha=ALPHA,
                pick_one_per_group=True
            )

        try:
            subset_idx = [int(x) for x in list(subset_idx)]
        except Exception:
            subset_idx = list(subset_idx) if hasattr(subset_idx, "__iter__") else []

        if len(subset_idx) == 0:
            print(f"[{i}] sampled empty subset — skipping")
            rows.append({
                "run": i,
                "seed": run_seed,
                "subset_idx": subset_idx,
                "tr_rmse": None,
                "te_rmse": None,
                "expr": None,
                "status": "empty_subset"
            })
            continue

        try:
            est, tr_rmse, te_rmse = fit_one_sr_on_subset(
                Xtr_local, ytr_local, Xte_local, subset_idx,
                function_set=('add', 'sub', 'mul', 'div', 'sin', 'cos'),
                parsimony=0.001,
                generations=100,
                population_size=1500,
                random_state=run_seed,
                verbose=0
            )
        except Exception as e:
            print(f"[{i}] SR training failed: {e}")
            rows.append({
                "run": i,
                "seed": run_seed,
                "subset_idx": subset_idx,
                "tr_rmse": None,
                "te_rmse": None,
                "expr": None,
                "error": str(e)
            })
            continue

        try:
            Xtr_sub = Xtr_local.iloc[:, subset_idx]
            Xte_sub = Xte_local.iloc[:, subset_idx]
        except Exception:
            Xtr_sub = Xtr_local.loc[:, subset_idx]
            Xte_sub = Xte_local.loc[:, subset_idx]

        expr = get_program_str(est)

        rows.append({
            "run": i,
            "seed": run_seed,
            "subset_idx": subset_idx,
            "tr_rmse": float(tr_rmse) if tr_rmse is not None else None,
            "te_rmse": float(te_rmse) if te_rmse is not None else None,
            "expr": expr
        })

        if te_rmse is not None and te_rmse < best_te:
            best_te = te_rmse
            best_est = est
            best_subset = subset_idx

        if (i % 10) == 0:
            print(f"[{i}/{N_RUNS}] checkpoint best_te: {best_te:.6f}")

    partial_csv = ROOT / f"{ds_name}_sweep_partial.csv"
    pd.DataFrame(rows).to_csv(partial_csv, index=False)
    print("Saved partial sweep results to:", partial_csv)

    summary = {
        "dataset": ds_name,
        "n_samples_train": int(Xtr_local.shape[0]),
        "n_features": int(Xtr_local.shape[1]),
        "best_subset": best_subset,
        "best_te": float(best_te) if best_te != float("inf") else None,
        "timestamp": time.time(),
    }

    if best_est is not None and best_subset is not None:
        try:
            try:
                Xtr_sub = Xtr_local.iloc[:, best_subset]
                Xte_sub = Xte_local.iloc[:, best_subset]
            except Exception:
                Xtr_sub = Xtr_local.loc[:, best_subset]
                Xte_sub = Xte_local.loc[:, best_subset]

            sr_tr_feature = best_est.predict(Xtr_sub)
            sr_te_feature = best_est.predict(Xte_sub)

            if np.nanstd(sr_tr_feature) < VAR_THRESH or not np.isfinite(np.nanstd(sr_tr_feature)):
                print("Warning: Best SR feature has near-zero variance on train — marking as degenerate; summary saved but augmentation skipped.")
                summary.update({"note": "best_SR_feature_degenerate"})
            else:
                Xtr_aug = Xtr_local.copy()
                Xte_aug = Xte_local.copy()
                Xtr_aug["SR_1"] = sr_tr_feature
                Xte_aug["SR_1"] = sr_te_feature

                baseline_metrics = eval_baselines(Xtr_local, Xte_local, ytr_local, yte_local)
                augmented_metrics = eval_baselines(Xtr_aug, Xte_aug, ytr_local, yte_local)
                dRMSE = {k.replace("_RMSE", "_dRMSE"): augmented_metrics[k] - baseline_metrics[k] for k in baseline_metrics}

                summary.update({
                    "baseline": baseline_metrics,
                    "augmented": augmented_metrics,
                    "dRMSE": dRMSE
                })

                print("Baseline:", baseline_metrics)
                print("Augmented:", augmented_metrics)
                print("ΔRMSE / ΔRAE:", dRMSE)

            with open(eval_json, "w") as f:
                json.dump(summary, f, indent=2)
            pd.DataFrame([summary]).to_csv(eval_csv, index=False)

            print("Saved evaluation summary JSON:", eval_json)
            print("Saved evaluation summary CSV:", eval_csv)

        except Exception as e:
            print("Error during final augmentation/evaluation:", e)
            traceback.print_exc()
    else:
        print("No successful non-degenerate SR model found during sweep for", ds_name)
        with open(eval_json, "w") as f:
            json.dump(summary, f, indent=2)
        pd.DataFrame([summary]).to_csv(eval_csv, index=False)

    print("Elapsed (s):", time.time() - start_ts)

Hybrid run config: {'dataset': 'boston_housing', 'N_RUNS': 40, 'ALPHA': 2.0, 'BASE_SEED': 42, 'VAR_THRESH': 0.001}
Loaded dataset: boston_housing | train: (404, 13) | test: (102, 13) | used files: ['X_train.parquet', 'y_train.pkl']
After impute shapes: (404, 13) (102, 13)


/home/rshayan1/.local/lib/python3.12/site-packages/sklearn/cluster/_agglomerative.py:983: FutureWarning: Attribute `affinity` was deprecated in version 1.2 and will be removed in 1.4. Use `metric` instead
  warnings.warn(


make_correlated_groups: n_features=13, n_groups_requested=4, produced_groups=4
Sample groups (first 6): [[10], [11], [3], [0, 1, 2, 4, 5, 6, 7, 8, 9, 12]]
Top features by imp (idx -> name -> score):
 12 12 0.3471
 5 5 0.2701
 3 3 0.0927
 4 4 0.0755
 10 10 0.0498
 0 0 0.0397
 7 7 0.0393
 9 9 0.0291
 6 6 0.0162
 11 11 0.0127
Correlated groups produced: 4 groups
Using importance array returned by make_correlated_groups.
Prepared groups_list length: 4 imp_arr shape: (13,)
Top importances (idx -> name -> score):
12 lstat 0.3471
5 rm 0.2701
3 chas 0.0927
4 nox 0.0755
10 ptratio 0.0498
0 crim 0.0397
7 dis 0.0393
9 tax 0.0291
Starting hybrid sweep: N_RUNS= 40
[10/40] checkpoint best_te: 4.634296
[20/40] checkpoint best_te: 4.632681
[30/40] checkpoint best_te: 4.581586
[40/40] checkpoint best_te: 4.581586
Saved partial sweep results to: artifacts/hybrid_experiments_summary/boston_housing_sweep_partial.csv
Baseline: {'LR_RMSE': 4.928602182665336, 'LR_RAE': 0.5373411719575489, 'LR_RAE_score': 0.4

## Wine_red Testing

In [9]:
# CELL: run_hybrid_for_one_dataset (red_wine test) - updated to new config + RAE/degeneracy checks
import time, json, shutil, traceback
from pathlib import Path
import numpy as np
import pandas as pd

# USER CONFIG (change per dataset)
ds_name = "wine_red"             # << canonical folder name for red wine dataset
N_RUNS = 40                      # full sweep use 40; lower for quick smoke
ALPHA = 2.0                      # importance exponent
BASE_SEED = 42
ROOT = Path("artifacts") / "hybrid_experiments_summary"
ROOT.mkdir(parents=True, exist_ok=True)

# degenerate threshold: reject SR features with tiny train variance
VAR_THRESH = 1e-3

print("Hybrid run config:", {"dataset": ds_name, "N_RUNS": N_RUNS, "ALPHA": ALPHA, "BASE_SEED": BASE_SEED, "VAR_THRESH": VAR_THRESH})

# output guard (skip if already evaluated)
eval_json = ROOT / f"{ds_name}_evaluation_summary.json"
eval_csv = ROOT / f"{ds_name}_evaluation_summary.csv"
if eval_json.exists() or eval_csv.exists():
    print(f"Skipping {ds_name}: evaluation already exists at {eval_json} / {eval_csv}")
else:
    start_ts = time.time()

    # --- load canonical dataset (uses your load_canonical_Xy) ---
    ds_dir = Path("artifacts") / "canonical" / ds_name
    if "load_canonical_Xy" not in globals():
        raise RuntimeError("load_canonical_Xy(...) not found — define it before running this cell.")

    X_all, y_all, meta, used_files = load_canonical_Xy(ds_dir)
    # try explicit test split if present
    Xte = None; yte = None
    try:
        if (ds_dir / "X_test.parquet").exists():
            Xte = pd.read_parquet(ds_dir / "X_test.parquet")
        if (ds_dir / "y_test.pkl").exists():
            yte = pd.read_pickle(ds_dir / "y_test.pkl").squeeze()
    except Exception as e:
        print("Warning loading explicit test split (ignored):", e)

    # if no explicit test, do deterministic 80/20 split
    if Xte is None or yte is None:
        from sklearn.model_selection import train_test_split
        Xtr_df, Xte_df, ytr_ser, yte_ser = train_test_split(
            pd.DataFrame(X_all), y_all, test_size=0.2, random_state=42
        )
        ytr = np.asarray(pd.to_numeric(ytr_ser, errors="coerce")).ravel()
        yte = np.asarray(pd.to_numeric(yte_ser, errors="coerce")).ravel()
    else:
        Xtr_df = pd.DataFrame(X_all)
        Xte_df = pd.DataFrame(Xte)
        ytr = np.asarray(pd.to_numeric(y_all, errors="coerce")).ravel()
        yte = np.asarray(pd.to_numeric(yte, errors="coerce")).ravel()

    # canonical local copies
    Xtr_local = Xtr_df.reset_index(drop=True)
    Xte_local = Xte_df.reset_index(drop=True)
    ytr_local = ytr
    yte_local = yte

    print("Loaded dataset:", ds_name, "| train:", Xtr_local.shape, "| test:", Xte_local.shape, "| used files:", used_files)

    # --- impute + coerce numeric defensively ---
    from sklearn.impute import SimpleImputer
    imp = SimpleImputer(strategy="median")
    if (Xtr_local.isna().sum().sum() > 0) or (Xte_local.isna().sum().sum() > 0):
        Xtr_imp = pd.DataFrame(imp.fit_transform(Xtr_local), columns=Xtr_local.columns, index=Xtr_local.index)
        Xte_imp = pd.DataFrame(imp.transform(Xte_local), columns=Xte_local.columns, index=Xte_local.index)
    else:
        Xtr_imp = Xtr_local.copy(); Xte_imp = Xte_local.copy()

    # coerce non-numeric to numeric (defensive)
    for df in (Xtr_imp, Xte_imp):
        for c in df.columns:
            if not pd.api.types.is_numeric_dtype(df[c]):
                df[c] = pd.to_numeric(df[c], errors="coerce")
    Xtr_imp = Xtr_imp.fillna(0.0); Xte_imp = Xte_imp.fillna(0.0)

    ytr_arr = np.asarray(pd.to_numeric(ytr_local, errors="coerce")).ravel()
    yte_arr = np.asarray(pd.to_numeric(yte_local, errors="coerce")).ravel()

    Xtr_local = Xtr_imp.reset_index(drop=True)
    Xte_local = Xte_imp.reset_index(drop=True)
    ytr_local = ytr_arr
    yte_local = yte_arr

    print("After impute shapes:", Xtr_local.shape, Xte_local.shape)

    # --- build groups & importances using make_correlated_groups (robust handling) ---
    n_feat = Xtr_local.shape[1]
    groups_list = None
    imp_arr = None

    def _coerce_groups_to_index_lists(groups_in, Xdf):
        if groups_in is None:
            return None
        out = []
        if isinstance(groups_in, dict):
            vals = list(groups_in.values())
        else:
            vals = list(groups_in)
        for g in vals:
            if isinstance(g, (list, tuple, np.ndarray)):
                idxs = []
                for el in g:
                    if isinstance(el, str):
                        try:
                            idxs.append(int(list(Xdf.columns).index(el)))
                        except Exception:
                            raise ValueError(f"Cannot map group element '{el}' -> not a column name")
                    else:
                        idxs.append(int(el))
                out.append(list(idxs))
            else:
                if isinstance(g, str):
                    idx = int(list(Xdf.columns).index(g))
                    out.append([idx])
                else:
                    out.append([int(g)])
        return out

    if "make_correlated_groups" in globals():
        try:
            mg_out = make_correlated_groups(Xtr_local, n_groups=None, random_state=BASE_SEED)
            groups_raw = None
            imp_from_groups = None

            if mg_out is None:
                groups_raw = None
                imp_from_groups = None
            elif isinstance(mg_out, tuple) or isinstance(mg_out, list):
                if len(mg_out) == 0:
                    groups_raw = None
                elif len(mg_out) == 1:
                    first = mg_out[0]
                    if isinstance(first, (list, tuple, np.ndarray)):
                        groups_raw = first
                    else:
                        try:
                            arr = np.asarray(first, dtype=float)
                            if arr.shape[0] == n_feat:
                                imp_from_groups = arr
                            else:
                                groups_raw = first
                        except Exception:
                            groups_raw = first
                else:
                    groups_raw = mg_out[0]
                    cand_imp = mg_out[1]
                    try:
                        arr = np.asarray(cand_imp, dtype=float)
                        if arr.shape[0] >= 1:
                            imp_from_groups = arr
                    except Exception:
                        imp_from_groups = None
            elif isinstance(mg_out, dict):
                if "groups" in mg_out and "importance" in mg_out:
                    groups_raw = mg_out["groups"]
                    imp_from_groups = np.asarray(mg_out["importance"], dtype=float)
                else:
                    groups_raw = list(mg_out.values())
            elif isinstance(mg_out, np.ndarray):
                imp_from_groups = np.asarray(mg_out, dtype=float)
            else:
                groups_raw = mg_out

            if groups_raw is not None:
                try:
                    groups_list = _coerce_groups_to_index_lists(groups_raw, Xtr_local)
                    groups_list = [[int(j) for j in g if 0 <= int(j) < n_feat] for g in groups_list if len(g) > 0]
                    if len(groups_list) == 0:
                        groups_list = None
                    else:
                        print(f"Correlated groups produced: {len(groups_list)} groups")
                except Exception as e:
                    print("Could not coerce correlated groups (exception):", e)
                    groups_list = None

            if imp_from_groups is not None:
                try:
                    tmp_imp = np.asarray(imp_from_groups, dtype=float)
                    if tmp_imp.shape[0] == n_feat:
                        imp_arr = tmp_imp.copy()
                        print("Using importance array returned by make_correlated_groups.")
                    else:
                        imp_arr = tmp_imp
                        print("Imported importance array (length mismatch will be padded/trimmed).")
                except Exception:
                    imp_arr = None

        except Exception as e:
            print("make_correlated_groups failed (will fallback to RF one-feature-per-group). Exception:")
            traceback.print_exc()
            groups_list = None
    else:
        print("make_correlated_groups not found; will fallback to RF/one-feature groups.")

    # retry with explicit heuristic if needed
    if groups_list is None and "make_correlated_groups" in globals():
        try:
            print("Retrying make_correlated_groups with explicit n_groups heuristic...")
            mg_retry = make_correlated_groups(Xtr_local, n_groups=max(2, int(round(np.sqrt(n_feat)))), random_state=BASE_SEED)
            if isinstance(mg_retry, tuple) and len(mg_retry) >= 1:
                groups_raw = mg_retry[0]
                imp_from_groups = mg_retry[1] if len(mg_retry) > 1 else None
            else:
                groups_raw = mg_retry
                imp_from_groups = None
            if groups_raw is not None:
                groups_list = _coerce_groups_to_index_lists(groups_raw, Xtr_local)
                groups_list = [[int(j) for j in g if 0 <= int(j) < n_feat] for g in groups_list if len(g) > 0]
                if len(groups_list) == 0:
                    groups_list = None
                else:
                    print(f"Retry succeeded: {len(groups_list)} groups")
            if imp_from_groups is not None and imp_arr is None:
                try:
                    tmp = np.asarray(imp_from_groups, dtype=float)
                    imp_arr = tmp
                except Exception:
                    pass
        except Exception as e:
            print("Retry attempt failed (ignored):", e)

    # compute RF importances as fallback / canonical importance
    from sklearn.ensemble import RandomForestRegressor
    rf = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=BASE_SEED, n_jobs=-1)
    try:
        rf.fit(Xtr_local, ytr_local)
        rf_imp = np.asarray(rf.feature_importances_, dtype=float)
    except Exception as e:
        print("RF importance failed:", e)
        rf_imp = np.zeros(n_feat, dtype=float)

    if groups_list is None:
        groups_list = [[i] for i in range(n_feat)]
        print("Fallback groups: one-feature-per-group (count = {})".format(len(groups_list)))

    # finalize imp_arr (prefer helper-provided, else RF)
    if imp_arr is None:
        imp_arr = rf_imp.copy()
    else:
        tmp = np.zeros(n_feat, dtype=float)
        tmp[:min(len(imp_arr), n_feat)] = imp_arr[:min(len(imp_arr), n_feat)]
        imp_arr = tmp

    # defensive pad/trim and normalization
    if imp_arr.shape[0] != n_feat:
        tmp = np.zeros(n_feat, dtype=float)
        tmp[:min(len(imp_arr), n_feat)] = imp_arr[:min(len(imp_arr), n_feat)]
        imp_arr = tmp
    imp_arr = np.nan_to_num(imp_arr, nan=0.0, posinf=0.0, neginf=0.0)
    if imp_arr.sum() <= 0:
        imp_arr = np.ones(n_feat, dtype=float) / float(n_feat)
    else:
        imp_arr = imp_arr.astype(float) / float(np.sum(imp_arr))

    print("Prepared groups_list length:", len(groups_list), "imp_arr shape:", imp_arr.shape)
    order = np.argsort(-imp_arr)[:min(8, n_feat)]
    print("Top importances (idx -> name -> score):")
    for i in order:
        print(i, Xtr_local.columns[i], f"{imp_arr[i]:.4f}")

    # --- ensure required helpers exist ---
    for helper in ("sample_one_subset", "fit_one_sr_on_subset", "eval_baselines"):
        if helper not in globals():
            raise RuntimeError(f"Helper '{helper}' missing — paste its definition before running this cell.")

    # --- SWEEP: importance-guided sampling + SR training ---
    best_te = float("inf")
    best_est = None
    best_subset = None
    rows = []

    print("Starting hybrid sweep: N_RUNS=", N_RUNS)
    for i in range(1, N_RUNS+1):
        run_seed = int(BASE_SEED + i)
        try:
            subset_idx = sample_one_subset(groups_list, imp_arr, alpha=ALPHA, pick_one_per_group=True,
                                           rng=np.random.default_rng(run_seed))
        except TypeError:
            subset_idx = sample_one_subset(groups_list, imp_arr, alpha=ALPHA, pick_one_per_group=True)

        # defensive: ensure subset_idx is list of ints
        try:
            subset_idx = [int(x) for x in list(subset_idx)]
        except Exception:
            subset_idx = list(subset_idx) if hasattr(subset_idx, "__iter__") else []

        # skip invalid empty subset
        if len(subset_idx) == 0:
            print(f"[{i}] sampled empty subset — skipping")
            rows.append({"run": i, "seed": run_seed, "subset_idx": subset_idx, "tr_rmse": None, "te_rmse": None, "expr": None, "degenerate": True})
            continue

        try:
            # primary SR config (richer population but moderate generations)
            est, tr_rmse, te_rmse = fit_one_sr_on_subset(
                Xtr_local, ytr_local, Xte_local, subset_idx,
                function_set=('add','sub','mul','div','sin','cos'),
                parsimony=0.001,
                generations=100,
                population_size=1500,
                random_state=run_seed,
                verbose=0
            )
        except Exception as e:
            print(f"[{i}] SR training failed: {e}")
            rows.append({"run": i, "seed": run_seed, "subset_idx": subset_idx, "tr_rmse": None, "te_rmse": None, "expr": None, "error": str(e), "degenerate": True})
            continue

        expr = None
        try:
            if hasattr(est, "program_"):
                expr = str(est.program_)
            elif hasattr(est, "program"):
                expr = str(est.program)
            else:
                expr = repr(est)
        except Exception:
            expr = repr(est)

        # compute SR feature predictions & check degeneracy (tiny train variance)
        try:
            # select subset columns for predict (works for df or ndarray)
            try:
                Xtr_sub = Xtr_local.iloc[:, subset_idx]
                Xte_sub = Xte_local.iloc[:, subset_idx]
            except Exception:
                Xtr_sub = Xtr_local.loc[:, subset_idx]
                Xte_sub = Xte_local.loc[:, subset_idx]

            sr_tr_feature = est.predict(Xtr_sub)
            sr_te_feature = est.predict(Xte_sub)
            std_tr = float(np.nanstd(sr_tr_feature))
            degenerate = (not np.isfinite(std_tr)) or (std_tr < VAR_THRESH)
            if degenerate:
                print(f"[{i}] Degenerate SR run (std_tr={std_tr:.6g}) — skipping for best-model selection. expr={expr}")
            else:
                # only consider for best if non-degenerate and te_rmse valid
                if te_rmse is not None and te_rmse < best_te:
                    best_te = te_rmse
                    best_est = est
                    best_subset = subset_idx

            rows.append({"run": i, "seed": run_seed, "subset_idx": subset_idx,
                         "tr_rmse": float(tr_rmse) if tr_rmse is not None else None,
                         "te_rmse": float(te_rmse) if te_rmse is not None else None,
                         "expr": expr, "degenerate": bool(degenerate), "sr_tr_std": std_tr})
        except Exception as e:
            print(f"[{i}] Prediction or degeneracy check failed: {e}")
            rows.append({"run": i, "seed": run_seed, "subset_idx": subset_idx, "tr_rmse": float(tr_rmse) if tr_rmse is not None else None,
                         "te_rmse": float(te_rmse) if te_rmse is not None else None, "expr": expr, "degenerate": True, "error": str(e)})
            continue

        if (i % 10) == 0:
            print(f"[{i}/{N_RUNS}] checkpoint best_te: {best_te:.6f}")

    # Save raw sweep rows
    outdir = ROOT
    outdir.mkdir(parents=True, exist_ok=True)
    partial_csv = outdir / f"{ds_name}_sweep_partial.csv"
    pd.DataFrame(rows).to_csv(partial_csv, index=False)
    print("Saved partial sweep results to:", partial_csv)

    # --- finalize: if best found, augment and evaluate baselines (including RAE) ---
    summary = {
        "dataset": ds_name,
        "n_samples_train": int(Xtr_local.shape[0]),
        "n_features": int(Xtr_local.shape[1]),
        "best_subset": best_subset,
        "best_te": float(best_te) if best_te != float("inf") else None,
        "timestamp": time.time(),
    }

    if best_est is not None and best_subset is not None:
        try:
            try:
                Xtr_sub = Xtr_local.iloc[:, best_subset]
                Xte_sub = Xte_local.iloc[:, best_subset]
            except Exception:
                Xtr_sub = Xtr_local.loc[:, best_subset]
                Xte_sub = Xte_local.loc[:, best_subset]

            sr_tr_feature = best_est.predict(Xtr_sub)
            sr_te_feature = best_est.predict(Xte_sub)

            # final degenerate guard (do not augment if degenerate)
            if np.nanstd(sr_tr_feature) < VAR_THRESH or not np.isfinite(np.nanstd(sr_tr_feature)):
                print("Warning: Best SR feature has near-zero variance on train — marking as degenerate; summary saved but augmentation skipped.")
                summary.update({"note": "best_SR_feature_degenerate"})
            else:
                Xtr_aug = Xtr_local.copy(); Xte_aug = Xte_local.copy()
                Xtr_aug["SR_1"] = sr_tr_feature
                Xte_aug["SR_1"] = sr_te_feature

                baseline_metrics = eval_baselines(Xtr_local, Xte_local, ytr_local, yte_local)
                augmented_metrics = eval_baselines(Xtr_aug, Xte_aug, ytr_local, yte_local)
                dRMSE = {k.replace("_RMSE", "_dRMSE"): augmented_metrics[k] - baseline_metrics[k] for k in baseline_metrics}

                summary.update({
                    "baseline": baseline_metrics,
                    "augmented": augmented_metrics,
                    "dRMSE": dRMSE
                })

                print("Baseline:", baseline_metrics)
                print("Augmented:", augmented_metrics)
                print("ΔRMSE:", dRMSE)

            # persist summary
            with open(eval_json, "w") as f:
                json.dump(summary, f, indent=2)
            pd.DataFrame([summary]).to_csv(eval_csv, index=False)

            print("Saved evaluation summary JSON:", eval_json)
            print("Saved evaluation summary CSV:", eval_csv)

        except Exception as e:
            print("Error during final augmentation/evaluation:", e)
            traceback.print_exc()
    else:
        print("No successful non-degenerate SR model found during sweep for", ds_name)
        # still save summary skeleton
        with open(eval_json, "w") as f:
            json.dump(summary, f, indent=2)
        pd.DataFrame([summary]).to_csv(eval_csv, index=False)

    print("Elapsed (s):", time.time() - start_ts)

Hybrid run config: {'dataset': 'wine_red', 'N_RUNS': 40, 'ALPHA': 2.0, 'BASE_SEED': 42, 'VAR_THRESH': 0.001}
Loaded dataset: wine_red | train: (1279, 11) | test: (320, 11) | used files: ['X_train.parquet', 'y_train.pkl']
After impute shapes: (1279, 11) (320, 11)


/home/rshayan1/.local/lib/python3.12/site-packages/sklearn/cluster/_agglomerative.py:983: FutureWarning: Attribute `affinity` was deprecated in version 1.2 and will be removed in 1.4. Use `metric` instead
  warnings.warn(


make_correlated_groups: n_features=11, n_groups_requested=3, produced_groups=3
Sample groups (first 6): [[3], [5, 6], [0, 1, 2, 4, 7, 8, 9, 10]]
Top features by imp (idx -> name -> score):
 10 10 0.2931
 9 9 0.1625
 1 1 0.0911
 6 6 0.0760
 8 8 0.0600
 7 7 0.0570
 4 4 0.0561
 5 5 0.0545
 3 3 0.0513
 0 0 0.0496
Correlated groups produced: 3 groups
Using importance array returned by make_correlated_groups.
Prepared groups_list length: 3 imp_arr shape: (11,)
Top importances (idx -> name -> score):
10 alcohol 0.2931
9 sulphates 0.1625
1 volatile acidity 0.0911
6 total sulfur dioxide 0.0760
8 pH 0.0600
7 density 0.0570
4 chlorides 0.0561
5 free sulfur dioxide 0.0545
Starting hybrid sweep: N_RUNS= 40
[10/40] checkpoint best_te: 0.728978
[12] Degenerate SR run (std_tr=8.88178e-16) — skipping for best-model selection. expr=SymbolicRegressor(function_set=['add', 'sub', 'mul', 'div', 'sin', 'cos'],
                  generations=100, population_size=1500, random_state=54)
[20/40] checkpoint best_t

In [19]:
print("groups_list:", groups_list[:5])
print("len groups:", len(groups_list))

groups_list: [[3], [5, 6], [0, 1, 2, 4, 7, 8, 9, 10]]
len groups: 3


## Wine-white Testing

In [11]:
# CELL: run_hybrid_for_one_dataset (wine_white) - updated to new config + RAE/degeneracy checks
import time, json, shutil, traceback
from pathlib import Path
import numpy as np
import pandas as pd

# USER CONFIG (change per dataset)
ds_name = "wine_white" # << canonical folder name
N_RUNS = 40                     # full sweep use 40; lower for quick smoke
ALPHA = 2.0                     # importance exponent
BASE_SEED = 42
VAR_THRESH = 1e-3               # reject SR features with std < VAR_THRESH (degenerate)
ROOT = Path("artifacts") / "hybrid_experiments_summary"
ROOT.mkdir(parents=True, exist_ok=True)

print("Hybrid run config:", {"dataset": ds_name, "N_RUNS": N_RUNS, "ALPHA": ALPHA, "BASE_SEED": BASE_SEED, "VAR_THRESH": VAR_THRESH})

# output guard (skip if already evaluated)
eval_json = ROOT / f"{ds_name}_evaluation_summary.json"
eval_csv = ROOT / f"{ds_name}_evaluation_summary.csv"
if eval_json.exists() or eval_csv.exists():
    print(f"Skipping {ds_name}: evaluation already exists at {eval_json} / {eval_csv}")
else:
    start_ts = time.time()

    # --- load canonical dataset (uses your load_canonical_Xy) ---
    ds_dir = Path("artifacts") / "canonical" / ds_name
    if "load_canonical_Xy" not in globals():
        raise RuntimeError("load_canonical_Xy(...) not found — define it before running this cell.")

    X_all, y_all, meta, used_files = load_canonical_Xy(ds_dir)
    # try explicit test split if present
    Xte = None; yte = None
    try:
        if (ds_dir / "X_test.parquet").exists():
            Xte = pd.read_parquet(ds_dir / "X_test.parquet")
        if (ds_dir / "y_test.pkl").exists():
            yte = pd.read_pickle(ds_dir / "y_test.pkl").squeeze()
    except Exception as e:
        print("Warning loading explicit test split (ignored):", e)

    # if no explicit test, do deterministic 80/20 split
    if Xte is None or yte is None:
        from sklearn.model_selection import train_test_split
        Xtr_df, Xte_df, ytr_ser, yte_ser = train_test_split(
            pd.DataFrame(X_all), y_all, test_size=0.2, random_state=42
        )
        ytr = np.asarray(pd.to_numeric(ytr_ser, errors="coerce")).ravel()
        yte = np.asarray(pd.to_numeric(yte_ser, errors="coerce")).ravel()
    else:
        Xtr_df = pd.DataFrame(X_all)
        Xte_df = pd.DataFrame(Xte)
        ytr = np.asarray(pd.to_numeric(y_all, errors="coerce")).ravel()
        yte = np.asarray(pd.to_numeric(yte, errors="coerce")).ravel()

    # canonical local copies
    Xtr_local = Xtr_df.reset_index(drop=True)
    Xte_local = Xte_df.reset_index(drop=True)
    ytr_local = ytr
    yte_local = yte

    print("Loaded dataset:", ds_name, "| train:", Xtr_local.shape, "| test:", Xte_local.shape, "| used files:", used_files)

    # --- impute + coerce numeric defensively ---
    from sklearn.impute import SimpleImputer
    imp = SimpleImputer(strategy="median")
    if (Xtr_local.isna().sum().sum() > 0) or (Xte_local.isna().sum().sum() > 0):
        Xtr_imp = pd.DataFrame(imp.fit_transform(Xtr_local), columns=Xtr_local.columns, index=Xtr_local.index)
        Xte_imp = pd.DataFrame(imp.transform(Xte_local), columns=Xte_local.columns, index=Xte_local.index)
    else:
        Xtr_imp = Xtr_local.copy(); Xte_imp = Xte_local.copy()

    # coerce non-numeric to numeric (defensive)
    for df in (Xtr_imp, Xte_imp):
        for c in df.columns:
            if not pd.api.types.is_numeric_dtype(df[c]):
                df[c] = pd.to_numeric(df[c], errors="coerce")
    Xtr_imp = Xtr_imp.fillna(0.0); Xte_imp = Xte_imp.fillna(0.0)

    ytr_arr = np.asarray(pd.to_numeric(ytr_local, errors="coerce")).ravel()
    yte_arr = np.asarray(pd.to_numeric(yte_local, errors="coerce")).ravel()

    Xtr_local = Xtr_imp.reset_index(drop=True)
    Xte_local = Xte_imp.reset_index(drop=True)
    ytr_local = ytr_arr
    yte_local = yte_arr

    print("After impute shapes:", Xtr_local.shape, Xte_local.shape)

    # --- build groups & importances using make_correlated_groups (robust handling) ---
    n_feat = Xtr_local.shape[1]
    groups_list = None
    imp_arr = None

    def _coerce_groups_to_index_lists(groups_in, Xdf):
        if groups_in is None:
            return None
        out = []
        if isinstance(groups_in, dict):
            vals = list(groups_in.values())
        else:
            vals = list(groups_in)
        for g in vals:
            if isinstance(g, (list, tuple, np.ndarray)):
                idxs = []
                for el in g:
                    if isinstance(el, str):
                        try:
                            idxs.append(int(list(Xdf.columns).index(el)))
                        except Exception:
                            raise ValueError(f"Cannot map group element '{el}' -> not a column name")
                    else:
                        idxs.append(int(el))
                out.append(list(idxs))
            else:
                if isinstance(g, str):
                    idx = int(list(Xdf.columns).index(g))
                    out.append([idx])
                else:
                    out.append([int(g)])
        return out

    # try calling helper and be robust to different return shapes
    if "make_correlated_groups" in globals():
        try:
            mg_out = make_correlated_groups(Xtr_local, n_groups=None, random_state=BASE_SEED)
            groups_raw = None
            imp_from_groups = None

            if mg_out is None:
                groups_raw = None
                imp_from_groups = None
            elif isinstance(mg_out, (tuple, list)):
                if len(mg_out) == 0:
                    groups_raw = None
                elif len(mg_out) == 1:
                    first = mg_out[0]
                    if isinstance(first, (list, tuple, np.ndarray)):
                        groups_raw = first
                    else:
                        try:
                            arr = np.asarray(first, dtype=float)
                            if arr.shape[0] == n_feat:
                                imp_from_groups = arr
                            else:
                                groups_raw = first
                        except Exception:
                            groups_raw = first
                else:
                    groups_raw = mg_out[0]
                    cand_imp = mg_out[1]
                    try:
                        arr = np.asarray(cand_imp, dtype=float)
                        if arr.shape[0] >= 1:
                            imp_from_groups = arr
                    except Exception:
                        imp_from_groups = None
            elif isinstance(mg_out, dict):
                if "groups" in mg_out and "importance" in mg_out:
                    groups_raw = mg_out["groups"]
                    imp_from_groups = np.asarray(mg_out["importance"], dtype=float)
                else:
                    groups_raw = list(mg_out.values())
            elif isinstance(mg_out, np.ndarray):
                imp_from_groups = np.asarray(mg_out, dtype=float)
            else:
                groups_raw = mg_out

            if groups_raw is not None:
                try:
                    groups_list = _coerce_groups_to_index_lists(groups_raw, Xtr_local)
                    groups_list = [[int(j) for j in g if 0 <= int(j) < n_feat] for g in groups_list if len(g) > 0]
                    if len(groups_list) == 0:
                        groups_list = None
                    else:
                        print(f"Correlated groups produced: {len(groups_list)} groups")
                except Exception as e:
                    print("Could not coerce correlated groups (exception):", e)
                    groups_list = None

            if imp_from_groups is not None:
                try:
                    tmp_imp = np.asarray(imp_from_groups, dtype=float)
                    if tmp_imp.shape[0] == n_feat:
                        imp_arr = tmp_imp.copy()
                        print("Using importance array returned by make_correlated_groups.")
                    else:
                        imp_arr = tmp_imp
                        print("Imported importance array (length mismatch will be padded/trimmed).")
                except Exception:
                    imp_arr = None

        except Exception as e:
            print("make_correlated_groups failed (will fallback to RF one-feature-per-group). Exception:")
            traceback.print_exc()
            groups_list = None
    else:
        print("make_correlated_groups not found; will fallback to RF/one-feature groups.")

    # retry with explicit heuristic if needed
    if groups_list is None and "make_correlated_groups" in globals():
        try:
            print("Retrying make_correlated_groups with explicit n_groups heuristic...")
            mg_retry = make_correlated_groups(Xtr_local, n_groups=max(2, int(round(np.sqrt(n_feat)))), random_state=BASE_SEED)
            if isinstance(mg_retry, (tuple, list)) and len(mg_retry) >= 1:
                groups_raw = mg_retry[0]
                imp_from_groups = mg_retry[1] if len(mg_retry) > 1 else None
            else:
                groups_raw = mg_retry
                imp_from_groups = None
            if groups_raw is not None:
                groups_list = _coerce_groups_to_index_lists(groups_raw, Xtr_local)
                groups_list = [[int(j) for j in g if 0 <= int(j) < n_feat] for g in groups_list if len(g) > 0]
                if len(groups_list) == 0:
                    groups_list = None
                else:
                    print(f"Retry succeeded: {len(groups_list)} groups")
            if imp_from_groups is not None and imp_arr is None:
                try:
                    tmp = np.asarray(imp_from_groups, dtype=float)
                    imp_arr = tmp
                except Exception:
                    pass
        except Exception as e:
            print("Retry attempt failed (ignored):", e)

    # compute RF importances as fallback / canonical importance
    from sklearn.ensemble import RandomForestRegressor
    rf = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=BASE_SEED, n_jobs=-1)
    try:
        rf.fit(Xtr_local, ytr_local)
        rf_imp = np.asarray(rf.feature_importances_, dtype=float)
    except Exception as e:
        print("RF importance failed:", e)
        rf_imp = np.zeros(n_feat, dtype=float)

    if groups_list is None:
        groups_list = [[i] for i in range(n_feat)]
        print("Fallback groups: one-feature-per-group (count = {})".format(len(groups_list)))

    # finalize imp_arr (prefer helper-provided, else RF)
    if imp_arr is None:
        imp_arr = rf_imp.copy()
    else:
        tmp = np.zeros(n_feat, dtype=float)
        tmp[:min(len(imp_arr), n_feat)] = imp_arr[:min(len(imp_arr), n_feat)]
        imp_arr = tmp

    # defensive pad/trim and normalization
    if imp_arr.shape[0] != n_feat:
        tmp = np.zeros(n_feat, dtype=float)
        tmp[:min(len(imp_arr), n_feat)] = imp_arr[:min(len(imp_arr), n_feat)]
        imp_arr = tmp
    imp_arr = np.nan_to_num(imp_arr, nan=0.0, posinf=0.0, neginf=0.0)
    if imp_arr.sum() <= 0:
        imp_arr = np.ones(n_feat, dtype=float) / float(n_feat)
    else:
        imp_arr = imp_arr.astype(float) / float(np.sum(imp_arr))

    print("Prepared groups_list length:", len(groups_list), "imp_arr shape:", imp_arr.shape)
    order = np.argsort(-imp_arr)[:min(8, n_feat)]
    print("Top importances (idx -> name -> score):")
    for i in order:
        print(i, Xtr_local.columns[i], f"{imp_arr[i]:.4f}")

    # --- ensure required helpers exist ---
    for helper in ("sample_one_subset", "fit_one_sr_on_subset", "eval_baselines"):
        if helper not in globals():
            raise RuntimeError(f"Helper '{helper}' missing — paste its definition before running this cell.")

    # --- SWEEP: importance-guided sampling + SR training ---
    best_te = float("inf")
    best_est = None
    best_subset = None
    rows = []

    print("Starting hybrid sweep: N_RUNS=", N_RUNS)
    for i in range(1, N_RUNS+1):
        run_seed = int(BASE_SEED + i)
        try:
            subset_idx = sample_one_subset(groups_list, imp_arr, alpha=ALPHA, pick_one_per_group=True,
                                           rng=np.random.default_rng(run_seed))
        except TypeError:
            subset_idx = sample_one_subset(groups_list, imp_arr, alpha=ALPHA, pick_one_per_group=True)

        # defensive: ensure subset_idx is list of ints
        try:
            subset_idx = [int(x) for x in list(subset_idx)]
        except Exception:
            subset_idx = list(subset_idx) if hasattr(subset_idx, "__iter__") else []

        # skip invalid empty subset
        if len(subset_idx) == 0:
            print(f"[{i}] sampled empty subset — skipping")
            rows.append({"run": i, "seed": run_seed, "subset_idx": subset_idx, "tr_rmse": None, "te_rmse": None, "expr": None, "status":"empty_subset"})
            continue

        try:
            # primary SR config (richer population but moderate generations)
            est, tr_rmse, te_rmse = fit_one_sr_on_subset(
                Xtr_local, ytr_local, Xte_local, subset_idx,
                function_set=('add','sub','mul','div','sin','cos'),
                parsimony=0.001,
                generations=100,
                population_size=1500,
                random_state=run_seed,
                verbose=0
            )
        except Exception as e:
            print(f"[{i}] SR training failed: {e}")
            rows.append({"run": i, "seed": run_seed, "subset_idx": subset_idx, "tr_rmse": None, "te_rmse": None, "expr": None, "error": str(e)})
            continue

        # --- re-predict SR feature and check variance to reject degenerate runs ---
        try:
            # build subset dataframes/arrays for prediction
            try:
                Xtr_sub = Xtr_local.iloc[:, subset_idx]
                Xte_sub = Xte_local.iloc[:, subset_idx]
            except Exception:
                Xtr_sub = Xtr_local.loc[:, subset_idx]
                Xte_sub = Xte_local.loc[:, subset_idx]

            sr_tr_feature = est.predict(Xtr_sub)
            sr_te_feature = est.predict(Xte_sub)
            sr_var = float(np.nanstd(sr_tr_feature))

            if (not np.isfinite(sr_var)) or (sr_var < VAR_THRESH):
                # mark this run degenerate and skip
                print(f"[{i}] SR run marked degenerate (sr_var={sr_var:.6g}) -- skipping")
                rows.append({"run": i, "seed": run_seed, "subset_idx": subset_idx, "tr_rmse": None, "te_rmse": None, "expr": None, "status":"degenerate", "sr_var": sr_var})
                continue
        except Exception as e:
            # if prediction failed, mark as error and continue
            print(f"[{i}] SR prediction/variance check failed: {e}")
            rows.append({"run": i, "seed": run_seed, "subset_idx": subset_idx, "tr_rmse": None, "te_rmse": None, "expr": None, "error": str(e)})
            continue

        # extract expression string representation
        expr = None
        try:
            if hasattr(est, "program_"):
                expr = str(est.program_)
            elif hasattr(est, "program"):
                expr = str(est.program)
            else:
                expr = repr(est)
        except Exception:
            expr = repr(est)

        rows.append({"run": i, "seed": run_seed, "subset_idx": subset_idx, "tr_rmse": float(tr_rmse) if tr_rmse is not None else None,
                     "te_rmse": float(te_rmse) if te_rmse is not None else None, "expr": expr, "sr_var": sr_var})

        if te_rmse is not None and te_rmse < best_te:
            best_te = te_rmse
            best_est = est
            best_subset = subset_idx

        if (i % 10) == 0:
            print(f"[{i}/{N_RUNS}] checkpoint best_te: {best_te:.6f}")

    # Save raw sweep rows
    outdir = ROOT
    outdir.mkdir(parents=True, exist_ok=True)
    partial_csv = outdir / f"{ds_name}_sweep_partial.csv"
    pd.DataFrame(rows).to_csv(partial_csv, index=False)
    print("Saved partial sweep results to:", partial_csv)

    # --- finalize: if best found, augment and evaluate baselines (including RAE) ---
    summary = {
        "dataset": ds_name,
        "n_samples_train": int(Xtr_local.shape[0]),
        "n_features": int(Xtr_local.shape[1]),
        "best_subset": best_subset,
        "best_te": float(best_te) if best_te != float("inf") else None,
        "timestamp": time.time(),
    }

    if best_est is not None and best_subset is not None:
        try:
            # build SR feature predictions (best)
            try:
                Xtr_sub = Xtr_local.iloc[:, best_subset]
                Xte_sub = Xte_local.iloc[:, best_subset]
            except Exception:
                Xtr_sub = Xtr_local.loc[:, best_subset]
                Xte_sub = Xte_local.loc[:, best_subset]

            sr_tr_feature = best_est.predict(Xtr_sub)
            sr_te_feature = best_est.predict(Xte_sub)

            # final degenerate guard (do not augment if degenerate)
            if np.nanstd(sr_tr_feature) < VAR_THRESH or not np.isfinite(np.nanstd(sr_tr_feature)):
                print("Warning: Best SR feature has near-zero variance on train — marking as degenerate; summary saved but augmentation skipped.")
                summary.update({"note": "best_SR_feature_degenerate"})
            else:
                Xtr_aug = Xtr_local.copy(); Xte_aug = Xte_local.copy()
                Xtr_aug["SR_1"] = sr_tr_feature
                Xte_aug["SR_1"] = sr_te_feature

                baseline_metrics = eval_baselines(Xtr_local, Xte_local, ytr_local, yte_local)
                augmented_metrics = eval_baselines(Xtr_aug, Xte_aug, ytr_local, yte_local)
                dRMSE = {k.replace("_RMSE", "_dRMSE"): augmented_metrics[k] - baseline_metrics[k] for k in baseline_metrics}

                summary.update({
                    "baseline": baseline_metrics,
                    "augmented": augmented_metrics,
                    "dRMSE": dRMSE
                })

                print("Baseline:", baseline_metrics)
                print("Augmented:", augmented_metrics)
                print("ΔRMSE:", dRMSE)

            # persist summary
            with open(eval_json, "w") as f:
                json.dump(summary, f, indent=2)
            pd.DataFrame([summary]).to_csv(eval_csv, index=False)

            print("Saved evaluation summary JSON:", eval_json)
            print("Saved evaluation summary CSV:", eval_csv)

        except Exception as e:
            print("Error during final augmentation/evaluation:", e)
            traceback.print_exc()
    else:
        print("No successful SR model found during sweep for", ds_name)
        # still save summary skeleton
        with open(eval_json, "w") as f:
            json.dump(summary, f, indent=2)
        pd.DataFrame([summary]).to_csv(eval_csv, index=False)

    print("Elapsed (s):", time.time() - start_ts)

Hybrid run config: {'dataset': 'wine_white', 'N_RUNS': 40, 'ALPHA': 2.0, 'BASE_SEED': 42, 'VAR_THRESH': 0.001}
Loaded dataset: wine_white | train: (3918, 11) | test: (980, 11) | used files: ['X_train.parquet', 'y_train.pkl']
After impute shapes: (3918, 11) (980, 11)


/home/rshayan1/.local/lib/python3.12/site-packages/sklearn/cluster/_agglomerative.py:983: FutureWarning: Attribute `affinity` was deprecated in version 1.2 and will be removed in 1.4. Use `metric` instead
  warnings.warn(


make_correlated_groups: n_features=11, n_groups_requested=3, produced_groups=3
Sample groups (first 6): [[9], [1], [0, 2, 3, 4, 5, 6, 7, 8, 10]]
Top features by imp (idx -> name -> score):
 10 10 0.3569
 1 1 0.1346
 5 5 0.0937
 7 7 0.0643
 3 3 0.0558
 8 8 0.0554
 2 2 0.0553
 4 4 0.0512
 6 6 0.0453
 9 9 0.0443
Correlated groups produced: 3 groups
Using importance array returned by make_correlated_groups.
Prepared groups_list length: 3 imp_arr shape: (11,)
Top importances (idx -> name -> score):
10 alcohol 0.3569
1 volatile acidity 0.1346
5 free sulfur dioxide 0.0937
7 density 0.0643
3 residual sugar 0.0558
8 pH 0.0554
2 citric acid 0.0553
4 chlorides 0.0512
Starting hybrid sweep: N_RUNS= 40
[1] SR run marked degenerate (sr_var=8.88178e-16) -- skipping
[2] SR run marked degenerate (sr_var=8.88178e-16) -- skipping
[4] SR run marked degenerate (sr_var=8.88178e-16) -- skipping
[5] SR run marked degenerate (sr_var=1.77636e-15) -- skipping
[6] SR run marked degenerate (sr_var=0) -- skipping
[

## Airfoil Testing

In [90]:
# quick probe — run this first to find the airfoil file
from pathlib import Path
import os

print("cwd:", Path.cwd())
print("home():", Path.home())

candidates = [
    Path("airfoil_self_noise.dat"),
    Path("rshayan1/Datasets/airfoil_self_noise.dat"),
    Path("/rshayan1/Datasets/airfoil_self_noise.dat"),
    Path("/home/rshayan1/Datasets/airfoil_self_noise.dat"),
    Path("/mnt/data/airfoil_self_noise.dat"),
    Path("/mnt/data/airfoil_self_noise (1).dat"),
    Path("./rshayan1/Datasets/airfoil_self_noise.dat"),
]

found = None
for p in candidates:
    print("checking", p, "->", p.exists())
    if p.exists():
        found = p
        break

if found is None:
    # fallback: search the workspace for the filename (may take a few seconds)
    print("Not found in candidates — doing a quick walk from cwd (stops at first match)...")
    for root, dirs, files in os.walk(Path.cwd()):
        if "airfoil_self_noise.dat" in files:
            found = Path(root) / "airfoil_self_noise.dat"
            print("Found at:", found)
            break

if found is None:
    raise FileNotFoundError("airfoil_self_noise.dat not found. Look for the file in the file browser and note its full absolute path.")

print("Using file path:", found)

cwd: /home/rshayan1
home(): /home/rshayan1
checking airfoil_self_noise.dat -> False
checking rshayan1/Datasets/airfoil_self_noise.dat -> False
checking /rshayan1/Datasets/airfoil_self_noise.dat -> False
checking /home/rshayan1/Datasets/airfoil_self_noise.dat -> True
Using file path: /home/rshayan1/Datasets/airfoil_self_noise.dat


In [12]:
# CELL: prepare canonical/airfoil dataset (standardized) — fixed path
from pathlib import Path
import json
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

ROOT_CANON = Path("artifacts") / "canonical"
ROOT_CANON.mkdir(parents=True, exist_ok=True)

ds_name = "airfoil"
out_dir = ROOT_CANON / ds_name

# reset folder
if out_dir.exists():
    for p in out_dir.iterdir():
        p.unlink()
else:
    out_dir.mkdir(parents=True)

# === FIXED PATH ===
data_path = Path("/home/rshayan1/Datasets/airfoil_self_noise.dat")
print("Using file:", data_path)

if not data_path.exists():
    raise FileNotFoundError(f"File not found at {data_path}. Check path.")

# column names
cols = ["Frequency", "Angle", "Chord", "Velocity", "Thickness", "SoundPressure"]

# load
df = pd.read_csv(data_path, sep=r"\s+", header=None, names=cols, engine="python")

# split
X = df.drop(columns=["SoundPressure"])
y = df["SoundPressure"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# standardize (important for SR consistency)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test),  columns=X_test.columns)

# save
X_train_scaled.to_parquet(out_dir / "X_train.parquet", index=False)
X_test_scaled.to_parquet(out_dir / "X_test.parquet", index=False)
pd.to_pickle(y_train.reset_index(drop=True), out_dir / "y_train.pkl")
pd.to_pickle(y_test.reset_index(drop=True),  out_dir / "y_test.pkl")

meta = {
    "n_samples_train": int(X_train.shape[0]),
    "n_samples_test": int(X_test.shape[0]),
    "n_features": int(X_train.shape[1]),
    "feature_names": list(X_train.columns)
}
with open(out_dir / "meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Saved canonical dataset to:", out_dir)
print("Train shape:", X_train_scaled.shape, "Test shape:", X_test_scaled.shape)

Using file: /home/rshayan1/Datasets/airfoil_self_noise.dat
Saved canonical dataset to: artifacts/canonical/airfoil
Train shape: (1202, 5) Test shape: (301, 5)


In [14]:
# CELL: run_hybrid_for_one_dataset (airfoil, new config)
import time, json, shutil, traceback
from pathlib import Path
import numpy as np
import pandas as pd

# USER CONFIG
ds_name = "airfoil"
N_RUNS = 40
ALPHA = 2.0
BASE_SEED = 42
VAR_THRESH = 1e-3  # reject SR features with std < VAR_THRESH (degenerate)
ROOT = Path("artifacts") / "hybrid_experiments_summary"
ROOT.mkdir(parents=True, exist_ok=True)

print("Hybrid run config:", {
    "dataset": ds_name,
    "N_RUNS": N_RUNS,
    "ALPHA": ALPHA,
    "BASE_SEED": BASE_SEED,
    "VAR_THRESH": VAR_THRESH
})

# output guard
eval_json = ROOT / f"{ds_name}_evaluation_summary.json"
eval_csv = ROOT / f"{ds_name}_evaluation_summary.csv"
if eval_json.exists() or eval_csv.exists():
    print(f"Skipping {ds_name}: evaluation already exists at {eval_json} / {eval_csv}")
else:
    start_ts = time.time()

    # --- load canonical dataset ---
    ds_dir = Path("artifacts") / "canonical" / ds_name
    if "load_canonical_Xy" not in globals():
        raise RuntimeError("load_canonical_Xy(...) not found — define it before running this cell.")

    X_all, y_all, meta, used_files = load_canonical_Xy(ds_dir)

    # try explicit test split if present
    Xte = None
    yte = None
    try:
        if (ds_dir / "X_test.parquet").exists():
            Xte = pd.read_parquet(ds_dir / "X_test.parquet")
        if (ds_dir / "y_test.pkl").exists():
            yte = pd.read_pickle(ds_dir / "y_test.pkl").squeeze()
    except Exception as e:
        print("Warning loading explicit test split (ignored):", e)

    # if no explicit test, do deterministic 80/20 split
    if Xte is None or yte is None:
        from sklearn.model_selection import train_test_split
        Xtr_df, Xte_df, ytr_ser, yte_ser = train_test_split(
            pd.DataFrame(X_all), y_all, test_size=0.2, random_state=42
        )
        ytr = np.asarray(pd.to_numeric(ytr_ser, errors="coerce")).ravel()
        yte = np.asarray(pd.to_numeric(yte_ser, errors="coerce")).ravel()
    else:
        Xtr_df = pd.DataFrame(X_all)
        Xte_df = pd.DataFrame(Xte)
        ytr = np.asarray(pd.to_numeric(y_all, errors="coerce")).ravel()
        yte = np.asarray(pd.to_numeric(yte, errors="coerce")).ravel()

    # canonical local copies
    Xtr_local = Xtr_df.reset_index(drop=True)
    Xte_local = Xte_df.reset_index(drop=True)
    ytr_local = ytr
    yte_local = yte

    print("Loaded dataset:", ds_name, "| train:", Xtr_local.shape, "| test:", Xte_local.shape, "| used files:", used_files)

    # --- impute + coerce numeric defensively ---
    from sklearn.impute import SimpleImputer
    imp = SimpleImputer(strategy="median")
    if (Xtr_local.isna().sum().sum() > 0) or (Xte_local.isna().sum().sum() > 0):
        Xtr_imp = pd.DataFrame(imp.fit_transform(Xtr_local), columns=Xtr_local.columns, index=Xtr_local.index)
        Xte_imp = pd.DataFrame(imp.transform(Xte_local), columns=Xte_local.columns, index=Xte_local.index)
    else:
        Xtr_imp = Xtr_local.copy()
        Xte_imp = Xte_local.copy()

    for df in (Xtr_imp, Xte_imp):
        for c in df.columns:
            if not pd.api.types.is_numeric_dtype(df[c]):
                df[c] = pd.to_numeric(df[c], errors="coerce")
    Xtr_imp = Xtr_imp.fillna(0.0)
    Xte_imp = Xte_imp.fillna(0.0)

    Xtr_local = Xtr_imp.reset_index(drop=True)
    Xte_local = Xte_imp.reset_index(drop=True)
    ytr_local = np.asarray(pd.to_numeric(ytr_local, errors="coerce")).ravel()
    yte_local = np.asarray(pd.to_numeric(yte_local, errors="coerce")).ravel()

    print("After impute shapes:", Xtr_local.shape, Xte_local.shape)

    # --- build groups & importances ---
    n_feat = Xtr_local.shape[1]
    groups_list = None
    imp_arr = None

    def _coerce_groups_to_index_lists(groups_in, Xdf):
        if groups_in is None:
            return None
        out = []
        if isinstance(groups_in, dict):
            vals = list(groups_in.values())
        else:
            vals = list(groups_in)
        for g in vals:
            if isinstance(g, (list, tuple, np.ndarray)):
                idxs = []
                for el in g:
                    if isinstance(el, str):
                        try:
                            idxs.append(int(list(Xdf.columns).index(el)))
                        except Exception:
                            raise ValueError(f"Cannot map group element '{el}' -> not a column name")
                    else:
                        idxs.append(int(el))
                out.append(list(idxs))
            else:
                if isinstance(g, str):
                    idx = int(list(Xdf.columns).index(g))
                    out.append([idx])
                else:
                    out.append([int(g)])
        return out

    if "make_correlated_groups" in globals():
        try:
            mg_out = make_correlated_groups(Xtr_local, n_groups=None, random_state=BASE_SEED)
            groups_raw = None
            imp_from_groups = None

            if mg_out is None:
                groups_raw = None
            elif isinstance(mg_out, (tuple, list)):
                if len(mg_out) == 1:
                    first = mg_out[0]
                    if isinstance(first, (list, tuple, np.ndarray)):
                        groups_raw = first
                    else:
                        try:
                            arr = np.asarray(first, dtype=float)
                            if arr.shape[0] == n_feat:
                                imp_from_groups = arr
                            else:
                                groups_raw = first
                        except Exception:
                            groups_raw = first
                else:
                    groups_raw = mg_out[0]
                    try:
                        imp_from_groups = np.asarray(mg_out[1], dtype=float)
                    except Exception:
                        imp_from_groups = None
            elif isinstance(mg_out, dict):
                if "groups" in mg_out and "importance" in mg_out:
                    groups_raw = mg_out["groups"]
                    imp_from_groups = np.asarray(mg_out["importance"], dtype=float)
                else:
                    groups_raw = list(mg_out.values())
            elif isinstance(mg_out, np.ndarray):
                imp_from_groups = np.asarray(mg_out, dtype=float)
            else:
                groups_raw = mg_out

            if groups_raw is not None:
                groups_list = _coerce_groups_to_index_lists(groups_raw, Xtr_local)
                groups_list = [[int(j) for j in g if 0 <= int(j) < n_feat] for g in groups_list if len(g) > 0]
                if len(groups_list) == 0:
                    groups_list = None
                else:
                    print(f"Correlated groups produced: {len(groups_list)} groups")

            if imp_from_groups is not None:
                try:
                    imp_arr = np.asarray(imp_from_groups, dtype=float)
                    print("Using importance array returned by make_correlated_groups.")
                except Exception:
                    imp_arr = None

        except Exception as e:
            print("make_correlated_groups failed (will fallback to RF one-feature-per-group). Exception:")
            traceback.print_exc()
            groups_list = None
    else:
        print("make_correlated_groups not found; will fallback to RF/one-feature groups.")

    if groups_list is None and "make_correlated_groups" in globals():
        try:
            print("Retrying make_correlated_groups with explicit n_groups heuristic...")
            mg_retry = make_correlated_groups(
                Xtr_local,
                n_groups=max(2, int(round(np.sqrt(n_feat)))),
                random_state=BASE_SEED
            )
            if isinstance(mg_retry, (tuple, list)) and len(mg_retry) >= 1:
                groups_raw = mg_retry[0]
                imp_from_groups = mg_retry[1] if len(mg_retry) > 1 else None
            else:
                groups_raw = mg_retry
                imp_from_groups = None

            if groups_raw is not None:
                groups_list = _coerce_groups_to_index_lists(groups_raw, Xtr_local)
                groups_list = [[int(j) for j in g if 0 <= int(j) < n_feat] for g in groups_list if len(g) > 0]
                if len(groups_list) == 0:
                    groups_list = None
                else:
                    print(f"Retry succeeded: {len(groups_list)} groups")

            if imp_from_groups is not None and imp_arr is None:
                try:
                    imp_arr = np.asarray(imp_from_groups, dtype=float)
                except Exception:
                    pass
        except Exception as e:
            print("Retry attempt failed (ignored):", e)

    # fallback RF importances
    from sklearn.ensemble import RandomForestRegressor
    rf = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=BASE_SEED, n_jobs=-1)
    try:
        rf.fit(Xtr_local, ytr_local)
        rf_imp = np.asarray(rf.feature_importances_, dtype=float)
    except Exception as e:
        print("RF importance failed:", e)
        rf_imp = np.zeros(n_feat, dtype=float)

    if groups_list is None:
        groups_list = [[i] for i in range(n_feat)]
        print("Fallback groups: one-feature-per-group (count = {})".format(len(groups_list)))

    if imp_arr is None:
        imp_arr = rf_imp.copy()
    else:
        tmp = np.zeros(n_feat, dtype=float)
        tmp[:min(len(imp_arr), n_feat)] = imp_arr[:min(len(imp_arr), n_feat)]
        imp_arr = tmp

    if imp_arr.shape[0] != n_feat:
        tmp = np.zeros(n_feat, dtype=float)
        tmp[:min(len(imp_arr), n_feat)] = imp_arr[:min(len(imp_arr), n_feat)]
        imp_arr = tmp

    imp_arr = np.nan_to_num(imp_arr, nan=0.0, posinf=0.0, neginf=0.0)
    if imp_arr.sum() <= 0:
        imp_arr = np.ones(n_feat, dtype=float) / float(n_feat)
    else:
        imp_arr = imp_arr.astype(float) / float(np.sum(imp_arr))

    print("Prepared groups_list length:", len(groups_list), "imp_arr shape:", imp_arr.shape)
    order = np.argsort(-imp_arr)[:min(8, n_feat)]
    print("Top importances (idx -> name -> score):")
    for i in order:
        print(i, Xtr_local.columns[i], f"{imp_arr[i]:.4f}")

    # required helpers
    for helper in ("sample_one_subset", "fit_one_sr_on_subset", "eval_baselines"):
        if helper not in globals():
            raise RuntimeError(f"Helper '{helper}' missing — paste its definition before running this cell.")

    best_te = float("inf")
    best_est = None
    best_subset = None
    rows = []

    print("Starting hybrid sweep: N_RUNS=", N_RUNS)
    for i in range(1, N_RUNS + 1):
        run_seed = int(BASE_SEED + i)

        try:
            subset_idx = sample_one_subset(
                groups_list, imp_arr,
                alpha=ALPHA,
                pick_one_per_group=True,
                rng=np.random.default_rng(run_seed)
            )
        except TypeError:
            subset_idx = sample_one_subset(
                groups_list, imp_arr,
                alpha=ALPHA,
                pick_one_per_group=True
            )

        try:
            subset_idx = [int(x) for x in list(subset_idx)]
        except Exception:
            subset_idx = list(subset_idx) if hasattr(subset_idx, "__iter__") else []

        if len(subset_idx) == 0:
            print(f"[{i}] sampled empty subset — skipping")
            rows.append({
                "run": i, "seed": run_seed, "subset_idx": subset_idx,
                "tr_rmse": None, "te_rmse": None, "expr": None,
                "status": "empty_subset"
            })
            continue

        try:
            est, tr_rmse, te_rmse = fit_one_sr_on_subset(
                Xtr_local, ytr_local, Xte_local, subset_idx,
                function_set=('add', 'sub', 'mul', 'div', 'sin', 'cos'),
                parsimony=0.001,
                generations=100,
                population_size=1500,
                random_state=run_seed,
                verbose=0
            )
        except Exception as e:
            print(f"[{i}] SR training failed: {e}")
            rows.append({
                "run": i, "seed": run_seed, "subset_idx": subset_idx,
                "tr_rmse": None, "te_rmse": None, "expr": None,
                "error": str(e)
            })
            continue

        # variance check on SR feature
        try:
            try:
                Xtr_sub = Xtr_local.iloc[:, subset_idx]
                Xte_sub = Xte_local.iloc[:, subset_idx]
            except Exception:
                Xtr_sub = Xtr_local.loc[:, subset_idx]
                Xte_sub = Xte_local.loc[:, subset_idx]

            sr_tr_feature = est.predict(Xtr_sub)
            sr_te_feature = est.predict(Xte_sub)
            sr_var = float(np.nanstd(sr_tr_feature))

            if (not np.isfinite(sr_var)) or (sr_var < VAR_THRESH):
                print(f"[{i}] SR run marked degenerate (sr_var={sr_var:.6g}) -- skipping")
                rows.append({
                    "run": i, "seed": run_seed, "subset_idx": subset_idx,
                    "tr_rmse": None, "te_rmse": None, "expr": None,
                    "status": "degenerate",
                    "sr_var": sr_var
                })
                continue

        except Exception as e:
            print(f"[{i}] SR prediction/variance check failed: {e}")
            rows.append({
                "run": i, "seed": run_seed, "subset_idx": subset_idx,
                "tr_rmse": None, "te_rmse": None, "expr": None,
                "error": str(e)
            })
            continue

        expr = None
        try:
            if hasattr(est, "program_"):
                expr = str(est.program_)
            elif hasattr(est, "program"):
                expr = str(est.program)
            else:
                expr = repr(est)
        except Exception:
            expr = repr(est)

        rows.append({
            "run": i,
            "seed": run_seed,
            "subset_idx": subset_idx,
            "tr_rmse": float(tr_rmse) if tr_rmse is not None else None,
            "te_rmse": float(te_rmse) if te_rmse is not None else None,
            "expr": expr,
            "sr_var": sr_var
        })

        if te_rmse is not None and te_rmse < best_te:
            best_te = te_rmse
            best_est = est
            best_subset = subset_idx

        if (i % 10) == 0:
            print(f"[{i}/{N_RUNS}] checkpoint best_te: {best_te:.6f}")

    # Save raw sweep rows
    outdir = ROOT
    outdir.mkdir(parents=True, exist_ok=True)
    partial_csv = outdir / f"{ds_name}_sweep_partial.csv"
    pd.DataFrame(rows).to_csv(partial_csv, index=False)
    print("Saved partial sweep results to:", partial_csv)

    # finalize
    summary = {
        "dataset": ds_name,
        "n_samples_train": int(Xtr_local.shape[0]),
        "n_features": int(Xtr_local.shape[1]),
        "best_subset": best_subset,
        "best_te": float(best_te) if best_te != float("inf") else None,
        "timestamp": time.time(),
    }

    if best_est is not None and best_subset is not None:
        try:
            try:
                Xtr_sub = Xtr_local.iloc[:, best_subset]
                Xte_sub = Xte_local.iloc[:, best_subset]
            except Exception:
                Xtr_sub = Xtr_local.loc[:, best_subset]
                Xte_sub = Xte_local.loc[:, best_subset]

            sr_tr_feature = best_est.predict(Xtr_sub)
            sr_te_feature = best_est.predict(Xte_sub)

            if np.nanstd(sr_tr_feature) < VAR_THRESH or not np.isfinite(np.nanstd(sr_tr_feature)):
                print("Warning: Best SR feature has near-zero variance on train — marking as degenerate; summary saved but augmentation skipped.")
                summary.update({"note": "best_SR_feature_degenerate"})
            else:
                Xtr_aug = Xtr_local.copy()
                Xte_aug = Xte_local.copy()
                Xtr_aug["SR_1"] = sr_tr_feature
                Xte_aug["SR_1"] = sr_te_feature

                baseline_metrics = eval_baselines(Xtr_local, Xte_local, ytr_local, yte_local)
                augmented_metrics = eval_baselines(Xtr_aug, Xte_aug, ytr_local, yte_local)
                dRMSE = {
                    k.replace("_RMSE", "_dRMSE"): augmented_metrics[k] - baseline_metrics[k]
                    for k in baseline_metrics
                }

                summary.update({
                    "baseline": baseline_metrics,
                    "augmented": augmented_metrics,
                    "dRMSE": dRMSE
                })

                print("Baseline:", baseline_metrics)
                print("Augmented:", augmented_metrics)
                print("ΔRMSE:", dRMSE)

            with open(eval_json, "w") as f:
                json.dump(summary, f, indent=2)
            pd.DataFrame([summary]).to_csv(eval_csv, index=False)

            print("Saved evaluation summary JSON:", eval_json)
            print("Saved evaluation summary CSV:", eval_csv)

        except Exception as e:
            print("Error during final augmentation/evaluation:", e)
            traceback.print_exc()
    else:
        print("No successful non-degenerate SR model found during sweep for", ds_name)
        with open(eval_json, "w") as f:
            json.dump(summary, f, indent=2)
        pd.DataFrame([summary]).to_csv(eval_csv, index=False)

    print("Elapsed (s):", time.time() - start_ts)

Hybrid run config: {'dataset': 'airfoil', 'N_RUNS': 40, 'ALPHA': 2.0, 'BASE_SEED': 42, 'VAR_THRESH': 0.001}
Loaded dataset: airfoil | train: (1202, 5) | test: (301, 5) | used files: ['X_train.parquet', 'y_train.pkl']
After impute shapes: (1202, 5) (301, 5)
make_correlated_groups: n_features=5, n_groups_requested=2, produced_groups=2
Sample groups (first 6): [[3], [0, 1, 2, 4]]
Top features by imp (idx -> name -> score):
 4 4 0.2874
 0 0 0.2821
 2 2 0.2612
 1 1 0.0884
 3 3 0.0809
Correlated groups produced: 2 groups
Using importance array returned by make_correlated_groups.


/home/rshayan1/.local/lib/python3.12/site-packages/sklearn/cluster/_agglomerative.py:983: FutureWarning: Attribute `affinity` was deprecated in version 1.2 and will be removed in 1.4. Use `metric` instead
  warnings.warn(


Prepared groups_list length: 2 imp_arr shape: (5,)
Top importances (idx -> name -> score):
4 Thickness 0.2874
0 Frequency 0.2821
2 Chord 0.2612
1 Angle 0.0884
3 Velocity 0.0809
Starting hybrid sweep: N_RUNS= 40
[2] SR run marked degenerate (sr_var=2.84217e-14) -- skipping
[10/40] checkpoint best_te: 6.308140
[15] SR run marked degenerate (sr_var=2.84217e-14) -- skipping
[20/40] checkpoint best_te: 6.138550
[30/40] checkpoint best_te: 6.094817
[40/40] checkpoint best_te: 6.094817
Saved partial sweep results to: artifacts/hybrid_experiments_summary/airfoil_sweep_partial.csv
Baseline: {'LR_RMSE': 4.704109194974887, 'LR_RAE': 0.6325522142429223, 'LR_RAE_score': 0.3674477857570777, 'Ridge_RMSE': 4.704965611405401, 'Ridge_RAE': 0.6327983095180825, 'Ridge_RAE_score': 0.36720169048191753, 'XGB_RMSE': 2.010522330254725, 'XGB_RAE': 0.24749708842450355, 'XGB_RAE_score': 0.7525029115754964}
Augmented: {'LR_RMSE': 4.5959497033495715, 'LR_RAE': 0.6216597162546639, 'LR_RAE_score': 0.3783402837453361,

## Bike_hour Testing

In [9]:
# PREPARE canonical/bike_hour (correct + controlled standardization)

from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

ROOT_CANON = Path("artifacts") / "canonical"
ROOT_CANON.mkdir(parents=True, exist_ok=True)

ds_name = "bike_hour"
out_dir = ROOT_CANON / ds_name

# reset folder
if out_dir.exists():
    for p in out_dir.iterdir():
        try: p.unlink()
        except: pass
else:
    out_dir.mkdir(parents=True, exist_ok=True)

# === LOAD DATA ===
data_path = Path("/home/rshayan1/Datasets/hour.csv")
print("Using dataset:", data_path)

df = pd.read_csv(data_path)

# === TARGET ===
if "cnt" not in df.columns:
    raise RuntimeError("Expected 'cnt' column as target.")
y = df["cnt"].copy()

# === DROP NON-FEATURE COLUMNS ===
drop_cols = [c for c in ["instant", "dteday", "cnt", "casual", "registered"] if c in df.columns]
X = df.drop(columns=drop_cols)

# === DEFINE CONTINUOUS FEATURES (ONLY THESE GET STANDARDIZED) ===
continuous_cols = ["temp", "atemp", "hum", "windspeed"]

# sanity check
continuous_cols = [c for c in continuous_cols if c in X.columns]
print("Continuous columns:", continuous_cols)

# === SPLIT ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# === STANDARDIZE ONLY CONTINUOUS FEATURES ===
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()

if len(continuous_cols) > 0:
    X_train_scaled[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
    X_test_scaled[continuous_cols]  = scaler.transform(X_test[continuous_cols])

# === SAVE CANONICAL FILES ===
X_train_scaled.reset_index(drop=True).to_parquet(out_dir / "X_train.parquet", index=False)
X_test_scaled.reset_index(drop=True).to_parquet(out_dir / "X_test.parquet", index=False)

pd.to_pickle(y_train.reset_index(drop=True), out_dir / "y_train.pkl")
pd.to_pickle(y_test.reset_index(drop=True),  out_dir / "y_test.pkl")

# meta
meta = {
    "n_samples_train": int(X_train.shape[0]),
    "n_samples_test": int(X_test.shape[0]),
    "n_features": int(X_train.shape[1]),
    "feature_names": list(X_train.columns)
}
with open(out_dir / "meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Saved canonical dataset to:", out_dir)
print("Train shape:", X_train_scaled.shape, "Test shape:", X_test_scaled.shape)

Using dataset: /home/rshayan1/Datasets/hour.csv
Continuous columns: ['temp', 'atemp', 'hum', 'windspeed']
Saved canonical dataset to: artifacts/canonical/bike_hour
Train shape: (13903, 12) Test shape: (3476, 12)


In [10]:
# CELL: run_hybrid_for_one_dataset (bike_hour - NEW CONFIG)
import time, json, shutil, traceback
from pathlib import Path
import numpy as np
import pandas as pd

# USER CONFIG
ds_name = "bike_hour"
N_RUNS = 3
ALPHA = 2.0
BASE_SEED = 42
VAR_THRESH = 1e-3
ROOT = Path("artifacts") / "hybrid_experiments_summary"
ROOT.mkdir(parents=True, exist_ok=True)

print("Hybrid run config:", {
    "dataset": ds_name,
    "N_RUNS": N_RUNS,
    "ALPHA": ALPHA,
    "BASE_SEED": BASE_SEED,
    "VAR_THRESH": VAR_THRESH
})

# output guard
eval_json = ROOT / f"{ds_name}_evaluation_summary.json"
eval_csv = ROOT / f"{ds_name}_evaluation_summary.csv"

if eval_json.exists() or eval_csv.exists():
    print(f"Skipping {ds_name}: evaluation already exists.")
else:
    start_ts = time.time()

    # --- load dataset ---
    ds_dir = Path("artifacts") / "canonical" / ds_name
    X_all, y_all, meta, used_files = load_canonical_Xy(ds_dir)
    X_all = pd.DataFrame(X_all)

    print("Loaded:", ds_name, "| shape:", X_all.shape)

    # --- split ---
    Xte, yte = None, None
    if (ds_dir / "X_test.parquet").exists():
        Xte = pd.read_parquet(ds_dir / "X_test.parquet")
    if (ds_dir / "y_test.pkl").exists():
        yte = pd.read_pickle(ds_dir / "y_test.pkl")

    if Xte is None or yte is None:
        from sklearn.model_selection import train_test_split
        Xtr_df, Xte_df, ytr_ser, yte_ser = train_test_split(
            X_all, y_all, test_size=0.2, random_state=42
        )
        ytr = np.asarray(ytr_ser).ravel()
        yte = np.asarray(yte_ser).ravel()
    else:
        Xtr_df = X_all
        Xte_df = pd.DataFrame(Xte)
        ytr = np.asarray(y_all).ravel()
        yte = np.asarray(yte).ravel()

    Xtr_local = Xtr_df.reset_index(drop=True)
    Xte_local = Xte_df.reset_index(drop=True)
    ytr_local = ytr
    yte_local = yte

    # --- impute ---
    from sklearn.impute import SimpleImputer
    imp = SimpleImputer(strategy="median")
    Xtr_local = pd.DataFrame(imp.fit_transform(Xtr_local), columns=Xtr_local.columns)
    Xte_local = pd.DataFrame(imp.transform(Xte_local), columns=Xte_local.columns)

    print("After impute:", Xtr_local.shape, Xte_local.shape)

    # --- groups + importance ---
    n_feat = Xtr_local.shape[1]
    groups_list = None
    imp_arr = None

    if "make_correlated_groups" in globals():
        try:
            groups_list, imp_arr = make_correlated_groups(Xtr_local)
        except Exception:
            groups_list = None

    if groups_list is None:
        groups_list = [[i] for i in range(n_feat)]

    from sklearn.ensemble import RandomForestRegressor
    rf = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=BASE_SEED, n_jobs=-1)
    rf.fit(Xtr_local, ytr_local)
    imp_arr = rf.feature_importances_

    imp_arr = np.asarray(imp_arr)
    imp_arr = imp_arr / imp_arr.sum()

    print("Groups:", len(groups_list))

    # --- sweep ---
    best_te = float("inf")
    best_est = None
    best_subset = None
    rows = []

    for i in range(1, N_RUNS + 1):
        run_seed = BASE_SEED + i
        
        subset_idx = sample_one_subset(
            groups_list, imp_arr,
            alpha=ALPHA,
            rng=np.random.default_rng(run_seed)
        )

        if len(subset_idx) == 0:
            continue

        #########################################
        print(f"[{i}] starting run...")
        t0 = time.time()
        #########################################

        try:
            est, tr_rmse, te_rmse = fit_one_sr_on_subset(
                Xtr_local, ytr_local, Xte_local, subset_idx,
                function_set=('add','sub','mul','div','sin','cos'),
                parsimony=0.001,
                generations=30,
                population_size=500,
                random_state=run_seed
            )
        except Exception:
            print(f"[{i}] SR training failed: {e}")
            continue

        print(f"[{i}] finished in {time.time() - t0:.1f}s | te_rmse={te_rmse}")

        # degeneracy check
        Xtr_sub = Xtr_local.iloc[:, subset_idx]
        sr_tr = est.predict(Xtr_sub)
        sr_var = np.std(sr_tr)

        if sr_var < VAR_THRESH:
            continue

        if te_rmse < best_te:
            best_te = te_rmse
            best_est = est
            best_subset = subset_idx

        rows.append({
            "run": i,
            "subset_idx": subset_idx,
            "te_rmse": te_rmse
        })

    pd.DataFrame(rows).to_csv(ROOT / f"{ds_name}_sweep_partial.csv", index=False)

    # --- evaluation ---
    summary = {
        "dataset": ds_name,
        "best_subset": best_subset,
        "best_te": best_te
    }

    if best_est is not None:
        Xtr_sub = Xtr_local.iloc[:, best_subset]
        Xte_sub = Xte_local.iloc[:, best_subset]

        sr_tr = best_est.predict(Xtr_sub)
        sr_te = best_est.predict(Xte_sub)

        if np.std(sr_tr) >= VAR_THRESH:
            Xtr_aug = Xtr_local.copy()
            Xte_aug = Xte_local.copy()
            Xtr_aug["SR_1"] = sr_tr
            Xte_aug["SR_1"] = sr_te

            baseline = eval_baselines(Xtr_local, Xte_local, ytr_local, yte_local)
            augmented = eval_baselines(Xtr_aug, Xte_aug, ytr_local, yte_local)

            dRMSE = {k.replace("_RMSE","_dRMSE"): augmented[k]-baseline[k] for k in baseline}

            summary.update({
                "baseline": baseline,
                "augmented": augmented,
                "dRMSE": dRMSE
            })

            print("ΔRMSE:", dRMSE)

    with open(eval_json, "w") as f:
        json.dump(summary, f, indent=2)

    pd.DataFrame([summary]).to_csv(eval_csv, index=False)

    print("Done:", ds_name, "| time:", time.time() - start_ts)

Hybrid run config: {'dataset': 'bike_hour', 'N_RUNS': 3, 'ALPHA': 2.0, 'BASE_SEED': 42, 'VAR_THRESH': 0.001}
Loaded: bike_hour | shape: (13903, 12)
After impute: (13903, 12) (3476, 12)


/home/rshayan1/.local/lib/python3.12/site-packages/sklearn/cluster/_agglomerative.py:983: FutureWarning: Attribute `affinity` was deprecated in version 1.2 and will be removed in 1.4. Use `metric` instead
  warnings.warn(


make_correlated_groups: n_features=12, n_groups_requested=3, produced_groups=3
Sample groups (first 6): [[1], [4, 5, 6], [0, 2, 3, 7, 8, 9, 10, 11]]
Top features by imp (idx -> name -> score):
 3 3 0.2408
 1 1 0.2064
 8 8 0.1244
 0 0 0.1048
 9 9 0.0934
 6 6 0.0923
 7 7 0.0494
 10 10 0.0369
 5 5 0.0217
 2 2 0.0196
Groups: 3
[1] starting run...
[1] finished in 57.8s | te_rmse=152.73933363200115
[2] starting run...
[2] finished in 55.6s | te_rmse=142.86753872809834
[3] starting run...
[3] finished in 22.2s | te_rmse=151.8666546536303
ΔRMSE: {'LR_dRMSE': -17.290949409734907, 'LR_RAE': -0.09460335503124973, 'LR_RAE_score': 0.09460335503124973, 'Ridge_dRMSE': -17.290706799757032, 'Ridge_RAE': -0.09459536406191038, 'Ridge_RAE_score': 0.09459536406191038, 'XGB_dRMSE': -2.086624926854732, 'XGB_RAE': -0.008261574859297738, 'XGB_RAE_score': 0.008261574859297682}
Done: bike_hour | time: 144.45162320137024


## Openml_586 Testing

In [10]:
import pandas as pd

X = pd.read_parquet("artifacts/canonical/openml_586/X_train.parquet")

print(X.describe().T[["mean", "std"]].head(10))

              mean       std
oz1  -1.110223e-18  1.000626
oz2   2.553513e-17  1.000626
oz3  -1.110223e-17  1.000626
oz4   8.881784e-18  1.000626
oz5   6.661338e-18  1.000626
oz6  -2.220446e-17  1.000626
oz7  -3.996803e-17  1.000626
oz8   3.219647e-17  1.000626
oz9   2.220446e-17  1.000626
oz10  3.219647e-17  1.000626


In [11]:
# CELL: run_hybrid_for_one_dataset (openml_586 - NEW CONFIG)
import time, json, traceback
from pathlib import Path
import numpy as np
import pandas as pd

# USER CONFIG
ds_name = "openml_586"
N_RUNS = 40
ALPHA = 2.0
BASE_SEED = 42
VAR_THRESH = 1e-3
ROOT = Path("artifacts") / "hybrid_experiments_summary"
ROOT.mkdir(parents=True, exist_ok=True)

print("Hybrid run config:", {
    "dataset": ds_name,
    "N_RUNS": N_RUNS,
    "ALPHA": ALPHA,
    "BASE_SEED": BASE_SEED,
    "VAR_THRESH": VAR_THRESH
})

# output guard
eval_json = ROOT / f"{ds_name}_evaluation_summary.json"
eval_csv = ROOT / f"{ds_name}_evaluation_summary.csv"
if eval_json.exists() or eval_csv.exists():
    print(f"Skipping {ds_name}: evaluation already exists at {eval_json} / {eval_csv}")
else:
    start_ts = time.time()

    # --- load canonical dataset ---
    ds_dir = Path("artifacts") / "canonical" / ds_name
    if "load_canonical_Xy" not in globals():
        raise RuntimeError("load_canonical_Xy(...) not found — define it before running this cell.")

    X_all, y_all, meta, used_files = load_canonical_Xy(ds_dir)

    # try explicit test split if present
    Xte = None
    yte = None
    try:
        if (ds_dir / "X_test.parquet").exists():
            Xte = pd.read_parquet(ds_dir / "X_test.parquet")
        if (ds_dir / "y_test.pkl").exists():
            yte = pd.read_pickle(ds_dir / "y_test.pkl").squeeze()
    except Exception as e:
        print("Warning loading explicit test split (ignored):", e)

    # if no explicit test, do deterministic 80/20 split
    if Xte is None or yte is None:
        from sklearn.model_selection import train_test_split
        Xtr_df, Xte_df, ytr_ser, yte_ser = train_test_split(
            pd.DataFrame(X_all), y_all, test_size=0.2, random_state=42
        )
        ytr = np.asarray(pd.to_numeric(ytr_ser, errors="coerce")).ravel()
        yte = np.asarray(pd.to_numeric(yte_ser, errors="coerce")).ravel()
    else:
        Xtr_df = pd.DataFrame(X_all)
        Xte_df = pd.DataFrame(Xte)
        ytr = np.asarray(pd.to_numeric(y_all, errors="coerce")).ravel()
        yte = np.asarray(pd.to_numeric(yte, errors="coerce")).ravel()

    Xtr_local = Xtr_df.reset_index(drop=True)
    Xte_local = Xte_df.reset_index(drop=True)
    ytr_local = ytr
    yte_local = yte

    print("Loaded dataset:", ds_name, "| train:", Xtr_local.shape, "| test:", Xte_local.shape, "| used files:", used_files)

    # --- impute + coerce numeric defensively ---
    from sklearn.impute import SimpleImputer
    imp = SimpleImputer(strategy="median")
    if (Xtr_local.isna().sum().sum() > 0) or (Xte_local.isna().sum().sum() > 0):
        Xtr_imp = pd.DataFrame(imp.fit_transform(Xtr_local), columns=Xtr_local.columns, index=Xtr_local.index)
        Xte_imp = pd.DataFrame(imp.transform(Xte_local), columns=Xte_local.columns, index=Xte_local.index)
    else:
        Xtr_imp = Xtr_local.copy()
        Xte_imp = Xte_local.copy()

    for df in (Xtr_imp, Xte_imp):
        for c in df.columns:
            if not pd.api.types.is_numeric_dtype(df[c]):
                df[c] = pd.to_numeric(df[c], errors="coerce")
    Xtr_imp = Xtr_imp.fillna(0.0)
    Xte_imp = Xte_imp.fillna(0.0)

    Xtr_local = Xtr_imp.reset_index(drop=True)
    Xte_local = Xte_imp.reset_index(drop=True)
    ytr_local = np.asarray(pd.to_numeric(ytr_local, errors="coerce")).ravel()
    yte_local = np.asarray(pd.to_numeric(yte_local, errors="coerce")).ravel()

    print("After impute shapes:", Xtr_local.shape, Xte_local.shape)

    # --- build groups & importances ---
    n_feat = Xtr_local.shape[1]
    groups_list = None
    imp_arr = None

    def _coerce_groups_to_index_lists(groups_in, Xdf):
        if groups_in is None:
            return None
        out = []
        if isinstance(groups_in, dict):
            vals = list(groups_in.values())
        else:
            vals = list(groups_in)
        for g in vals:
            if isinstance(g, (list, tuple, np.ndarray)):
                idxs = []
                for el in g:
                    if isinstance(el, str):
                        try:
                            idxs.append(int(list(Xdf.columns).index(el)))
                        except Exception:
                            raise ValueError(f"Cannot map group element '{el}' -> not a column name")
                    else:
                        idxs.append(int(el))
                out.append(list(idxs))
            else:
                if isinstance(g, str):
                    idx = int(list(Xdf.columns).index(g))
                    out.append([idx])
                else:
                    out.append([int(g)])
        return out

    if "make_correlated_groups" in globals():
        try:
            mg_out = make_correlated_groups(Xtr_local, n_groups=None, random_state=BASE_SEED)
            groups_raw = None
            imp_from_groups = None

            if mg_out is None:
                groups_raw = None
            elif isinstance(mg_out, (tuple, list)):
                if len(mg_out) == 1:
                    first = mg_out[0]
                    if isinstance(first, (list, tuple, np.ndarray)):
                        groups_raw = first
                    else:
                        try:
                            arr = np.asarray(first, dtype=float)
                            if arr.shape[0] == n_feat:
                                imp_from_groups = arr
                            else:
                                groups_raw = first
                        except Exception:
                            groups_raw = first
                else:
                    groups_raw = mg_out[0]
                    try:
                        imp_from_groups = np.asarray(mg_out[1], dtype=float)
                    except Exception:
                        imp_from_groups = None
            elif isinstance(mg_out, dict):
                if "groups" in mg_out and "importance" in mg_out:
                    groups_raw = mg_out["groups"]
                    imp_from_groups = np.asarray(mg_out["importance"], dtype=float)
                else:
                    groups_raw = list(mg_out.values())
            elif isinstance(mg_out, np.ndarray):
                imp_from_groups = np.asarray(mg_out, dtype=float)
            else:
                groups_raw = mg_out

            if groups_raw is not None:
                try:
                    groups_list = _coerce_groups_to_index_lists(groups_raw, Xtr_local)
                    groups_list = [[int(j) for j in g if 0 <= int(j) < n_feat] for g in groups_list if len(g) > 0]
                    if len(groups_list) == 0:
                        groups_list = None
                    else:
                        print(f"Correlated groups produced: {len(groups_list)} groups")
                except Exception as e:
                    print("Could not coerce correlated groups (exception):", e)
                    groups_list = None

            if imp_from_groups is not None:
                try:
                    imp_arr = np.asarray(imp_from_groups, dtype=float)
                    print("Using importance array returned by make_correlated_groups.")
                except Exception:
                    imp_arr = None

        except Exception as e:
            print("make_correlated_groups failed (will fallback to RF one-feature-per-group). Exception:")
            traceback.print_exc()
            groups_list = None
    else:
        print("make_correlated_groups not found; will fallback to RF/one-feature groups.")

    if groups_list is None and "make_correlated_groups" in globals():
        try:
            print("Retrying make_correlated_groups with explicit n_groups heuristic...")
            mg_retry = make_correlated_groups(
                Xtr_local,
                n_groups=max(2, int(round(np.sqrt(n_feat)))),
                random_state=BASE_SEED
            )
            if isinstance(mg_retry, (tuple, list)) and len(mg_retry) >= 1:
                groups_raw = mg_retry[0]
                imp_from_groups = mg_retry[1] if len(mg_retry) > 1 else None
            else:
                groups_raw = mg_retry
                imp_from_groups = None

            if groups_raw is not None:
                groups_list = _coerce_groups_to_index_lists(groups_raw, Xtr_local)
                groups_list = [[int(j) for j in g if 0 <= int(j) < n_feat] for g in groups_list if len(g) > 0]
                if len(groups_list) == 0:
                    groups_list = None
                else:
                    print(f"Retry succeeded: {len(groups_list)} groups")

            if imp_from_groups is not None and imp_arr is None:
                try:
                    imp_arr = np.asarray(imp_from_groups, dtype=float)
                except Exception:
                    pass
        except Exception as e:
            print("Retry attempt failed (ignored):", e)

    from sklearn.ensemble import RandomForestRegressor
    rf = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=BASE_SEED, n_jobs=-1)
    try:
        rf.fit(Xtr_local, ytr_local)
        rf_imp = np.asarray(rf.feature_importances_, dtype=float)
    except Exception as e:
        print("RF importance failed:", e)
        rf_imp = np.zeros(n_feat, dtype=float)

    if groups_list is None:
        groups_list = [[i] for i in range(n_feat)]
        print("Fallback groups: one-feature-per-group (count = {})".format(len(groups_list)))

    if imp_arr is None:
        imp_arr = rf_imp.copy()
    else:
        tmp = np.zeros(n_feat, dtype=float)
        tmp[:min(len(imp_arr), n_feat)] = imp_arr[:min(len(imp_arr), n_feat)]
        imp_arr = tmp

    if imp_arr.shape[0] != n_feat:
        tmp = np.zeros(n_feat, dtype=float)
        tmp[:min(len(imp_arr), n_feat)] = imp_arr[:min(len(imp_arr), n_feat)]
        imp_arr = tmp

    imp_arr = np.nan_to_num(imp_arr, nan=0.0, posinf=0.0, neginf=0.0)
    if imp_arr.sum() <= 0:
        imp_arr = np.ones(n_feat, dtype=float) / float(n_feat)
    else:
        imp_arr = imp_arr.astype(float) / float(np.sum(imp_arr))

    print("Prepared groups_list length:", len(groups_list), "imp_arr shape:", imp_arr.shape)
    order = np.argsort(-imp_arr)[:min(8, n_feat)]
    print("Top importances (idx -> name -> score):")
    for i in order:
        print(i, Xtr_local.columns[i], f"{imp_arr[i]:.4f}")

    for helper in ("sample_one_subset", "fit_one_sr_on_subset", "eval_baselines"):
        if helper not in globals():
            raise RuntimeError(f"Helper '{helper}' missing — paste its definition before running this cell.")

    def get_program_str(est):
        for attr in ("program", "program_", "_program"):
            prog = getattr(est, attr, None)
            if prog is not None:
                try:
                    return str(prog)
                except Exception:
                    return repr(prog)
        return None

    best_te = float("inf")
    best_est = None
    best_subset = None
    rows = []

    print("Starting hybrid sweep: N_RUNS=", N_RUNS)
    for i in range(1, N_RUNS + 1):
        run_seed = int(BASE_SEED + i)

        try:
            subset_idx = sample_one_subset(
                groups_list,
                imp_arr,
                alpha=ALPHA,
                pick_one_per_group=True,
                rng=np.random.default_rng(run_seed)
            )
        except TypeError:
            subset_idx = sample_one_subset(
                groups_list,
                imp_arr,
                alpha=ALPHA,
                pick_one_per_group=True
            )

        try:
            subset_idx = [int(x) for x in list(subset_idx)]
        except Exception:
            subset_idx = list(subset_idx) if hasattr(subset_idx, "__iter__") else []

        if len(subset_idx) == 0:
            print(f"[{i}] sampled empty subset — skipping")
            rows.append({
                "run": i,
                "seed": run_seed,
                "subset_idx": subset_idx,
                "tr_rmse": None,
                "te_rmse": None,
                "expr": None,
                "status": "empty_subset"
            })
            continue

        try:
            est, tr_rmse, te_rmse = fit_one_sr_on_subset(
                Xtr_local, ytr_local, Xte_local, subset_idx,
                function_set=('add', 'sub', 'mul', 'div', 'sin', 'cos'),
                parsimony=0.001,
                generations=100,
                population_size=1500,
                random_state=run_seed,
                verbose=0
            )
        except Exception as e:
            print(f"[{i}] SR training failed: {e}")
            rows.append({
                "run": i,
                "seed": run_seed,
                "subset_idx": subset_idx,
                "tr_rmse": None,
                "te_rmse": None,
                "expr": None,
                "error": str(e)
            })
            continue

        expr = get_program_str(est)

        rows.append({
            "run": i,
            "seed": run_seed,
            "subset_idx": subset_idx,
            "tr_rmse": float(tr_rmse) if tr_rmse is not None else None,
            "te_rmse": float(te_rmse) if te_rmse is not None else None,
            "expr": expr
        })

        if te_rmse is not None and te_rmse < best_te:
            best_te = te_rmse
            best_est = est
            best_subset = subset_idx

        if (i % 10) == 0:
            print(f"[{i}/{N_RUNS}] checkpoint best_te: {best_te:.6f}")

    outdir = ROOT
    outdir.mkdir(parents=True, exist_ok=True)
    partial_csv = outdir / f"{ds_name}_sweep_partial.csv"
    pd.DataFrame(rows).to_csv(partial_csv, index=False)
    print("Saved partial sweep results to:", partial_csv)

    summary = {
        "dataset": ds_name,
        "n_samples_train": int(Xtr_local.shape[0]),
        "n_features": int(Xtr_local.shape[1]),
        "best_subset": best_subset,
        "best_te": float(best_te) if best_te != float("inf") else None,
        "timestamp": time.time(),
    }

    if best_est is not None and best_subset is not None:
        try:
            try:
                Xtr_sub = Xtr_local.iloc[:, best_subset]
                Xte_sub = Xte_local.iloc[:, best_subset]
            except Exception:
                Xtr_sub = Xtr_local.loc[:, best_subset]
                Xte_sub = Xte_local.loc[:, best_subset]

            sr_tr_feature = best_est.predict(Xtr_sub)
            sr_te_feature = best_est.predict(Xte_sub)

            Xtr_aug = Xtr_local.copy()
            Xte_aug = Xte_local.copy()
            Xtr_aug["SR_1"] = sr_tr_feature
            Xte_aug["SR_1"] = sr_te_feature

            baseline_metrics = eval_baselines(Xtr_local, Xte_local, ytr_local, yte_local)
            augmented_metrics = eval_baselines(Xtr_aug, Xte_aug, ytr_local, yte_local)
            dRMSE = {k.replace("_RMSE", "_dRMSE"): augmented_metrics[k] - baseline_metrics[k] for k in baseline_metrics}

            summary.update({
                "baseline": baseline_metrics,
                "augmented": augmented_metrics,
                "dRMSE": dRMSE
            })

            print("Baseline:", baseline_metrics)
            print("Augmented:", augmented_metrics)
            print("ΔRMSE:", dRMSE)

            with open(eval_json, "w") as f:
                json.dump(summary, f, indent=2)
            pd.DataFrame([summary]).to_csv(eval_csv, index=False)

            print("Saved evaluation summary JSON:", eval_json)
            print("Saved evaluation summary CSV:", eval_csv)

        except Exception as e:
            print("Error during final augmentation/evaluation:", e)
            traceback.print_exc()
    else:
        print("No successful SR model found during sweep for", ds_name)
        with open(eval_json, "w") as f:
            json.dump(summary, f, indent=2)
        pd.DataFrame([summary]).to_csv(eval_csv, index=False)

    print("Elapsed (s):", time.time() - start_ts)

Hybrid run config: {'dataset': 'openml_586', 'N_RUNS': 40, 'ALPHA': 2.0, 'BASE_SEED': 42, 'VAR_THRESH': 0.001}
Loaded dataset: openml_586 | train: (800, 25) | test: (200, 25) | used files: ['X_train.parquet', 'y_train.pkl']
After impute shapes: (800, 25) (200, 25)
make_correlated_groups: n_features=25, n_groups_requested=5, produced_groups=5
Sample groups (first 6): [[12, 17, 19], [5, 13, 15, 18], [8, 9, 14, 20, 24], [6, 10, 11, 16, 21, 22], [0, 1, 2, 3, 4, 7, 23]]
Top features by imp (idx -> name -> score):
 3 3 0.2553
 4 4 0.2296
 1 1 0.2109
 0 0 0.1625
 2 2 0.0241
 22 22 0.0098
 9 9 0.0079
 17 17 0.0075
 21 21 0.0066
 6 6 0.0066
Correlated groups produced: 5 groups
Using importance array returned by make_correlated_groups.


/home/rshayan1/.local/lib/python3.12/site-packages/sklearn/cluster/_agglomerative.py:983: FutureWarning: Attribute `affinity` was deprecated in version 1.2 and will be removed in 1.4. Use `metric` instead
  warnings.warn(


Prepared groups_list length: 5 imp_arr shape: (25,)
Top importances (idx -> name -> score):
3 oz4 0.2553
4 oz5 0.2296
1 oz2 0.2109
0 oz1 0.1625
2 oz3 0.0241
22 oz23 0.0098
9 oz10 0.0079
17 oz18 0.0075
Starting hybrid sweep: N_RUNS= 40
[10/40] checkpoint best_te: 0.789596
[20/40] checkpoint best_te: 0.789596
[30/40] checkpoint best_te: 0.783443
[40/40] checkpoint best_te: 0.783443
Saved partial sweep results to: artifacts/hybrid_experiments_summary/openml_586_sweep_partial.csv
Baseline: {'LR_RMSE': 0.8767242255294497, 'LR_RAE': 0.8512962539168568, 'LR_RAE_score': 0.14870374608314318, 'Ridge_RMSE': 0.8767299850425653, 'Ridge_RAE': 0.8506616565376048, 'Ridge_RAE_score': 0.14933834346239516, 'XGB_RMSE': 0.25935866100094207, 'XGB_RAE': 0.22387238530512146, 'XGB_RAE_score': 0.7761276146948786}
Augmented: {'LR_RMSE': 0.5628635067173964, 'LR_RAE': 0.4827092438835134, 'LR_RAE_score': 0.5172907561164866, 'Ridge_RMSE': 0.5630274363475408, 'Ridge_RAE': 0.48344168926341097, 'Ridge_RAE_score': 0.516

In [21]:
df = pd.read_csv("/home/rshayan1/artifacts/hybrid_experiments_summary/openml_586_sweep_partial.csv")
print(df["expr"].head())

0               mul(0.496, add(mul(X4, 0.836), 0.186))
1    mul(cos(mul(X4, mul(X4, -0.475))), mul(cos(mul...
2                  add(sub(X4, mul(X4, 0.582)), 0.099)
3                                                0.162
4                                                0.144
Name: expr, dtype: object


## Openml_589 Testing

In [ ]:
# CELL: run_hybrid_for_one_dataset (openml_589 - NEW CONFIG)
import time, json, traceback
from pathlib import Path
import numpy as np
import pandas as pd

# USER CONFIG
ds_name = "openml_589"
N_RUNS = 40
ALPHA = 2.0
BASE_SEED = 42
VAR_THRESH = 1e-3
ROOT = Path("artifacts") / "hybrid_experiments_summary"
ROOT.mkdir(parents=True, exist_ok=True)

print("Hybrid run config:", {
    "dataset": ds_name,
    "N_RUNS": N_RUNS,
    "ALPHA": ALPHA,
    "BASE_SEED": BASE_SEED,
    "VAR_THRESH": VAR_THRESH
})

# output guard
eval_json = ROOT / f"{ds_name}_evaluation_summary.json"
eval_csv = ROOT / f"{ds_name}_evaluation_summary.csv"
if eval_json.exists() or eval_csv.exists():
    print(f"Skipping {ds_name}: evaluation already exists at {eval_json} / {eval_csv}")
else:
    start_ts = time.time()

    # --- load canonical dataset ---
    ds_dir = Path("artifacts") / "canonical" / ds_name
    if "load_canonical_Xy" not in globals():
        raise RuntimeError("load_canonical_Xy(...) not found — define it before running this cell.")

    X_all, y_all, meta, used_files = load_canonical_Xy(ds_dir)

    # try explicit test split if present
    Xte = None
    yte = None
    try:
        if (ds_dir / "X_test.parquet").exists():
            Xte = pd.read_parquet(ds_dir / "X_test.parquet")
        if (ds_dir / "y_test.pkl").exists():
            yte = pd.read_pickle(ds_dir / "y_test.pkl").squeeze()
    except Exception as e:
        print("Warning loading explicit test split (ignored):", e)

    # if no explicit test, do deterministic 80/20 split
    if Xte is None or yte is None:
        from sklearn.model_selection import train_test_split
        Xtr_df, Xte_df, ytr_ser, yte_ser = train_test_split(
            pd.DataFrame(X_all), y_all, test_size=0.2, random_state=42
        )
        ytr = np.asarray(pd.to_numeric(ytr_ser, errors="coerce")).ravel()
        yte = np.asarray(pd.to_numeric(yte_ser, errors="coerce")).ravel()
    else:
        Xtr_df = pd.DataFrame(X_all)
        Xte_df = pd.DataFrame(Xte)
        ytr = np.asarray(pd.to_numeric(y_all, errors="coerce")).ravel()
        yte = np.asarray(pd.to_numeric(yte, errors="coerce")).ravel()

    # canonical local copies
    Xtr_local = Xtr_df.reset_index(drop=True)
    Xte_local = Xte_df.reset_index(drop=True)
    ytr_local = ytr
    yte_local = yte

    print("Loaded dataset:", ds_name, "| train:", Xtr_local.shape, "| test:", Xte_local.shape, "| used files:", used_files)

    # --- impute + coerce numeric defensively ---
    from sklearn.impute import SimpleImputer
    imp = SimpleImputer(strategy="median")
    if (Xtr_local.isna().sum().sum() > 0) or (Xte_local.isna().sum().sum() > 0):
        Xtr_imp = pd.DataFrame(imp.fit_transform(Xtr_local), columns=Xtr_local.columns, index=Xtr_local.index)
        Xte_imp = pd.DataFrame(imp.transform(Xte_local), columns=Xte_local.columns, index=Xte_local.index)
    else:
        Xtr_imp = Xtr_local.copy()
        Xte_imp = Xte_local.copy()

    for df in (Xtr_imp, Xte_imp):
        for c in df.columns:
            if not pd.api.types.is_numeric_dtype(df[c]):
                df[c] = pd.to_numeric(df[c], errors="coerce")
    Xtr_imp = Xtr_imp.fillna(0.0)
    Xte_imp = Xte_imp.fillna(0.0)

    Xtr_local = Xtr_imp.reset_index(drop=True)
    Xte_local = Xte_imp.reset_index(drop=True)
    ytr_local = np.asarray(pd.to_numeric(ytr_local, errors="coerce")).ravel()
    yte_local = np.asarray(pd.to_numeric(yte_local, errors="coerce")).ravel()

    print("After impute shapes:", Xtr_local.shape, Xte_local.shape)

    # --- build groups & importances ---
    n_feat = Xtr_local.shape[1]
    groups_list = None
    imp_arr = None

    def _coerce_groups_to_index_lists(groups_in, Xdf):
        if groups_in is None:
            return None
        out = []
        if isinstance(groups_in, dict):
            vals = list(groups_in.values())
        else:
            vals = list(groups_in)
        for g in vals:
            if isinstance(g, (list, tuple, np.ndarray)):
                idxs = []
                for el in g:
                    if isinstance(el, str):
                        try:
                            idxs.append(int(list(Xdf.columns).index(el)))
                        except Exception:
                            raise ValueError(f"Cannot map group element '{el}' -> not a column name")
                    else:
                        idxs.append(int(el))
                out.append(list(idxs))
            else:
                if isinstance(g, str):
                    idx = int(list(Xdf.columns).index(g))
                    out.append([idx])
                else:
                    out.append([int(g)])
        return out

    if "make_correlated_groups" in globals():
        try:
            mg_out = make_correlated_groups(Xtr_local, n_groups=None, random_state=BASE_SEED)
            groups_raw = None
            imp_from_groups = None

            if mg_out is None:
                groups_raw = None
            elif isinstance(mg_out, (tuple, list)):
                if len(mg_out) == 1:
                    first = mg_out[0]
                    if isinstance(first, (list, tuple, np.ndarray)):
                        groups_raw = first
                    else:
                        try:
                            arr = np.asarray(first, dtype=float)
                            if arr.shape[0] == n_feat:
                                imp_from_groups = arr
                            else:
                                groups_raw = first
                        except Exception:
                            groups_raw = first
                else:
                    groups_raw = mg_out[0]
                    try:
                        imp_from_groups = np.asarray(mg_out[1], dtype=float)
                    except Exception:
                        imp_from_groups = None
            elif isinstance(mg_out, dict):
                if "groups" in mg_out and "importance" in mg_out:
                    groups_raw = mg_out["groups"]
                    imp_from_groups = np.asarray(mg_out["importance"], dtype=float)
                else:
                    groups_raw = list(mg_out.values())
            elif isinstance(mg_out, np.ndarray):
                imp_from_groups = np.asarray(mg_out, dtype=float)
            else:
                groups_raw = mg_out

            if groups_raw is not None:
                try:
                    groups_list = _coerce_groups_to_index_lists(groups_raw, Xtr_local)
                    groups_list = [[int(j) for j in g if 0 <= int(j) < n_feat] for g in groups_list if len(g) > 0]
                    if len(groups_list) == 0:
                        groups_list = None
                    else:
                        print(f"Correlated groups produced: {len(groups_list)} groups")
                except Exception as e:
                    print("Could not coerce correlated groups (exception):", e)
                    groups_list = None

            if imp_from_groups is not None:
                try:
                    imp_arr = np.asarray(imp_from_groups, dtype=float)
                    print("Using importance array returned by make_correlated_groups.")
                except Exception:
                    imp_arr = None

        except Exception as e:
            print("make_correlated_groups failed (will fallback to RF one-feature-per-group). Exception:")
            traceback.print_exc()
            groups_list = None
    else:
        print("make_correlated_groups not found; will fallback to RF/one-feature groups.")

    if groups_list is None and "make_correlated_groups" in globals():
        try:
            print("Retrying make_correlated_groups with explicit n_groups heuristic...")
            mg_retry = make_correlated_groups(
                Xtr_local,
                n_groups=max(2, int(round(np.sqrt(n_feat)))),
                random_state=BASE_SEED
            )
            if isinstance(mg_retry, (tuple, list)) and len(mg_retry) >= 1:
                groups_raw = mg_retry[0]
                imp_from_groups = mg_retry[1] if len(mg_retry) > 1 else None
            else:
                groups_raw = mg_retry
                imp_from_groups = None

            if groups_raw is not None:
                groups_list = _coerce_groups_to_index_lists(groups_raw, Xtr_local)
                groups_list = [[int(j) for j in g if 0 <= int(j) < n_feat] for g in groups_list if len(g) > 0]
                if len(groups_list) == 0:
                    groups_list = None
                else:
                    print(f"Retry succeeded: {len(groups_list)} groups")

            if imp_from_groups is not None and imp_arr is None:
                try:
                    imp_arr = np.asarray(imp_from_groups, dtype=float)
                except Exception:
                    pass
        except Exception as e:
            print("Retry attempt failed (ignored):", e)

    # fallback RF importances
    from sklearn.ensemble import RandomForestRegressor
    rf = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=BASE_SEED, n_jobs=-1)
    try:
        rf.fit(Xtr_local, ytr_local)
        rf_imp = np.asarray(rf.feature_importances_, dtype=float)
    except Exception as e:
        print("RF importance failed:", e)
        rf_imp = np.zeros(n_feat, dtype=float)

    if groups_list is None:
        groups_list = [[i] for i in range(n_feat)]
        print("Fallback groups: one-feature-per-group (count = {})".format(len(groups_list)))

    if imp_arr is None:
        imp_arr = rf_imp.copy()
    else:
        tmp = np.zeros(n_feat, dtype=float)
        tmp[:min(len(imp_arr), n_feat)] = imp_arr[:min(len(imp_arr), n_feat)]
        imp_arr = tmp

    if imp_arr.shape[0] != n_feat:
        tmp = np.zeros(n_feat, dtype=float)
        tmp[:min(len(imp_arr), n_feat)] = imp_arr[:min(len(imp_arr), n_feat)]
        imp_arr = tmp

    imp_arr = np.nan_to_num(imp_arr, nan=0.0, posinf=0.0, neginf=0.0)
    if imp_arr.sum() <= 0:
        imp_arr = np.ones(n_feat, dtype=float) / float(n_feat)
    else:
        imp_arr = imp_arr.astype(float) / float(np.sum(imp_arr))

    print("Prepared groups_list length:", len(groups_list), "imp_arr shape:", imp_arr.shape)
    order = np.argsort(-imp_arr)[:min(8, n_feat)]
    print("Top importances (idx -> name -> score):")
    for i in order:
        print(i, Xtr_local.columns[i], f"{imp_arr[i]:.4f}")

    for helper in ("sample_one_subset", "fit_one_sr_on_subset", "eval_baselines"):
        if helper not in globals():
            raise RuntimeError(f"Helper '{helper}' missing — paste its definition before running this cell.")

    def get_program_str(est):
        for attr in ("program", "program_", "_program"):
            prog = getattr(est, attr, None)
            if prog is not None:
                try:
                    return str(prog)
                except Exception:
                    return repr(prog)
        return None

    # --- SWEEP ---
    best_te = float("inf")
    best_est = None
    best_subset = None
    rows = []

    print("Starting hybrid sweep: N_RUNS=", N_RUNS)
    for i in range(1, N_RUNS + 1):
        run_seed = int(BASE_SEED + i)

        try:
            subset_idx = sample_one_subset(
                groups_list,
                imp_arr,
                alpha=ALPHA,
                pick_one_per_group=True,
                rng=np.random.default_rng(run_seed)
            )
        except TypeError:
            subset_idx = sample_one_subset(
                groups_list,
                imp_arr,
                alpha=ALPHA,
                pick_one_per_group=True
            )

        try:
            subset_idx = [int(x) for x in list(subset_idx)]
        except Exception:
            subset_idx = list(subset_idx) if hasattr(subset_idx, "__iter__") else []

        if len(subset_idx) == 0:
            print(f"[{i}] sampled empty subset — skipping")
            rows.append({
                "run": i,
                "seed": run_seed,
                "subset_idx": subset_idx,
                "tr_rmse": None,
                "te_rmse": None,
                "expr": None,
                "status": "empty_subset"
            })
            continue

        try:
            est, tr_rmse, te_rmse = fit_one_sr_on_subset(
                Xtr_local, ytr_local, Xte_local, subset_idx,
                function_set=('add', 'sub', 'mul', 'div', 'sin', 'cos'),
                parsimony=0.001,
                generations=100,
                population_size=1500,
                random_state=run_seed,
                verbose=0
            )
        except Exception as e:
            print(f"[{i}] SR training failed: {e}")
            rows.append({
                "run": i,
                "seed": run_seed,
                "subset_idx": subset_idx,
                "tr_rmse": None,
                "te_rmse": None,
                "expr": None,
                "error": str(e)
            })
            continue

        expr = get_program_str(est)

        rows.append({
            "run": i,
            "seed": run_seed,
            "subset_idx": subset_idx,
            "tr_rmse": float(tr_rmse) if tr_rmse is not None else None,
            "te_rmse": float(te_rmse) if te_rmse is not None else None,
            "expr": expr
        })

        if te_rmse is not None and te_rmse < best_te:
            best_te = te_rmse
            best_est = est
            best_subset = subset_idx

        if (i % 10) == 0:
            print(f"[{i}/{N_RUNS}] checkpoint best_te: {best_te:.6f}")

    # Save raw sweep rows
    outdir = ROOT
    outdir.mkdir(parents=True, exist_ok=True)
    partial_csv = outdir / f"{ds_name}_sweep_partial.csv"
    pd.DataFrame(rows).to_csv(partial_csv, index=False)
    print("Saved partial sweep results to:", partial_csv)

    # --- finalize: evaluate baselines + RAE ---
    summary = {
        "dataset": ds_name,
        "n_samples_train": int(Xtr_local.shape[0]),
        "n_features": int(Xtr_local.shape[1]),
        "best_subset": best_subset,
        "best_te": float(best_te) if best_te != float("inf") else None,
        "timestamp": time.time(),
    }

    if best_est is not None and best_subset is not None:
        try:
            try:
                Xtr_sub = Xtr_local.iloc[:, best_subset]
                Xte_sub = Xte_local.iloc[:, best_subset]
            except Exception:
                Xtr_sub = Xtr_local.loc[:, best_subset]
                Xte_sub = Xte_local.loc[:, best_subset]

            sr_tr_feature = best_est.predict(Xtr_sub)
            sr_te_feature = best_est.predict(Xte_sub)

            Xtr_aug = Xtr_local.copy()
            Xte_aug = Xte_local.copy()
            Xtr_aug["SR_1"] = sr_tr_feature
            Xte_aug["SR_1"] = sr_te_feature

            baseline_metrics = eval_baselines(Xtr_local, Xte_local, ytr_local, yte_local)
            augmented_metrics = eval_baselines(Xtr_aug, Xte_aug, ytr_local, yte_local)
            dRMSE = {k.replace("_RMSE", "_dRMSE"): augmented_metrics[k] - baseline_metrics[k] for k in baseline_metrics}

            summary.update({
                "baseline": baseline_metrics,
                "augmented": augmented_metrics,
                "dRMSE": dRMSE
            })

            print("Baseline:", baseline_metrics)
            print("Augmented:", augmented_metrics)
            print("ΔRMSE:", dRMSE)

            with open(eval_json, "w") as f:
                json.dump(summary, f, indent=2)
            pd.DataFrame([summary]).to_csv(eval_csv, index=False)

            print("Saved evaluation summary JSON:", eval_json)
            print("Saved evaluation summary CSV:", eval_csv)

        except Exception as e:
            print("Error during final augmentation/evaluation:", e)
            traceback.print_exc()
    else:
        print("No successful SR model found during sweep for", ds_name)
        with open(eval_json, "w") as f:
            json.dump(summary, f, indent=2)
        pd.DataFrame([summary]).to_csv(eval_csv, index=False)

    print("Elapsed (s):", time.time() - start_ts)

Hybrid run config: {'dataset': 'openml_589', 'N_RUNS': 40, 'ALPHA': 2.0, 'BASE_SEED': 42, 'VAR_THRESH': 0.001}
Loaded dataset: openml_589 | train: (800, 25) | test: (200, 25) | used files: ['X_train.parquet', 'y_train.pkl']
After impute shapes: (800, 25) (200, 25)


/home/rshayan1/.local/lib/python3.12/site-packages/sklearn/cluster/_agglomerative.py:983: FutureWarning: Attribute `affinity` was deprecated in version 1.2 and will be removed in 1.4. Use `metric` instead
  warnings.warn(


make_correlated_groups: n_features=25, n_groups_requested=5, produced_groups=5
Sample groups (first 6): [[5, 19], [11, 17], [4, 8, 13, 21, 22], [14, 15, 16, 18, 23, 24], [0, 1, 2, 3, 6, 7, 9, 10, 12, 20]]
Top features by imp (idx -> name -> score):
 1 1 0.3340
 0 0 0.2857
 3 3 0.0765
 4 4 0.0693
 2 2 0.0393
 20 20 0.0165
 9 9 0.0141
 14 14 0.0136
 12 12 0.0121
 8 8 0.0117
Correlated groups produced: 5 groups
Using importance array returned by make_correlated_groups.
Prepared groups_list length: 5 imp_arr shape: (25,)
Top importances (idx -> name -> score):
1 oz2 0.3340
0 oz1 0.2857
3 oz4 0.0765
4 oz5 0.0693
2 oz3 0.0393
20 oz21 0.0165
9 oz10 0.0141
14 oz15 0.0136
Starting hybrid sweep: N_RUNS= 40


In [23]:
print(best_est._program)

mul(cos(X4), sub(sub(mul(X2, 0.248), X4), mul(sub(sin(sin(sub(sub(sub(cos(X4), X4), X4), X4))), X4), -0.603)))


## Openml_607 Testing

In [22]:
# CELL: run_hybrid_for_one_dataset (openml_607 - NEW CONFIG)
import time, json, traceback
from pathlib import Path
import numpy as np
import pandas as pd

# USER CONFIG
ds_name = "openml_607"
N_RUNS = 40
ALPHA = 2.0
BASE_SEED = 42
VAR_THRESH = 1e-3
ROOT = Path("artifacts") / "hybrid_experiments_summary"
ROOT.mkdir(parents=True, exist_ok=True)

print("Hybrid run config:", {
    "dataset": ds_name,
    "N_RUNS": N_RUNS,
    "ALPHA": ALPHA,
    "BASE_SEED": BASE_SEED,
    "VAR_THRESH": VAR_THRESH
})

# output guard
eval_json = ROOT / f"{ds_name}_evaluation_summary.json"
eval_csv = ROOT / f"{ds_name}_evaluation_summary.csv"
if eval_json.exists() or eval_csv.exists():
    print(f"Skipping {ds_name}: evaluation already exists at {eval_json} / {eval_csv}")
else:
    start_ts = time.time()

    # --- load canonical dataset ---
    ds_dir = Path("artifacts") / "canonical" / ds_name
    if "load_canonical_Xy" not in globals():
        raise RuntimeError("load_canonical_Xy(...) not found — define it before running this cell.")

    X_all, y_all, meta, used_files = load_canonical_Xy(ds_dir)

    # try explicit test split if present
    Xte = None
    yte = None
    try:
        if (ds_dir / "X_test.parquet").exists():
            Xte = pd.read_parquet(ds_dir / "X_test.parquet")
        if (ds_dir / "y_test.pkl").exists():
            yte = pd.read_pickle(ds_dir / "y_test.pkl").squeeze()
    except Exception as e:
        print("Warning loading explicit test split (ignored):", e)

    # if no explicit test, do deterministic 80/20 split
    if Xte is None or yte is None:
        from sklearn.model_selection import train_test_split
        Xtr_df, Xte_df, ytr_ser, yte_ser = train_test_split(
            pd.DataFrame(X_all), y_all, test_size=0.2, random_state=42
        )
        ytr = np.asarray(pd.to_numeric(ytr_ser, errors="coerce")).ravel()
        yte = np.asarray(pd.to_numeric(yte_ser, errors="coerce")).ravel()
    else:
        Xtr_df = pd.DataFrame(X_all)
        Xte_df = pd.DataFrame(Xte)
        ytr = np.asarray(pd.to_numeric(y_all, errors="coerce")).ravel()
        yte = np.asarray(pd.to_numeric(yte, errors="coerce")).ravel()

    # canonical local copies
    Xtr_local = Xtr_df.reset_index(drop=True)
    Xte_local = Xte_df.reset_index(drop=True)
    ytr_local = ytr
    yte_local = yte

    print("Loaded dataset:", ds_name, "| train:", Xtr_local.shape, "| test:", Xte_local.shape, "| used files:", used_files)

    # --- impute + coerce numeric defensively ---
    from sklearn.impute import SimpleImputer
    imp = SimpleImputer(strategy="median")
    if (Xtr_local.isna().sum().sum() > 0) or (Xte_local.isna().sum().sum() > 0):
        Xtr_imp = pd.DataFrame(imp.fit_transform(Xtr_local), columns=Xtr_local.columns, index=Xtr_local.index)
        Xte_imp = pd.DataFrame(imp.transform(Xte_local), columns=Xte_local.columns, index=Xte_local.index)
    else:
        Xtr_imp = Xtr_local.copy()
        Xte_imp = Xte_local.copy()

    for df in (Xtr_imp, Xte_imp):
        for c in df.columns:
            if not pd.api.types.is_numeric_dtype(df[c]):
                df[c] = pd.to_numeric(df[c], errors="coerce")
    Xtr_imp = Xtr_imp.fillna(0.0)
    Xte_imp = Xte_imp.fillna(0.0)

    Xtr_local = Xtr_imp.reset_index(drop=True)
    Xte_local = Xte_imp.reset_index(drop=True)
    ytr_local = np.asarray(pd.to_numeric(ytr_local, errors="coerce")).ravel()
    yte_local = np.asarray(pd.to_numeric(yte_local, errors="coerce")).ravel()

    print("After impute shapes:", Xtr_local.shape, Xte_local.shape)

    # --- build groups & importances ---
    n_feat = Xtr_local.shape[1]
    groups_list = None
    imp_arr = None

    def _coerce_groups_to_index_lists(groups_in, Xdf):
        if groups_in is None:
            return None
        out = []
        if isinstance(groups_in, dict):
            vals = list(groups_in.values())
        else:
            vals = list(groups_in)
        for g in vals:
            if isinstance(g, (list, tuple, np.ndarray)):
                idxs = []
                for el in g:
                    if isinstance(el, str):
                        try:
                            idxs.append(int(list(Xdf.columns).index(el)))
                        except Exception:
                            raise ValueError(f"Cannot map group element '{el}' -> not a column name")
                    else:
                        idxs.append(int(el))
                out.append(list(idxs))
            else:
                if isinstance(g, str):
                    idx = int(list(Xdf.columns).index(g))
                    out.append([idx])
                else:
                    out.append([int(g)])
        return out

    if "make_correlated_groups" in globals():
        try:
            mg_out = make_correlated_groups(Xtr_local, n_groups=None, random_state=BASE_SEED)
            groups_raw = None
            imp_from_groups = None

            if mg_out is None:
                groups_raw = None
            elif isinstance(mg_out, (tuple, list)):
                if len(mg_out) == 1:
                    first = mg_out[0]
                    if isinstance(first, (list, tuple, np.ndarray)):
                        groups_raw = first
                    else:
                        try:
                            arr = np.asarray(first, dtype=float)
                            if arr.shape[0] == n_feat:
                                imp_from_groups = arr
                            else:
                                groups_raw = first
                        except Exception:
                            groups_raw = first
                else:
                    groups_raw = mg_out[0]
                    try:
                        imp_from_groups = np.asarray(mg_out[1], dtype=float)
                    except Exception:
                        imp_from_groups = None
            elif isinstance(mg_out, dict):
                if "groups" in mg_out and "importance" in mg_out:
                    groups_raw = mg_out["groups"]
                    imp_from_groups = np.asarray(mg_out["importance"], dtype=float)
                else:
                    groups_raw = list(mg_out.values())
            elif isinstance(mg_out, np.ndarray):
                imp_from_groups = np.asarray(mg_out, dtype=float)
            else:
                groups_raw = mg_out

            if groups_raw is not None:
                try:
                    groups_list = _coerce_groups_to_index_lists(groups_raw, Xtr_local)
                    groups_list = [[int(j) for j in g if 0 <= int(j) < n_feat] for g in groups_list if len(g) > 0]
                    if len(groups_list) == 0:
                        groups_list = None
                    else:
                        print(f"Correlated groups produced: {len(groups_list)} groups")
                except Exception as e:
                    print("Could not coerce correlated groups (exception):", e)
                    groups_list = None

            if imp_from_groups is not None:
                try:
                    imp_arr = np.asarray(imp_from_groups, dtype=float)
                    print("Using importance array returned by make_correlated_groups.")
                except Exception:
                    imp_arr = None

        except Exception as e:
            print("make_correlated_groups failed (will fallback to RF one-feature-per-group). Exception:")
            traceback.print_exc()
            groups_list = None
    else:
        print("make_correlated_groups not found; will fallback to RF/one-feature groups.")

    if groups_list is None and "make_correlated_groups" in globals():
        try:
            print("Retrying make_correlated_groups with explicit n_groups heuristic...")
            mg_retry = make_correlated_groups(
                Xtr_local,
                n_groups=max(2, int(round(np.sqrt(n_feat)))),
                random_state=BASE_SEED
            )
            if isinstance(mg_retry, (tuple, list)) and len(mg_retry) >= 1:
                groups_raw = mg_retry[0]
                imp_from_groups = mg_retry[1] if len(mg_retry) > 1 else None
            else:
                groups_raw = mg_retry
                imp_from_groups = None

            if groups_raw is not None:
                groups_list = _coerce_groups_to_index_lists(groups_raw, Xtr_local)
                groups_list = [[int(j) for j in g if 0 <= int(j) < n_feat] for g in groups_list if len(g) > 0]
                if len(groups_list) == 0:
                    groups_list = None
                else:
                    print(f"Retry succeeded: {len(groups_list)} groups")

            if imp_from_groups is not None and imp_arr is None:
                try:
                    imp_arr = np.asarray(imp_from_groups, dtype=float)
                except Exception:
                    pass
        except Exception as e:
            print("Retry attempt failed (ignored):", e)

    # fallback RF importances
    from sklearn.ensemble import RandomForestRegressor
    rf = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=BASE_SEED, n_jobs=-1)
    try:
        rf.fit(Xtr_local, ytr_local)
        rf_imp = np.asarray(rf.feature_importances_, dtype=float)
    except Exception as e:
        print("RF importance failed:", e)
        rf_imp = np.zeros(n_feat, dtype=float)

    if groups_list is None:
        groups_list = [[i] for i in range(n_feat)]
        print("Fallback groups: one-feature-per-group (count = {})".format(len(groups_list)))

    if imp_arr is None:
        imp_arr = rf_imp.copy()
    else:
        tmp = np.zeros(n_feat, dtype=float)
        tmp[:min(len(imp_arr), n_feat)] = imp_arr[:min(len(imp_arr), n_feat)]
        imp_arr = tmp

    if imp_arr.shape[0] != n_feat:
        tmp = np.zeros(n_feat, dtype=float)
        tmp[:min(len(imp_arr), n_feat)] = imp_arr[:min(len(imp_arr), n_feat)]
        imp_arr = tmp

    imp_arr = np.nan_to_num(imp_arr, nan=0.0, posinf=0.0, neginf=0.0)
    if imp_arr.sum() <= 0:
        imp_arr = np.ones(n_feat, dtype=float) / float(n_feat)
    else:
        imp_arr = imp_arr.astype(float) / float(np.sum(imp_arr))

    print("Prepared groups_list length:", len(groups_list), "imp_arr shape:", imp_arr.shape)
    order = np.argsort(-imp_arr)[:min(8, n_feat)]
    print("Top importances (idx -> name -> score):")
    for i in order:
        print(i, Xtr_local.columns[i], f"{imp_arr[i]:.4f}")

    for helper in ("sample_one_subset", "fit_one_sr_on_subset", "eval_baselines"):
        if helper not in globals():
            raise RuntimeError(f"Helper '{helper}' missing — paste its definition before running this cell.")

    def get_program_str(est):
        for attr in ("program", "program_", "_program"):
            prog = getattr(est, attr, None)
            if prog is not None:
                try:
                    return str(prog)
                except Exception:
                    return repr(prog)
        return None

    # --- SWEEP ---
    best_te = float("inf")
    best_est = None
    best_subset = None
    rows = []

    print("Starting hybrid sweep: N_RUNS=", N_RUNS)
    for i in range(1, N_RUNS + 1):
        run_seed = int(BASE_SEED + i)

        try:
            subset_idx = sample_one_subset(
                groups_list,
                imp_arr,
                alpha=ALPHA,
                pick_one_per_group=True,
                rng=np.random.default_rng(run_seed)
            )
        except TypeError:
            subset_idx = sample_one_subset(
                groups_list,
                imp_arr,
                alpha=ALPHA,
                pick_one_per_group=True
            )

        try:
            subset_idx = [int(x) for x in list(subset_idx)]
        except Exception:
            subset_idx = list(subset_idx) if hasattr(subset_idx, "__iter__") else []

        if len(subset_idx) == 0:
            print(f"[{i}] sampled empty subset — skipping")
            rows.append({
                "run": i,
                "seed": run_seed,
                "subset_idx": subset_idx,
                "tr_rmse": None,
                "te_rmse": None,
                "expr": None,
                "status": "empty_subset"
            })
            continue

        try:
            est, tr_rmse, te_rmse = fit_one_sr_on_subset(
                Xtr_local, ytr_local, Xte_local, subset_idx,
                function_set=('add', 'sub', 'mul', 'div', 'sin', 'cos'),
                parsimony=0.001,
                generations=100,
                population_size=1500,
                random_state=run_seed,
                verbose=0
            )
        except Exception as e:
            print(f"[{i}] SR training failed: {e}")
            rows.append({
                "run": i,
                "seed": run_seed,
                "subset_idx": subset_idx,
                "tr_rmse": None,
                "te_rmse": None,
                "expr": None,
                "error": str(e)
            })
            continue

        expr = get_program_str(est)

        rows.append({
            "run": i,
            "seed": run_seed,
            "subset_idx": subset_idx,
            "tr_rmse": float(tr_rmse) if tr_rmse is not None else None,
            "te_rmse": float(te_rmse) if te_rmse is not None else None,
            "expr": expr
        })

        if te_rmse is not None and te_rmse < best_te:
            best_te = te_rmse
            best_est = est
            best_subset = subset_idx

        if (i % 10) == 0:
            print(f"[{i}/{N_RUNS}] checkpoint best_te: {best_te:.6f}")

    # Save raw sweep rows
    outdir = ROOT
    outdir.mkdir(parents=True, exist_ok=True)
    partial_csv = outdir / f"{ds_name}_sweep_partial.csv"
    pd.DataFrame(rows).to_csv(partial_csv, index=False)
    print("Saved partial sweep results to:", partial_csv)

    # --- finalize: evaluate baselines + RAE ---
    summary = {
        "dataset": ds_name,
        "n_samples_train": int(Xtr_local.shape[0]),
        "n_features": int(Xtr_local.shape[1]),
        "best_subset": best_subset,
        "best_te": float(best_te) if best_te != float("inf") else None,
        "timestamp": time.time(),
    }

    if best_est is not None and best_subset is not None:
        try:
            try:
                Xtr_sub = Xtr_local.iloc[:, best_subset]
                Xte_sub = Xte_local.iloc[:, best_subset]
            except Exception:
                Xtr_sub = Xtr_local.loc[:, best_subset]
                Xte_sub = Xte_local.loc[:, best_subset]

            sr_tr_feature = best_est.predict(Xtr_sub)
            sr_te_feature = best_est.predict(Xte_sub)

            Xtr_aug = Xtr_local.copy()
            Xte_aug = Xte_local.copy()
            Xtr_aug["SR_1"] = sr_tr_feature
            Xte_aug["SR_1"] = sr_te_feature

            baseline_metrics = eval_baselines(Xtr_local, Xte_local, ytr_local, yte_local)
            augmented_metrics = eval_baselines(Xtr_aug, Xte_aug, ytr_local, yte_local)
            dRMSE = {k.replace("_RMSE", "_dRMSE"): augmented_metrics[k] - baseline_metrics[k] for k in baseline_metrics}

            summary.update({
                "baseline": baseline_metrics,
                "augmented": augmented_metrics,
                "dRMSE": dRMSE
            })

            print("Baseline:", baseline_metrics)
            print("Augmented:", augmented_metrics)
            print("ΔRMSE:", dRMSE)

            with open(eval_json, "w") as f:
                json.dump(summary, f, indent=2)
            pd.DataFrame([summary]).to_csv(eval_csv, index=False)

            print("Saved evaluation summary JSON:", eval_json)
            print("Saved evaluation summary CSV:", eval_csv)

        except Exception as e:
            print("Error during final augmentation/evaluation:", e)
            traceback.print_exc()
    else:
        print("No successful SR model found during sweep for", ds_name)
        with open(eval_json, "w") as f:
            json.dump(summary, f, indent=2)
        pd.DataFrame([summary]).to_csv(eval_csv, index=False)

    print("Elapsed (s):", time.time() - start_ts)

Hybrid run config: {'dataset': 'openml_607', 'N_RUNS': 40, 'ALPHA': 2.0, 'BASE_SEED': 42, 'VAR_THRESH': 0.001}
Loaded dataset: openml_607 | train: (800, 50) | test: (200, 50) | used files: ['X_train.parquet', 'y_train.pkl']
After impute shapes: (800, 50) (200, 50)


/home/rshayan1/.local/lib/python3.12/site-packages/sklearn/cluster/_agglomerative.py:983: FutureWarning: Attribute `affinity` was deprecated in version 1.2 and will be removed in 1.4. Use `metric` instead
  warnings.warn(


make_correlated_groups: n_features=50, n_groups_requested=7, produced_groups=7
Sample groups (first 6): [[13, 31, 34], [8, 28, 37, 40], [12, 14, 44, 49], [29, 33, 36, 39, 41], [11, 18, 24, 32, 47], [5, 6, 7, 9, 10, 26, 30]]
Top features by imp (idx -> name -> score):
 4 4 0.3062
 1 1 0.1704
 3 3 0.1463
 0 0 0.1067
 2 2 0.0413
 45 45 0.0145
 26 26 0.0142
 17 17 0.0109
 49 49 0.0100
 24 24 0.0074
Correlated groups produced: 7 groups
Using importance array returned by make_correlated_groups.
Prepared groups_list length: 7 imp_arr shape: (50,)
Top importances (idx -> name -> score):
4 oz5 0.3062
1 oz2 0.1704
3 oz4 0.1463
0 oz1 0.1067
2 oz3 0.0413
45 oz46 0.0145
26 oz27 0.0142
17 oz18 0.0109
Starting hybrid sweep: N_RUNS= 40
[10/40] checkpoint best_te: 0.686191
[20/40] checkpoint best_te: 0.686191
[30/40] checkpoint best_te: 0.683257
[40/40] checkpoint best_te: 0.683257
Saved partial sweep results to: artifacts/hybrid_experiments_summary/openml_607_sweep_partial.csv
Baseline: {'LR_RMSE': 0.

## Openml_616 Testing

In [17]:
# CELL: run_hybrid_for_one_dataset (openml_616 - NEW CONFIG)
import time, json, traceback
from pathlib import Path
import numpy as np
import pandas as pd

# USER CONFIG
ds_name = "openml_616"
N_RUNS = 40
ALPHA = 2.0
BASE_SEED = 42
VAR_THRESH = 1e-3
ROOT = Path("artifacts") / "hybrid_experiments_summary"
ROOT.mkdir(parents=True, exist_ok=True)

print("Hybrid run config:", {
    "dataset": ds_name,
    "N_RUNS": N_RUNS,
    "ALPHA": ALPHA,
    "BASE_SEED": BASE_SEED,
    "VAR_THRESH": VAR_THRESH
})

# output guard
eval_json = ROOT / f"{ds_name}_evaluation_summary.json"
eval_csv = ROOT / f"{ds_name}_evaluation_summary.csv"
if eval_json.exists() or eval_csv.exists():
    print(f"Skipping {ds_name}: evaluation already exists at {eval_json} / {eval_csv}")
else:
    start_ts = time.time()

    # --- load canonical dataset ---
    ds_dir = Path("artifacts") / "canonical" / ds_name
    if "load_canonical_Xy" not in globals():
        raise RuntimeError("load_canonical_Xy(...) not found — define it before running this cell.")

    X_all, y_all, meta, used_files = load_canonical_Xy(ds_dir)

    # try explicit test split if present
    Xte = None
    yte = None
    try:
        if (ds_dir / "X_test.parquet").exists():
            Xte = pd.read_parquet(ds_dir / "X_test.parquet")
        if (ds_dir / "y_test.pkl").exists():
            yte = pd.read_pickle(ds_dir / "y_test.pkl").squeeze()
    except Exception as e:
        print("Warning loading explicit test split (ignored):", e)

    # if no explicit test, do deterministic 80/20 split
    if Xte is None or yte is None:
        from sklearn.model_selection import train_test_split
        Xtr_df, Xte_df, ytr_ser, yte_ser = train_test_split(
            pd.DataFrame(X_all), y_all, test_size=0.2, random_state=42
        )
        ytr = np.asarray(pd.to_numeric(ytr_ser, errors="coerce")).ravel()
        yte = np.asarray(pd.to_numeric(yte_ser, errors="coerce")).ravel()
    else:
        Xtr_df = pd.DataFrame(X_all)
        Xte_df = pd.DataFrame(Xte)
        ytr = np.asarray(pd.to_numeric(y_all, errors="coerce")).ravel()
        yte = np.asarray(pd.to_numeric(yte, errors="coerce")).ravel()

    # canonical local copies
    Xtr_local = Xtr_df.reset_index(drop=True)
    Xte_local = Xte_df.reset_index(drop=True)
    ytr_local = ytr
    yte_local = yte

    print("Loaded dataset:", ds_name, "| train:", Xtr_local.shape, "| test:", Xte_local.shape, "| used files:", used_files)

    # --- impute + coerce numeric defensively ---
    from sklearn.impute import SimpleImputer
    imp = SimpleImputer(strategy="median")
    if (Xtr_local.isna().sum().sum() > 0) or (Xte_local.isna().sum().sum() > 0):
        Xtr_imp = pd.DataFrame(imp.fit_transform(Xtr_local), columns=Xtr_local.columns, index=Xtr_local.index)
        Xte_imp = pd.DataFrame(imp.transform(Xte_local), columns=Xte_local.columns, index=Xte_local.index)
    else:
        Xtr_imp = Xtr_local.copy()
        Xte_imp = Xte_local.copy()

    for df in (Xtr_imp, Xte_imp):
        for c in df.columns:
            if not pd.api.types.is_numeric_dtype(df[c]):
                df[c] = pd.to_numeric(df[c], errors="coerce")
    Xtr_imp = Xtr_imp.fillna(0.0)
    Xte_imp = Xte_imp.fillna(0.0)

    Xtr_local = Xtr_imp.reset_index(drop=True)
    Xte_local = Xte_imp.reset_index(drop=True)
    ytr_local = np.asarray(pd.to_numeric(ytr_local, errors="coerce")).ravel()
    yte_local = np.asarray(pd.to_numeric(yte_local, errors="coerce")).ravel()

    print("After impute shapes:", Xtr_local.shape, Xte_local.shape)

    # --- build groups & importances ---
    n_feat = Xtr_local.shape[1]
    groups_list = None
    imp_arr = None

    def _coerce_groups_to_index_lists(groups_in, Xdf):
        if groups_in is None:
            return None
        out = []
        if isinstance(groups_in, dict):
            vals = list(groups_in.values())
        else:
            vals = list(groups_in)
        for g in vals:
            if isinstance(g, (list, tuple, np.ndarray)):
                idxs = []
                for el in g:
                    if isinstance(el, str):
                        try:
                            idxs.append(int(list(Xdf.columns).index(el)))
                        except Exception:
                            raise ValueError(f"Cannot map group element '{el}' -> not a column name")
                    else:
                        idxs.append(int(el))
                out.append(list(idxs))
            else:
                if isinstance(g, str):
                    idx = int(list(Xdf.columns).index(g))
                    out.append([idx])
                else:
                    out.append([int(g)])
        return out

    if "make_correlated_groups" in globals():
        try:
            mg_out = make_correlated_groups(Xtr_local, n_groups=None, random_state=BASE_SEED)
            groups_raw = None
            imp_from_groups = None

            if mg_out is None:
                groups_raw = None
            elif isinstance(mg_out, (tuple, list)):
                if len(mg_out) == 1:
                    first = mg_out[0]
                    if isinstance(first, (list, tuple, np.ndarray)):
                        groups_raw = first
                    else:
                        try:
                            arr = np.asarray(first, dtype=float)
                            if arr.shape[0] == n_feat:
                                imp_from_groups = arr
                            else:
                                groups_raw = first
                        except Exception:
                            groups_raw = first
                else:
                    groups_raw = mg_out[0]
                    try:
                        imp_from_groups = np.asarray(mg_out[1], dtype=float)
                    except Exception:
                        imp_from_groups = None
            elif isinstance(mg_out, dict):
                if "groups" in mg_out and "importance" in mg_out:
                    groups_raw = mg_out["groups"]
                    imp_from_groups = np.asarray(mg_out["importance"], dtype=float)
                else:
                    groups_raw = list(mg_out.values())
            elif isinstance(mg_out, np.ndarray):
                imp_from_groups = np.asarray(mg_out, dtype=float)
            else:
                groups_raw = mg_out

            if groups_raw is not None:
                try:
                    groups_list = _coerce_groups_to_index_lists(groups_raw, Xtr_local)
                    groups_list = [[int(j) for j in g if 0 <= int(j) < n_feat] for g in groups_list if len(g) > 0]
                    if len(groups_list) == 0:
                        groups_list = None
                    else:
                        print(f"Correlated groups produced: {len(groups_list)} groups")
                except Exception as e:
                    print("Could not coerce correlated groups (exception):", e)
                    groups_list = None

            if imp_from_groups is not None:
                try:
                    imp_arr = np.asarray(imp_from_groups, dtype=float)
                    print("Using importance array returned by make_correlated_groups.")
                except Exception:
                    imp_arr = None

        except Exception as e:
            print("make_correlated_groups failed (will fallback to RF one-feature-per-group). Exception:")
            traceback.print_exc()
            groups_list = None
    else:
        print("make_correlated_groups not found; will fallback to RF/one-feature groups.")

    if groups_list is None and "make_correlated_groups" in globals():
        try:
            print("Retrying make_correlated_groups with explicit n_groups heuristic...")
            mg_retry = make_correlated_groups(
                Xtr_local,
                n_groups=max(2, int(round(np.sqrt(n_feat)))),
                random_state=BASE_SEED
            )
            if isinstance(mg_retry, (tuple, list)) and len(mg_retry) >= 1:
                groups_raw = mg_retry[0]
                imp_from_groups = mg_retry[1] if len(mg_retry) > 1 else None
            else:
                groups_raw = mg_retry
                imp_from_groups = None

            if groups_raw is not None:
                groups_list = _coerce_groups_to_index_lists(groups_raw, Xtr_local)
                groups_list = [[int(j) for j in g if 0 <= int(j) < n_feat] for g in groups_list if len(g) > 0]
                if len(groups_list) == 0:
                    groups_list = None
                else:
                    print(f"Retry succeeded: {len(groups_list)} groups")

            if imp_from_groups is not None and imp_arr is None:
                try:
                    imp_arr = np.asarray(imp_from_groups, dtype=float)
                except Exception:
                    pass
        except Exception as e:
            print("Retry attempt failed (ignored):", e)

    from sklearn.ensemble import RandomForestRegressor
    rf = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=BASE_SEED, n_jobs=-1)
    try:
        rf.fit(Xtr_local, ytr_local)
        rf_imp = np.asarray(rf.feature_importances_, dtype=float)
    except Exception as e:
        print("RF importance failed:", e)
        rf_imp = np.zeros(n_feat, dtype=float)

    if groups_list is None:
        groups_list = [[i] for i in range(n_feat)]
        print("Fallback groups: one-feature-per-group (count = {})".format(len(groups_list)))

    if imp_arr is None:
        imp_arr = rf_imp.copy()
    else:
        tmp = np.zeros(n_feat, dtype=float)
        tmp[:min(len(imp_arr), n_feat)] = imp_arr[:min(len(imp_arr), n_feat)]
        imp_arr = tmp

    if imp_arr.shape[0] != n_feat:
        tmp = np.zeros(n_feat, dtype=float)
        tmp[:min(len(imp_arr), n_feat)] = imp_arr[:min(len(imp_arr), n_feat)]
        imp_arr = tmp

    imp_arr = np.nan_to_num(imp_arr, nan=0.0, posinf=0.0, neginf=0.0)
    if imp_arr.sum() <= 0:
        imp_arr = np.ones(n_feat, dtype=float) / float(n_feat)
    else:
        imp_arr = imp_arr.astype(float) / float(np.sum(imp_arr))

    print("Prepared groups_list length:", len(groups_list), "imp_arr shape:", imp_arr.shape)
    order = np.argsort(-imp_arr)[:min(8, n_feat)]
    print("Top importances (idx -> name -> score):")
    for i in order:
        print(i, Xtr_local.columns[i], f"{imp_arr[i]:.4f}")

    for helper in ("sample_one_subset", "fit_one_sr_on_subset", "eval_baselines"):
        if helper not in globals():
            raise RuntimeError(f"Helper '{helper}' missing — paste its definition before running this cell.")

    def get_program_str(est):
        for attr in ("program", "program_", "_program"):
            prog = getattr(est, attr, None)
            if prog is not None:
                try:
                    return str(prog)
                except Exception:
                    return repr(prog)
        return None

    # --- SWEEP ---
    best_te = float("inf")
    best_est = None
    best_subset = None
    rows = []

    print("Starting hybrid sweep: N_RUNS=", N_RUNS)
    for i in range(1, N_RUNS + 1):
        run_seed = int(BASE_SEED + i)

        try:
            subset_idx = sample_one_subset(
                groups_list,
                imp_arr,
                alpha=ALPHA,
                pick_one_per_group=True,
                rng=np.random.default_rng(run_seed)
            )
        except TypeError:
            subset_idx = sample_one_subset(
                groups_list,
                imp_arr,
                alpha=ALPHA,
                pick_one_per_group=True
            )

        try:
            subset_idx = [int(x) for x in list(subset_idx)]
        except Exception:
            subset_idx = list(subset_idx) if hasattr(subset_idx, "__iter__") else []

        if len(subset_idx) == 0:
            print(f"[{i}] sampled empty subset — skipping")
            rows.append({
                "run": i,
                "seed": run_seed,
                "subset_idx": subset_idx,
                "tr_rmse": None,
                "te_rmse": None,
                "expr": None,
                "status": "empty_subset"
            })
            continue

        try:
            est, tr_rmse, te_rmse = fit_one_sr_on_subset(
                Xtr_local, ytr_local, Xte_local, subset_idx,
                function_set=('add', 'sub', 'mul', 'div', 'sin', 'cos'),
                parsimony=0.001,
                generations=100,
                population_size=1500,
                random_state=run_seed,
                verbose=0
            )
        except Exception as e:
            print(f"[{i}] SR training failed: {e}")
            rows.append({
                "run": i,
                "seed": run_seed,
                "subset_idx": subset_idx,
                "tr_rmse": None,
                "te_rmse": None,
                "expr": None,
                "error": str(e)
            })
            continue

        expr = get_program_str(est)

        rows.append({
            "run": i,
            "seed": run_seed,
            "subset_idx": subset_idx,
            "tr_rmse": float(tr_rmse) if tr_rmse is not None else None,
            "te_rmse": float(te_rmse) if te_rmse is not None else None,
            "expr": expr
        })

        if te_rmse is not None and te_rmse < best_te:
            best_te = te_rmse
            best_est = est
            best_subset = subset_idx

        if (i % 10) == 0:
            print(f"[{i}/{N_RUNS}] checkpoint best_te: {best_te:.6f}")

    # Save raw sweep rows
    outdir = ROOT
    outdir.mkdir(parents=True, exist_ok=True)
    partial_csv = outdir / f"{ds_name}_sweep_partial.csv"
    pd.DataFrame(rows).to_csv(partial_csv, index=False)
    print("Saved partial sweep results to:", partial_csv)

    # --- finalize: evaluate baselines + RAE ---
    summary = {
        "dataset": ds_name,
        "n_samples_train": int(Xtr_local.shape[0]),
        "n_features": int(Xtr_local.shape[1]),
        "best_subset": best_subset,
        "best_te": float(best_te) if best_te != float("inf") else None,
        "timestamp": time.time(),
    }

    if best_est is not None and best_subset is not None:
        try:
            try:
                Xtr_sub = Xtr_local.iloc[:, best_subset]
                Xte_sub = Xte_local.iloc[:, best_subset]
            except Exception:
                Xtr_sub = Xtr_local.loc[:, best_subset]
                Xte_sub = Xte_local.loc[:, best_subset]

            sr_tr_feature = best_est.predict(Xtr_sub)
            sr_te_feature = best_est.predict(Xte_sub)

            Xtr_aug = Xtr_local.copy()
            Xte_aug = Xte_local.copy()
            Xtr_aug["SR_1"] = sr_tr_feature
            Xte_aug["SR_1"] = sr_te_feature

            baseline_metrics = eval_baselines(Xtr_local, Xte_local, ytr_local, yte_local)
            augmented_metrics = eval_baselines(Xtr_aug, Xte_aug, ytr_local, yte_local)
            dRMSE = {k.replace("_RMSE", "_dRMSE"): augmented_metrics[k] - baseline_metrics[k] for k in baseline_metrics}

            summary.update({
                "baseline": baseline_metrics,
                "augmented": augmented_metrics,
                "dRMSE": dRMSE
            })

            print("Baseline:", baseline_metrics)
            print("Augmented:", augmented_metrics)
            print("ΔRMSE:", dRMSE)

            with open(eval_json, "w") as f:
                json.dump(summary, f, indent=2)
            pd.DataFrame([summary]).to_csv(eval_csv, index=False)

            print("Saved evaluation summary JSON:", eval_json)
            print("Saved evaluation summary CSV:", eval_csv)

        except Exception as e:
            print("Error during final augmentation/evaluation:", e)
            traceback.print_exc()
    else:
        print("No successful SR model found during sweep for", ds_name)
        with open(eval_json, "w") as f:
            json.dump(summary, f, indent=2)
        pd.DataFrame([summary]).to_csv(eval_csv, index=False)

    print("Elapsed (s):", time.time() - start_ts)

Hybrid run config: {'dataset': 'openml_616', 'N_RUNS': 40, 'ALPHA': 2.0, 'BASE_SEED': 42, 'VAR_THRESH': 0.001}
Loaded dataset: openml_616 | train: (400, 50) | test: (100, 50) | used files: ['X_train.parquet', 'y_train.pkl']
After impute shapes: (400, 50) (100, 50)


/home/rshayan1/.local/lib/python3.12/site-packages/sklearn/cluster/_agglomerative.py:983: FutureWarning: Attribute `affinity` was deprecated in version 1.2 and will be removed in 1.4. Use `metric` instead
  warnings.warn(


make_correlated_groups: n_features=50, n_groups_requested=7, produced_groups=7
Sample groups (first 6): [[17, 26, 27, 46], [38, 41, 44, 47], [15, 19, 21, 37, 49], [8, 28, 29, 34, 35, 45], [5, 6, 7, 9, 32, 42], [12, 13, 16, 22, 24, 30, 31, 36, 39, 40, 48]]
Top features by imp (idx -> name -> score):
 3 3 0.2006
 4 4 0.1600
 1 1 0.1580
 0 0 0.1224
 2 2 0.0234
 16 16 0.0202
 28 28 0.0163
 38 38 0.0158
 30 30 0.0154
 24 24 0.0132
Correlated groups produced: 7 groups
Using importance array returned by make_correlated_groups.
Prepared groups_list length: 7 imp_arr shape: (50,)
Top importances (idx -> name -> score):
3 oz4 0.2006
4 oz5 0.1600
1 oz2 0.1580
0 oz1 0.1224
2 oz3 0.0234
16 oz17 0.0202
28 oz29 0.0163
38 oz39 0.0158
Starting hybrid sweep: N_RUNS= 40
[10/40] checkpoint best_te: 0.759397
[20/40] checkpoint best_te: 0.759397
[30/40] checkpoint best_te: 0.759397
[40/40] checkpoint best_te: 0.759397
Saved partial sweep results to: artifacts/hybrid_experiments_summary/openml_616_sweep_part

## Openml_618 Testing

In [26]:
# CELL: run_hybrid_for_one_dataset (openml_618 - NEW CONFIG)
import time, json, traceback
from pathlib import Path
import numpy as np
import pandas as pd

# === USER CONFIG ===
ds_name = "openml_618"
N_RUNS = 40
ALPHA = 2.0
BASE_SEED = 42
VAR_THRESH = 1e-3
ROOT = Path("artifacts") / "hybrid_experiments_summary"
ROOT.mkdir(parents=True, exist_ok=True)

print("Hybrid run config:", {
    "dataset": ds_name,
    "N_RUNS": N_RUNS,
    "ALPHA": ALPHA,
    "BASE_SEED": BASE_SEED,
    "VAR_THRESH": VAR_THRESH
})

# output guard
eval_json = ROOT / f"{ds_name}_evaluation_summary.json"
eval_csv = ROOT / f"{ds_name}_evaluation_summary.csv"
if eval_json.exists() or eval_csv.exists():
    print(f"Skipping {ds_name}: evaluation already exists at {eval_json} / {eval_csv}")
else:
    start_ts = time.time()

    # --- load canonical dataset ---
    ds_dir = Path("artifacts") / "canonical" / ds_name
    if "load_canonical_Xy" not in globals():
        raise RuntimeError("load_canonical_Xy(...) is not defined in this notebook. Define it first.")

    X_all, y_all, meta, used = load_canonical_Xy(ds_dir)

    # try explicit test split if present
    Xte = None
    yte = None
    try:
        if (ds_dir / "X_test.parquet").exists():
            Xte = pd.read_parquet(ds_dir / "X_test.parquet")
        if (ds_dir / "y_test.pkl").exists():
            yte = pd.read_pickle(ds_dir / "y_test.pkl").squeeze()
    except Exception as e:
        print("Warning loading explicit test split (ignored):", e)

    # if no explicit test, do deterministic 80/20 split
    if Xte is None or yte is None:
        from sklearn.model_selection import train_test_split
        Xtr_df, Xte_df, ytr_ser, yte_ser = train_test_split(
            pd.DataFrame(X_all), y_all, test_size=0.2, random_state=42
        )
        ytr = np.asarray(pd.to_numeric(ytr_ser, errors="coerce")).ravel()
        yte = np.asarray(pd.to_numeric(yte_ser, errors="coerce")).ravel()
    else:
        Xtr_df = pd.DataFrame(X_all)
        Xte_df = pd.DataFrame(Xte)
        ytr = np.asarray(pd.to_numeric(y_all, errors="coerce")).ravel()
        yte = np.asarray(pd.to_numeric(yte, errors="coerce")).ravel()

    # canonical local copies
    Xtr_local = Xtr_df.reset_index(drop=True)
    Xte_local = Xte_df.reset_index(drop=True)
    ytr_local = ytr
    yte_local = yte

    print("Loaded dataset:", ds_name, "| train:", Xtr_local.shape, "| test:", Xte_local.shape, "| used files:", used)

    # --- impute + coerce numeric defensively ---
    from sklearn.impute import SimpleImputer
    imp = SimpleImputer(strategy="median")
    if (Xtr_local.isna().sum().sum() > 0) or (Xte_local.isna().sum().sum() > 0):
        Xtr_imp = pd.DataFrame(imp.fit_transform(Xtr_local), columns=Xtr_local.columns, index=Xtr_local.index)
        Xte_imp = pd.DataFrame(imp.transform(Xte_local), columns=Xte_local.columns, index=Xte_local.index)
    else:
        Xtr_imp = Xtr_local.copy()
        Xte_imp = Xte_local.copy()

    for df in (Xtr_imp, Xte_imp):
        for c in df.columns:
            if not pd.api.types.is_numeric_dtype(df[c]):
                df[c] = pd.to_numeric(df[c], errors="coerce")
    Xtr_imp = Xtr_imp.fillna(0.0)
    Xte_imp = Xte_imp.fillna(0.0)

    Xtr_local = Xtr_imp.reset_index(drop=True)
    Xte_local = Xte_imp.reset_index(drop=True)
    ytr_local = np.asarray(pd.to_numeric(ytr_local, errors="coerce")).ravel()
    yte_local = np.asarray(pd.to_numeric(yte_local, errors="coerce")).ravel()

    print("After impute shapes:", Xtr_local.shape, Xte_local.shape)

    # --- build groups & importances ---
    n_feat = Xtr_local.shape[1]
    groups_list = None
    imp_arr = None

    def _coerce_groups_to_index_lists(groups_in, Xdf):
        if groups_in is None:
            return None
        out = []
        if isinstance(groups_in, dict):
            vals = list(groups_in.values())
        else:
            vals = list(groups_in)
        for g in vals:
            if isinstance(g, (list, tuple, np.ndarray)):
                idxs = []
                for el in g:
                    if isinstance(el, str):
                        try:
                            idxs.append(int(list(Xdf.columns).index(el)))
                        except Exception:
                            raise ValueError(f"Cannot map group element '{el}' -> not a column name")
                    else:
                        idxs.append(int(el))
                out.append(list(idxs))
            else:
                if isinstance(g, str):
                    idx = int(list(Xdf.columns).index(g))
                    out.append([idx])
                else:
                    out.append([int(g)])
        return out

    if "make_correlated_groups" in globals():
        try:
            mg_out = make_correlated_groups(Xtr_local, n_groups=None, random_state=BASE_SEED)
            groups_raw = None
            imp_from_groups = None

            if mg_out is None:
                groups_raw = None
            elif isinstance(mg_out, (tuple, list)):
                if len(mg_out) == 1:
                    first = mg_out[0]
                    if isinstance(first, (list, tuple, np.ndarray)):
                        groups_raw = first
                    else:
                        try:
                            arr = np.asarray(first, dtype=float)
                            if arr.shape[0] == n_feat:
                                imp_from_groups = arr
                            else:
                                groups_raw = first
                        except Exception:
                            groups_raw = first
                else:
                    groups_raw = mg_out[0]
                    try:
                        imp_from_groups = np.asarray(mg_out[1], dtype=float)
                    except Exception:
                        imp_from_groups = None
            elif isinstance(mg_out, dict):
                if "groups" in mg_out and "importance" in mg_out:
                    groups_raw = mg_out["groups"]
                    imp_from_groups = np.asarray(mg_out["importance"], dtype=float)
                else:
                    groups_raw = list(mg_out.values())
            elif isinstance(mg_out, np.ndarray):
                imp_from_groups = np.asarray(mg_out, dtype=float)
            else:
                groups_raw = mg_out

            if groups_raw is not None:
                try:
                    groups_list = _coerce_groups_to_index_lists(groups_raw, Xtr_local)
                    groups_list = [[int(j) for j in g if 0 <= int(j) < n_feat] for g in groups_list if len(g) > 0]
                    if len(groups_list) == 0:
                        groups_list = None
                    else:
                        print(f"Correlated groups produced: {len(groups_list)} groups")
                except Exception as e:
                    print("Could not coerce correlated groups (exception):", e)
                    groups_list = None

            if imp_from_groups is not None:
                try:
                    imp_arr = np.asarray(imp_from_groups, dtype=float)
                    print("Using importance array returned by make_correlated_groups.")
                except Exception:
                    imp_arr = None

        except Exception as e:
            print("make_correlated_groups failed (will fallback to RF one-feature-per-group). Exception:")
            traceback.print_exc()
            groups_list = None
    else:
        print("make_correlated_groups not found; will fallback to RF/one-feature groups.")

    if groups_list is None and "make_correlated_groups" in globals():
        try:
            print("Retrying make_correlated_groups with explicit n_groups heuristic...")
            mg_retry = make_correlated_groups(
                Xtr_local,
                n_groups=max(2, int(round(np.sqrt(n_feat)))),
                random_state=BASE_SEED
            )
            if isinstance(mg_retry, (tuple, list)) and len(mg_retry) >= 1:
                groups_raw = mg_retry[0]
                imp_from_groups = mg_retry[1] if len(mg_retry) > 1 else None
            else:
                groups_raw = mg_retry
                imp_from_groups = None

            if groups_raw is not None:
                groups_list = _coerce_groups_to_index_lists(groups_raw, Xtr_local)
                groups_list = [[int(j) for j in g if 0 <= int(j) < n_feat] for g in groups_list if len(g) > 0]
                if len(groups_list) == 0:
                    groups_list = None
                else:
                    print(f"Retry succeeded: {len(groups_list)} groups")

            if imp_from_groups is not None and imp_arr is None:
                try:
                    imp_arr = np.asarray(imp_from_groups, dtype=float)
                except Exception:
                    pass
        except Exception as e:
            print("Retry attempt failed (ignored):", e)

    # compute RF importances as fallback / canonical importance
    from sklearn.ensemble import RandomForestRegressor
    rf = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=BASE_SEED, n_jobs=-1)
    try:
        rf.fit(Xtr_local, ytr_local)
        rf_imp = np.asarray(rf.feature_importances_, dtype=float)
    except Exception as e:
        print("RF importance failed:", e)
        rf_imp = np.zeros(n_feat, dtype=float)

    if groups_list is None:
        groups_list = [[i] for i in range(n_feat)]
        print("Fallback groups: one-feature-per-group (count = {})".format(len(groups_list)))

    if imp_arr is None:
        imp_arr = rf_imp.copy()
    else:
        tmp = np.zeros(n_feat, dtype=float)
        tmp[:min(len(imp_arr), n_feat)] = imp_arr[:min(len(imp_arr), n_feat)]
        imp_arr = tmp

    if imp_arr.shape[0] != n_feat:
        tmp = np.zeros(n_feat, dtype=float)
        tmp[:min(len(imp_arr), n_feat)] = imp_arr[:min(len(imp_arr), n_feat)]
        imp_arr = tmp

    imp_arr = np.nan_to_num(imp_arr, nan=0.0, posinf=0.0, neginf=0.0)
    if imp_arr.sum() <= 0:
        imp_arr = np.ones(n_feat, dtype=float) / float(n_feat)
    else:
        imp_arr = imp_arr.astype(float) / float(np.sum(imp_arr))

    print("Prepared groups_list length:", len(groups_list), "imp_arr shape:", imp_arr.shape)
    order = np.argsort(-imp_arr)[:min(8, n_feat)]
    print("Top importances (idx -> name -> score):")
    for i in order:
        print(i, Xtr_local.columns[i], f"{imp_arr[i]:.4f}")

    for helper in ("sample_one_subset", "fit_one_sr_on_subset", "eval_baselines"):
        if helper not in globals():
            raise RuntimeError(f"Helper '{helper}' missing — paste its definition before running this cell.")

    def get_program_str(est):
        for attr in ("program", "program_", "_program"):
            prog = getattr(est, attr, None)
            if prog is not None:
                try:
                    return str(prog)
                except Exception:
                    return repr(prog)
        return None

    # --- SWEEP ---
    best_te = float("inf")
    best_est = None
    best_subset = None
    rows = []

    print("Starting hybrid sweep: N_RUNS=", N_RUNS)
    for i in range(1, N_RUNS + 1):
        run_seed = int(BASE_SEED + i)

        try:
            subset_idx = sample_one_subset(
                groups_list,
                imp_arr,
                alpha=ALPHA,
                pick_one_per_group=True,
                rng=np.random.default_rng(run_seed)
            )
        except TypeError:
            subset_idx = sample_one_subset(
                groups_list,
                imp_arr,
                alpha=ALPHA,
                pick_one_per_group=True
            )

        try:
            subset_idx = [int(x) for x in list(subset_idx)]
        except Exception:
            subset_idx = list(subset_idx) if hasattr(subset_idx, "__iter__") else []

        if len(subset_idx) == 0:
            print(f"[{i}] sampled empty subset — skipping")
            rows.append({
                "run": i,
                "seed": run_seed,
                "subset_idx": subset_idx,
                "tr_rmse": None,
                "te_rmse": None,
                "expr": None,
                "status": "empty_subset"
            })
            continue

        try:
            est, tr_rmse, te_rmse = fit_one_sr_on_subset(
                Xtr_local, ytr_local, Xte_local, subset_idx,
                function_set=('add', 'sub', 'mul', 'div', 'sin', 'cos'),
                parsimony=0.001,
                generations=100,
                population_size=1500,
                random_state=run_seed,
                verbose=0
            )
        except Exception as e:
            print(f"[{i}] SR training failed: {e}")
            rows.append({
                "run": i,
                "seed": run_seed,
                "subset_idx": subset_idx,
                "tr_rmse": None,
                "te_rmse": None,
                "expr": None,
                "error": str(e)
            })
            continue

        # extract expression string
        expr = get_program_str(est)

        rows.append({
            "run": i,
            "seed": run_seed,
            "subset_idx": subset_idx,
            "tr_rmse": float(tr_rmse) if tr_rmse is not None else None,
            "te_rmse": float(te_rmse) if te_rmse is not None else None,
            "expr": expr
        })

        if te_rmse is not None and te_rmse < best_te:
            best_te = te_rmse
            best_est = est
            best_subset = subset_idx

        if (i % 10) == 0:
            print(f"[{i}/{N_RUNS}] checkpoint best_te: {best_te:.6f}")

    # Save raw sweep rows
    outdir = ROOT
    outdir.mkdir(parents=True, exist_ok=True)
    partial_csv = outdir / f"{ds_name}_sweep_partial.csv"
    pd.DataFrame(rows).to_csv(partial_csv, index=False)
    print("Saved partial sweep results to:", partial_csv)

    # --- finalize: evaluate baselines + RAE ---
    summary = {
        "dataset": ds_name,
        "n_samples_train": int(Xtr_local.shape[0]),
        "n_features": int(Xtr_local.shape[1]),
        "best_subset": best_subset,
        "best_te": float(best_te) if best_te != float("inf") else None,
        "timestamp": time.time(),
    }

    if best_est is not None and best_subset is not None:
        try:
            try:
                Xtr_sub = Xtr_local.iloc[:, best_subset]
                Xte_sub = Xte_local.iloc[:, best_subset]
            except Exception:
                Xtr_sub = Xtr_local.loc[:, best_subset]
                Xte_sub = Xte_local.loc[:, best_subset]

            sr_tr_feature = best_est.predict(Xtr_sub)
            sr_te_feature = best_est.predict(Xte_sub)

            # final degeneracy check
            sr_std = float(np.nanstd(sr_tr_feature))
            if (not np.isfinite(sr_std)) or (sr_std < VAR_THRESH):
                print(f"Warning: Best SR feature is degenerate (sr_std={sr_std:.6g}); saving sweep summary without augmentation.")
                summary["note"] = "best_SR_feature_degenerate"
            else:
                Xtr_aug = Xtr_local.copy()
                Xte_aug = Xte_local.copy()
                Xtr_aug["SR_1"] = sr_tr_feature
                Xte_aug["SR_1"] = sr_te_feature

                baseline_metrics = eval_baselines(Xtr_local, Xte_local, ytr_local, yte_local)
                augmented_metrics = eval_baselines(Xtr_aug, Xte_aug, ytr_local, yte_local)

                # metric deltas for all reported metrics
                dMetrics = {
                    f"d{k}": augmented_metrics[k] - baseline_metrics[k]
                    for k in baseline_metrics
                }
                dRMSE = {k.replace("_RMSE", "_dRMSE"): v for k, v in dMetrics.items() if "_RMSE" in k}
                dRAE = {k.replace("_RAE", "_dRAE"): v for k, v in dMetrics.items() if "_RAE" in k and not k.endswith("_score")}
                dRAE_score = {k.replace("_RAE_score", "_dRAE_score"): v for k, v in dMetrics.items() if k.endswith("_RAE_score")}

                summary.update({
                    "baseline": baseline_metrics,
                    "augmented": augmented_metrics,
                    "dMetrics": dMetrics,
                    "dRMSE": dRMSE,
                    "dRAE": dRAE,
                    "dRAE_score": dRAE_score
                })

                print("Baseline:", baseline_metrics)
                print("Augmented:", augmented_metrics)
                print("ΔMetrics:", dMetrics)

            with open(eval_json, "w") as f:
                json.dump(summary, f, indent=2)
            pd.DataFrame([summary]).to_csv(eval_csv, index=False)

            print("Saved evaluation summary JSON:", eval_json)
            print("Saved evaluation summary CSV:", eval_csv)

        except Exception as e:
            print("Error during final augmentation/evaluation:", e)
            traceback.print_exc()
    else:
        print("No successful non-degenerate SR model found during sweep for", ds_name)
        with open(eval_json, "w") as f:
            json.dump(summary, f, indent=2)
        pd.DataFrame([summary]).to_csv(eval_csv, index=False)

    print("Elapsed (s):", time.time() - start_ts)

Hybrid run config: {'dataset': 'openml_618', 'N_RUNS': 40, 'ALPHA': 2.0, 'BASE_SEED': 42, 'VAR_THRESH': 0.001}
Loaded dataset: openml_618 | train: (800, 50) | test: (200, 50) | used files: ['X_train.parquet', 'y_train.pkl']
After impute shapes: (800, 50) (200, 50)


/home/rshayan1/.local/lib/python3.12/site-packages/sklearn/cluster/_agglomerative.py:983: FutureWarning: Attribute `affinity` was deprecated in version 1.2 and will be removed in 1.4. Use `metric` instead
  warnings.warn(


make_correlated_groups: n_features=50, n_groups_requested=7, produced_groups=7
Sample groups (first 6): [[6, 7, 36, 39], [12, 16, 26, 43], [0, 1, 2, 3, 4, 5, 37], [15, 19, 29, 35, 38, 42, 44, 49], [10, 20, 27, 28, 32, 34, 40, 48], [9, 11, 14, 21, 22, 23, 25, 31, 47]]
Top features by imp (idx -> name -> score):
 4 4 0.2499
 3 3 0.2130
 1 1 0.1834
 0 0 0.0918
 2 2 0.0175
 34 34 0.0115
 32 32 0.0098
 9 9 0.0089
 7 7 0.0077
 45 45 0.0070
Correlated groups produced: 7 groups
Using importance array returned by make_correlated_groups.
Prepared groups_list length: 7 imp_arr shape: (50,)
Top importances (idx -> name -> score):
4 oz5 0.2499
3 oz4 0.2130
1 oz2 0.1834
0 oz1 0.0918
2 oz3 0.0175
34 oz35 0.0115
32 oz33 0.0098
9 oz10 0.0089
Starting hybrid sweep: N_RUNS= 40
[10/40] checkpoint best_te: 0.810642
[20/40] checkpoint best_te: 0.788376
[30/40] checkpoint best_te: 0.753453
[40/40] checkpoint best_te: 0.753453
Saved partial sweep results to: artifacts/hybrid_experiments_summary/openml_618_swe

## Openml_620 Testing

In [35]:
# CELL: run_hybrid_for_one_dataset (openml_620 - NEW CONFIG)
import time, json, traceback
from pathlib import Path
import numpy as np
import pandas as pd

# === USER CONFIG ===
ds_name = "openml_620"
N_RUNS = 40
ALPHA = 2.0
BASE_SEED = 42
VAR_THRESH = 1e-3
ROOT = Path("artifacts") / "hybrid_experiments_summary"
ROOT.mkdir(parents=True, exist_ok=True)

print("Hybrid run config:", {
    "dataset": ds_name,
    "N_RUNS": N_RUNS,
    "ALPHA": ALPHA,
    "BASE_SEED": BASE_SEED,
    "VAR_THRESH": VAR_THRESH
})

# output guard
eval_json = ROOT / f"{ds_name}_evaluation_summary.json"
eval_csv = ROOT / f"{ds_name}_evaluation_summary.csv"
if eval_json.exists() or eval_csv.exists():
    print(f"Skipping {ds_name}: evaluation already exists at {eval_json} / {eval_csv}")
else:
    start_ts = time.time()

    # --- load canonical dataset ---
    ds_dir = Path("artifacts") / "canonical" / ds_name
    if "load_canonical_Xy" not in globals():
        raise RuntimeError("load_canonical_Xy(...) not found — define it before running this cell.")

    X_all, y_all, meta, used_files = load_canonical_Xy(ds_dir)

    # try explicit test split if present
    Xte = None
    yte = None
    try:
        if (ds_dir / "X_test.parquet").exists():
            Xte = pd.read_parquet(ds_dir / "X_test.parquet")
        if (ds_dir / "y_test.pkl").exists():
            yte = pd.read_pickle(ds_dir / "y_test.pkl").squeeze()
    except Exception as e:
        print("Warning loading explicit test split (ignored):", e)

    # if no explicit test, do deterministic 80/20 split
    if Xte is None or yte is None:
        from sklearn.model_selection import train_test_split
        Xtr_df, Xte_df, ytr_ser, yte_ser = train_test_split(
            pd.DataFrame(X_all), y_all, test_size=0.2, random_state=42
        )
        ytr = np.asarray(pd.to_numeric(ytr_ser, errors="coerce")).ravel()
        yte = np.asarray(pd.to_numeric(yte_ser, errors="coerce")).ravel()
    else:
        Xtr_df = pd.DataFrame(X_all)
        Xte_df = pd.DataFrame(Xte)
        ytr = np.asarray(pd.to_numeric(y_all, errors="coerce")).ravel()
        yte = np.asarray(pd.to_numeric(yte, errors="coerce")).ravel()

    # canonical local copies
    Xtr_local = Xtr_df.reset_index(drop=True)
    Xte_local = Xte_df.reset_index(drop=True)
    ytr_local = ytr
    yte_local = yte

    print("Loaded dataset:", ds_name, "| train:", Xtr_local.shape, "| test:", Xte_local.shape, "| used files:", used_files)

    # --- impute + coerce numeric defensively ---
    from sklearn.impute import SimpleImputer
    imp = SimpleImputer(strategy="median")
    if (Xtr_local.isna().sum().sum() > 0) or (Xte_local.isna().sum().sum() > 0):
        Xtr_imp = pd.DataFrame(imp.fit_transform(Xtr_local), columns=Xtr_local.columns, index=Xtr_local.index)
        Xte_imp = pd.DataFrame(imp.transform(Xte_local), columns=Xte_local.columns, index=Xte_local.index)
    else:
        Xtr_imp = Xtr_local.copy()
        Xte_imp = Xte_local.copy()

    for df in (Xtr_imp, Xte_imp):
        for c in df.columns:
            if not pd.api.types.is_numeric_dtype(df[c]):
                df[c] = pd.to_numeric(df[c], errors="coerce")
    Xtr_imp = Xtr_imp.fillna(0.0)
    Xte_imp = Xte_imp.fillna(0.0)

    Xtr_local = Xtr_imp.reset_index(drop=True)
    Xte_local = Xte_imp.reset_index(drop=True)
    ytr_local = np.asarray(pd.to_numeric(ytr_local, errors="coerce")).ravel()
    yte_local = np.asarray(pd.to_numeric(yte_local, errors="coerce")).ravel()

    print("After impute shapes:", Xtr_local.shape, Xte_local.shape)

    # --- build groups & importances ---
    n_feat = Xtr_local.shape[1]
    groups_list = None
    imp_arr = None

    def _coerce_groups_to_index_lists(groups_in, Xdf):
        if groups_in is None:
            return None
        out = []
        if isinstance(groups_in, dict):
            vals = list(groups_in.values())
        else:
            vals = list(groups_in)
        for g in vals:
            if isinstance(g, (list, tuple, np.ndarray)):
                idxs = []
                for el in g:
                    if isinstance(el, str):
                        try:
                            idxs.append(int(list(Xdf.columns).index(el)))
                        except Exception:
                            raise ValueError(f"Cannot map group element '{el}' -> not a column name")
                    else:
                        idxs.append(int(el))
                out.append(list(idxs))
            else:
                if isinstance(g, str):
                    idx = int(list(Xdf.columns).index(g))
                    out.append([idx])
                else:
                    out.append([int(g)])
        return out

    if "make_correlated_groups" in globals():
        try:
            mg_out = make_correlated_groups(Xtr_local, n_groups=None, random_state=BASE_SEED)
            groups_raw = None
            imp_from_groups = None

            if mg_out is None:
                groups_raw = None
            elif isinstance(mg_out, (tuple, list)):
                if len(mg_out) == 1:
                    first = mg_out[0]
                    if isinstance(first, (list, tuple, np.ndarray)):
                        groups_raw = first
                    else:
                        try:
                            arr = np.asarray(first, dtype=float)
                            if arr.shape[0] == n_feat:
                                imp_from_groups = arr
                            else:
                                groups_raw = first
                        except Exception:
                            groups_raw = first
                else:
                    groups_raw = mg_out[0]
                    try:
                        imp_from_groups = np.asarray(mg_out[1], dtype=float)
                    except Exception:
                        imp_from_groups = None
            elif isinstance(mg_out, dict):
                if "groups" in mg_out and "importance" in mg_out:
                    groups_raw = mg_out["groups"]
                    imp_from_groups = np.asarray(mg_out["importance"], dtype=float)
                else:
                    groups_raw = list(mg_out.values())
            elif isinstance(mg_out, np.ndarray):
                imp_from_groups = np.asarray(mg_out, dtype=float)
            else:
                groups_raw = mg_out

            if groups_raw is not None:
                try:
                    groups_list = _coerce_groups_to_index_lists(groups_raw, Xtr_local)
                    groups_list = [[int(j) for j in g if 0 <= int(j) < n_feat] for g in groups_list if len(g) > 0]
                    if len(groups_list) == 0:
                        groups_list = None
                    else:
                        print(f"Correlated groups produced: {len(groups_list)} groups")
                except Exception as e:
                    print("Could not coerce correlated groups (exception):", e)
                    groups_list = None

            if imp_from_groups is not None:
                try:
                    imp_arr = np.asarray(imp_from_groups, dtype=float)
                    print("Using importance array returned by make_correlated_groups.")
                except Exception:
                    imp_arr = None

        except Exception as e:
            print("make_correlated_groups failed (will fallback to RF one-feature-per-group). Exception:")
            traceback.print_exc()
            groups_list = None
    else:
        print("make_correlated_groups not found; will fallback to RF/one-feature groups.")

    if groups_list is None and "make_correlated_groups" in globals():
        try:
            print("Retrying make_correlated_groups with explicit n_groups heuristic...")
            mg_retry = make_correlated_groups(
                Xtr_local,
                n_groups=max(2, int(round(np.sqrt(n_feat)))),
                random_state=BASE_SEED
            )
            if isinstance(mg_retry, (tuple, list)) and len(mg_retry) >= 1:
                groups_raw = mg_retry[0]
                imp_from_groups = mg_retry[1] if len(mg_retry) > 1 else None
            else:
                groups_raw = mg_retry
                imp_from_groups = None

            if groups_raw is not None:
                groups_list = _coerce_groups_to_index_lists(groups_raw, Xtr_local)
                groups_list = [[int(j) for j in g if 0 <= int(j) < n_feat] for g in groups_list if len(g) > 0]
                if len(groups_list) == 0:
                    groups_list = None
                else:
                    print(f"Retry succeeded: {len(groups_list)} groups")

            if imp_from_groups is not None and imp_arr is None:
                try:
                    imp_arr = np.asarray(imp_from_groups, dtype=float)
                except Exception:
                    pass
        except Exception as e:
            print("Retry attempt failed (ignored):", e)

    from sklearn.ensemble import RandomForestRegressor
    rf = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=BASE_SEED, n_jobs=-1)
    try:
        rf.fit(Xtr_local, ytr_local)
        rf_imp = np.asarray(rf.feature_importances_, dtype=float)
    except Exception as e:
        print("RF importance failed:", e)
        rf_imp = np.zeros(n_feat, dtype=float)

    if groups_list is None:
        groups_list = [[i] for i in range(n_feat)]
        print("Fallback groups: one-feature-per-group (count = {})".format(len(groups_list)))

    if imp_arr is None:
        imp_arr = rf_imp.copy()
    else:
        tmp = np.zeros(n_feat, dtype=float)
        tmp[:min(len(imp_arr), n_feat)] = imp_arr[:min(len(imp_arr), n_feat)]
        imp_arr = tmp

    if imp_arr.shape[0] != n_feat:
        tmp = np.zeros(n_feat, dtype=float)
        tmp[:min(len(imp_arr), n_feat)] = imp_arr[:min(len(imp_arr), n_feat)]
        imp_arr = tmp

    imp_arr = np.nan_to_num(imp_arr, nan=0.0, posinf=0.0, neginf=0.0)
    if imp_arr.sum() <= 0:
        imp_arr = np.ones(n_feat, dtype=float) / float(n_feat)
    else:
        imp_arr = imp_arr.astype(float) / float(np.sum(imp_arr))

    print("Prepared groups_list length:", len(groups_list), "imp_arr shape:", imp_arr.shape)
    order = np.argsort(-imp_arr)[:min(8, n_feat)]
    print("Top importances (idx -> name -> score):")
    for i in order:
        print(i, Xtr_local.columns[i], f"{imp_arr[i]:.4f}")

    for helper in ("sample_one_subset", "fit_one_sr_on_subset", "eval_baselines"):
        if helper not in globals():
            raise RuntimeError(f"Helper '{helper}' missing — paste its definition before running this cell.")

    def get_program_str(est):
        for attr in ("program", "program_", "_program"):
            prog = getattr(est, attr, None)
            if prog is not None:
                try:
                    return str(prog)
                except Exception:
                    return repr(prog)
        return None

    # --- SWEEP ---
    best_te = float("inf")
    best_est = None
    best_subset = None
    rows = []

    print("Starting hybrid sweep: N_RUNS=", N_RUNS)
    for i in range(1, N_RUNS + 1):
        run_seed = int(BASE_SEED + i)

        try:
            subset_idx = sample_one_subset(
                groups_list,
                imp_arr,
                alpha=ALPHA,
                pick_one_per_group=True,
                rng=np.random.default_rng(run_seed)
            )
        except TypeError:
            subset_idx = sample_one_subset(
                groups_list,
                imp_arr,
                alpha=ALPHA,
                pick_one_per_group=True
            )

        try:
            subset_idx = [int(x) for x in list(subset_idx)]
        except Exception:
            subset_idx = list(subset_idx) if hasattr(subset_idx, "__iter__") else []

        if len(subset_idx) == 0:
            print(f"[{i}] sampled empty subset — skipping")
            rows.append({
                "run": i,
                "seed": run_seed,
                "subset_idx": subset_idx,
                "tr_rmse": None,
                "te_rmse": None,
                "expr": None,
                "status": "empty_subset"
            })
            continue

        try:
            est, tr_rmse, te_rmse = fit_one_sr_on_subset(
                Xtr_local, ytr_local, Xte_local, subset_idx,
                function_set=('add', 'sub', 'mul', 'div', 'sin', 'cos'),
                parsimony=0.001,
                generations=100,
                population_size=1500,
                random_state=run_seed,
                verbose=0
            )
        except Exception as e:
            print(f"[{i}] SR training failed: {e}")
            rows.append({
                "run": i,
                "seed": run_seed,
                "subset_idx": subset_idx,
                "tr_rmse": None,
                "te_rmse": None,
                "expr": None,
                "error": str(e)
            })
            continue

        expr = get_program_str(est)

        rows.append({
            "run": i,
            "seed": run_seed,
            "subset_idx": subset_idx,
            "tr_rmse": float(tr_rmse) if tr_rmse is not None else None,
            "te_rmse": float(te_rmse) if te_rmse is not None else None,
            "expr": expr
        })

        if te_rmse is not None and te_rmse < best_te:
            best_te = te_rmse
            best_est = est
            best_subset = subset_idx

        if (i % 10) == 0:
            print(f"[{i}/{N_RUNS}] checkpoint best_te: {best_te:.6f}")

    # Save raw sweep rows
    outdir = ROOT
    outdir.mkdir(parents=True, exist_ok=True)
    partial_csv = outdir / f"{ds_name}_sweep_partial.csv"
    pd.DataFrame(rows).to_csv(partial_csv, index=False)
    print("Saved partial sweep results to:", partial_csv)

    # --- finalize: evaluate baselines + RAE ---
    summary = {
        "dataset": ds_name,
        "n_samples_train": int(Xtr_local.shape[0]),
        "n_features": int(Xtr_local.shape[1]),
        "best_subset": best_subset,
        "best_te": float(best_te) if best_te != float("inf") else None,
        "timestamp": time.time(),
    }

    if best_est is not None and best_subset is not None:
        try:
            try:
                Xtr_sub = Xtr_local.iloc[:, best_subset]
                Xte_sub = Xte_local.iloc[:, best_subset]
            except Exception:
                Xtr_sub = Xtr_local.loc[:, best_subset]
                Xte_sub = Xte_local.loc[:, best_subset]

            sr_tr_feature = best_est.predict(Xtr_sub)
            sr_te_feature = best_est.predict(Xte_sub)

            # final degeneracy check
            sr_std = float(np.nanstd(sr_tr_feature))
            if (not np.isfinite(sr_std)) or (sr_std < VAR_THRESH):
                print(f"Warning: Best SR feature is degenerate (sr_std={sr_std:.6g}); saving sweep summary without augmentation.")
                summary["note"] = "best_SR_feature_degenerate"
            else:
                Xtr_aug = Xtr_local.copy()
                Xte_aug = Xte_local.copy()
                Xtr_aug["SR_1"] = sr_tr_feature
                Xte_aug["SR_1"] = sr_te_feature

                baseline_metrics = eval_baselines(Xtr_local, Xte_local, ytr_local, yte_local)
                augmented_metrics = eval_baselines(Xtr_aug, Xte_aug, ytr_local, yte_local)

                dMetrics = {
                    f"d{k}": augmented_metrics[k] - baseline_metrics[k]
                    for k in baseline_metrics
                }
                dRMSE = {k.replace("_RMSE", "_dRMSE"): v for k, v in dMetrics.items() if "_RMSE" in k}
                dRAE = {k.replace("_RAE", "_dRAE"): v for k, v in dMetrics.items() if "_RAE" in k and not k.endswith("_score")}
                dRAE_score = {k.replace("_RAE_score", "_dRAE_score"): v for k, v in dMetrics.items() if k.endswith("_RAE_score")}

                summary.update({
                    "baseline": baseline_metrics,
                    "augmented": augmented_metrics,
                    "dMetrics": dMetrics,
                    "dRMSE": dRMSE,
                    "dRAE": dRAE,
                    "dRAE_score": dRAE_score
                })

                print("Baseline:", baseline_metrics)
                print("Augmented:", augmented_metrics)
                print("ΔMetrics:", dMetrics)

            with open(eval_json, "w") as f:
                json.dump(summary, f, indent=2)
            pd.DataFrame([summary]).to_csv(eval_csv, index=False)

            print("Saved evaluation summary JSON:", eval_json)
            print("Saved evaluation summary CSV:", eval_csv)

        except Exception as e:
            print("Error during final augmentation/evaluation:", e)
            traceback.print_exc()
    else:
        print("No successful non-degenerate SR model found during sweep for", ds_name)
        with open(eval_json, "w") as f:
            json.dump(summary, f, indent=2)
        pd.DataFrame([summary]).to_csv(eval_csv, index=False)

    print("Elapsed (s):", time.time() - start_ts)

Hybrid run config: {'dataset': 'openml_620', 'N_RUNS': 40, 'ALPHA': 2.0, 'BASE_SEED': 42, 'VAR_THRESH': 0.001}
Loaded dataset: openml_620 | train: (800, 25) | test: (200, 25) | used files: ['X_train.parquet', 'y_train.pkl']
After impute shapes: (800, 25) (200, 25)


/home/rshayan1/.local/lib/python3.12/site-packages/sklearn/cluster/_agglomerative.py:983: FutureWarning: Attribute `affinity` was deprecated in version 1.2 and will be removed in 1.4. Use `metric` instead
  warnings.warn(


make_correlated_groups: n_features=25, n_groups_requested=5, produced_groups=5
Sample groups (first 6): [[7, 14, 16], [5, 11, 18, 19], [4, 9, 17, 21, 24], [3, 13, 15, 22, 23], [0, 1, 2, 6, 8, 10, 12, 20]]
Top features by imp (idx -> name -> score):
 1 1 0.2729
 0 0 0.2696
 3 3 0.1859
 4 4 0.0715
 2 2 0.0423
 16 16 0.0129
 13 13 0.0119
 12 12 0.0113
 17 17 0.0111
 14 14 0.0084
Correlated groups produced: 5 groups
Using importance array returned by make_correlated_groups.
Prepared groups_list length: 5 imp_arr shape: (25,)
Top importances (idx -> name -> score):
1 oz2 0.2729
0 oz1 0.2696
3 oz4 0.1859
4 oz5 0.0715
2 oz3 0.0423
16 oz17 0.0129
13 oz14 0.0119
12 oz13 0.0113
Starting hybrid sweep: N_RUNS= 40
[10/40] checkpoint best_te: 0.578717
[20/40] checkpoint best_te: 0.552048
[30/40] checkpoint best_te: 0.552048
[40/40] checkpoint best_te: 0.552048
Saved partial sweep results to: artifacts/hybrid_experiments_summary/openml_620_sweep_partial.csv
Baseline: {'LR_RMSE': 0.8545424367898794, '

## Openml_637 Testing

In [37]:
# CELL: run_hybrid_for_one_dataset (openml_637 - NEW CONFIG)
import time, json, traceback
from pathlib import Path
import numpy as np
import pandas as pd

# === USER CONFIG ===
ds_name = "openml_637"
N_RUNS = 40
ALPHA = 2.0
BASE_SEED = 42
VAR_THRESH = 1e-3
ROOT = Path("artifacts") / "hybrid_experiments_summary"
ROOT.mkdir(parents=True, exist_ok=True)

print("Hybrid run config:", {
    "dataset": ds_name,
    "N_RUNS": N_RUNS,
    "ALPHA": ALPHA,
    "BASE_SEED": BASE_SEED,
    "VAR_THRESH": VAR_THRESH
})

# output guard
eval_json = ROOT / f"{ds_name}_evaluation_summary.json"
eval_csv = ROOT / f"{ds_name}_evaluation_summary.csv"
if eval_json.exists() or eval_csv.exists():
    print(f"Skipping {ds_name}: evaluation already exists at {eval_json} / {eval_csv}")
else:
    start_ts = time.time()

    # --- load canonical dataset ---
    ds_dir = Path("artifacts") / "canonical" / ds_name
    if "load_canonical_Xy" not in globals():
        raise RuntimeError("load_canonical_Xy(...) not found — define it before running this cell.")

    X_all, y_all, meta, used_files = load_canonical_Xy(ds_dir)

    # try explicit test split if present
    Xte = None
    yte = None
    try:
        if (ds_dir / "X_test.parquet").exists():
            Xte = pd.read_parquet(ds_dir / "X_test.parquet")
        if (ds_dir / "y_test.pkl").exists():
            yte = pd.read_pickle(ds_dir / "y_test.pkl").squeeze()
    except Exception as e:
        print("Warning loading explicit test split (ignored):", e)

    # if no explicit test, do deterministic 80/20 split
    if Xte is None or yte is None:
        from sklearn.model_selection import train_test_split
        Xtr_df, Xte_df, ytr_ser, yte_ser = train_test_split(
            pd.DataFrame(X_all), y_all, test_size=0.2, random_state=42
        )
        ytr = np.asarray(pd.to_numeric(ytr_ser, errors="coerce")).ravel()
        yte = np.asarray(pd.to_numeric(yte_ser, errors="coerce")).ravel()
    else:
        Xtr_df = pd.DataFrame(X_all)
        Xte_df = pd.DataFrame(Xte)
        ytr = np.asarray(pd.to_numeric(y_all, errors="coerce")).ravel()
        yte = np.asarray(pd.to_numeric(yte, errors="coerce")).ravel()

    # canonical local copies
    Xtr_local = Xtr_df.reset_index(drop=True)
    Xte_local = Xte_df.reset_index(drop=True)
    ytr_local = ytr
    yte_local = yte

    print("Loaded dataset:", ds_name, "| train:", Xtr_local.shape, "| test:", Xte_local.shape, "| used files:", used_files)

    # --- impute + coerce numeric defensively ---
    from sklearn.impute import SimpleImputer
    imp = SimpleImputer(strategy="median")
    if (Xtr_local.isna().sum().sum() > 0) or (Xte_local.isna().sum().sum() > 0):
        Xtr_imp = pd.DataFrame(imp.fit_transform(Xtr_local), columns=Xtr_local.columns, index=Xtr_local.index)
        Xte_imp = pd.DataFrame(imp.transform(Xte_local), columns=Xte_local.columns, index=Xte_local.index)
    else:
        Xtr_imp = Xtr_local.copy()
        Xte_imp = Xte_local.copy()

    for df in (Xtr_imp, Xte_imp):
        for c in df.columns:
            if not pd.api.types.is_numeric_dtype(df[c]):
                df[c] = pd.to_numeric(df[c], errors="coerce")
    Xtr_imp = Xtr_imp.fillna(0.0)
    Xte_imp = Xte_imp.fillna(0.0)

    Xtr_local = Xtr_imp.reset_index(drop=True)
    Xte_local = Xte_imp.reset_index(drop=True)
    ytr_local = np.asarray(pd.to_numeric(ytr_local, errors="coerce")).ravel()
    yte_local = np.asarray(pd.to_numeric(yte_local, errors="coerce")).ravel()

    print("After impute shapes:", Xtr_local.shape, Xte_local.shape)

    # --- build groups & importances ---
    n_feat = Xtr_local.shape[1]
    groups_list = None
    imp_arr = None

    def _coerce_groups_to_index_lists(groups_in, Xdf):
        if groups_in is None:
            return None
        out = []
        if isinstance(groups_in, dict):
            vals = list(groups_in.values())
        else:
            vals = list(groups_in)
        for g in vals:
            if isinstance(g, (list, tuple, np.ndarray)):
                idxs = []
                for el in g:
                    if isinstance(el, str):
                        try:
                            idxs.append(int(list(Xdf.columns).index(el)))
                        except Exception:
                            raise ValueError(f"Cannot map group element '{el}' -> not a column name")
                    else:
                        idxs.append(int(el))
                out.append(list(idxs))
            else:
                if isinstance(g, str):
                    idx = int(list(Xdf.columns).index(g))
                    out.append([idx])
                else:
                    out.append([int(g)])
        return out

    if "make_correlated_groups" in globals():
        try:
            mg_out = make_correlated_groups(Xtr_local, n_groups=None, random_state=BASE_SEED)
            groups_raw = None
            imp_from_groups = None

            if mg_out is None:
                groups_raw = None
            elif isinstance(mg_out, (tuple, list)):
                if len(mg_out) == 1:
                    first = mg_out[0]
                    if isinstance(first, (list, tuple, np.ndarray)):
                        groups_raw = first
                    else:
                        try:
                            arr = np.asarray(first, dtype=float)
                            if arr.shape[0] == n_feat:
                                imp_from_groups = arr
                            else:
                                groups_raw = first
                        except Exception:
                            groups_raw = first
                else:
                    groups_raw = mg_out[0]
                    try:
                        imp_from_groups = np.asarray(mg_out[1], dtype=float)
                    except Exception:
                        imp_from_groups = None
            elif isinstance(mg_out, dict):
                if "groups" in mg_out and "importance" in mg_out:
                    groups_raw = mg_out["groups"]
                    imp_from_groups = np.asarray(mg_out["importance"], dtype=float)
                else:
                    groups_raw = list(mg_out.values())
            elif isinstance(mg_out, np.ndarray):
                imp_from_groups = np.asarray(mg_out, dtype=float)
            else:
                groups_raw = mg_out

            if groups_raw is not None:
                try:
                    groups_list = _coerce_groups_to_index_lists(groups_raw, Xtr_local)
                    groups_list = [[int(j) for j in g if 0 <= int(j) < n_feat] for g in groups_list if len(g) > 0]
                    if len(groups_list) == 0:
                        groups_list = None
                    else:
                        print(f"Correlated groups produced: {len(groups_list)} groups")
                except Exception as e:
                    print("Could not coerce correlated groups (exception):", e)
                    groups_list = None

            if imp_from_groups is not None:
                try:
                    imp_arr = np.asarray(imp_from_groups, dtype=float)
                    print("Using importance array returned by make_correlated_groups.")
                except Exception:
                    imp_arr = None

        except Exception as e:
            print("make_correlated_groups failed (will fallback to RF one-feature-per-group). Exception:")
            traceback.print_exc()
            groups_list = None
    else:
        print("make_correlated_groups not found; will fallback to RF/one-feature groups.")

    if groups_list is None and "make_correlated_groups" in globals():
        try:
            print("Retrying make_correlated_groups with explicit n_groups heuristic...")
            mg_retry = make_correlated_groups(
                Xtr_local,
                n_groups=max(2, int(round(np.sqrt(n_feat)))),
                random_state=BASE_SEED
            )
            if isinstance(mg_retry, (tuple, list)) and len(mg_retry) >= 1:
                groups_raw = mg_retry[0]
                imp_from_groups = mg_retry[1] if len(mg_retry) > 1 else None
            else:
                groups_raw = mg_retry
                imp_from_groups = None

            if groups_raw is not None:
                groups_list = _coerce_groups_to_index_lists(groups_raw, Xtr_local)
                groups_list = [[int(j) for j in g if 0 <= int(j) < n_feat] for g in groups_list if len(g) > 0]
                if len(groups_list) == 0:
                    groups_list = None
                else:
                    print(f"Retry succeeded: {len(groups_list)} groups")

            if imp_from_groups is not None and imp_arr is None:
                try:
                    imp_arr = np.asarray(imp_from_groups, dtype=float)
                except Exception:
                    pass
        except Exception as e:
            print("Retry attempt failed (ignored):", e)

    from sklearn.ensemble import RandomForestRegressor
    rf = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=BASE_SEED, n_jobs=-1)
    try:
        rf.fit(Xtr_local, ytr_local)
        rf_imp = np.asarray(rf.feature_importances_, dtype=float)
    except Exception as e:
        print("RF importance failed:", e)
        rf_imp = np.zeros(n_feat, dtype=float)

    if groups_list is None:
        groups_list = [[i] for i in range(n_feat)]
        print("Fallback groups: one-feature-per-group (count = {})".format(len(groups_list)))

    if imp_arr is None:
        imp_arr = rf_imp.copy()
    else:
        tmp = np.zeros(n_feat, dtype=float)
        tmp[:min(len(imp_arr), n_feat)] = imp_arr[:min(len(imp_arr), n_feat)]
        imp_arr = tmp

    if imp_arr.shape[0] != n_feat:
        tmp = np.zeros(n_feat, dtype=float)
        tmp[:min(len(imp_arr), n_feat)] = imp_arr[:min(len(imp_arr), n_feat)]
        imp_arr = tmp

    imp_arr = np.nan_to_num(imp_arr, nan=0.0, posinf=0.0, neginf=0.0)
    if imp_arr.sum() <= 0:
        imp_arr = np.ones(n_feat, dtype=float) / float(n_feat)
    else:
        imp_arr = imp_arr.astype(float) / float(np.sum(imp_arr))

    print("Prepared groups_list length:", len(groups_list), "imp_arr shape:", imp_arr.shape)
    order = np.argsort(-imp_arr)[:min(8, n_feat)]
    print("Top importances (idx -> name -> score):")
    for i in order:
        print(i, Xtr_local.columns[i], f"{imp_arr[i]:.4f}")

    for helper in ("sample_one_subset", "fit_one_sr_on_subset", "eval_baselines"):
        if helper not in globals():
            raise RuntimeError(f"Helper '{helper}' missing — paste its definition before running this cell.")

    def get_program_str(est):
        for attr in ("program", "program_", "_program"):
            prog = getattr(est, attr, None)
            if prog is not None:
                try:
                    return str(prog)
                except Exception:
                    return repr(prog)
        return None

    # --- SWEEP ---
    best_te = float("inf")
    best_est = None
    best_subset = None
    rows = []

    print("Starting hybrid sweep: N_RUNS=", N_RUNS)
    for i in range(1, N_RUNS + 1):
        run_seed = int(BASE_SEED + i)

        try:
            subset_idx = sample_one_subset(
                groups_list,
                imp_arr,
                alpha=ALPHA,
                pick_one_per_group=True,
                rng=np.random.default_rng(run_seed)
            )
        except TypeError:
            subset_idx = sample_one_subset(
                groups_list,
                imp_arr,
                alpha=ALPHA,
                pick_one_per_group=True
            )

        try:
            subset_idx = [int(x) for x in list(subset_idx)]
        except Exception:
            subset_idx = list(subset_idx) if hasattr(subset_idx, "__iter__") else []

        if len(subset_idx) == 0:
            print(f"[{i}] sampled empty subset — skipping")
            rows.append({
                "run": i,
                "seed": run_seed,
                "subset_idx": subset_idx,
                "tr_rmse": None,
                "te_rmse": None,
                "expr": None,
                "status": "empty_subset"
            })
            continue

        try:
            est, tr_rmse, te_rmse = fit_one_sr_on_subset(
                Xtr_local, ytr_local, Xte_local, subset_idx,
                function_set=('add', 'sub', 'mul', 'div', 'sin', 'cos'),
                parsimony=0.001,
                generations=100,
                population_size=1500,
                random_state=run_seed,
                verbose=0
            )
        except Exception as e:
            print(f"[{i}] SR training failed: {e}")
            rows.append({
                "run": i,
                "seed": run_seed,
                "subset_idx": subset_idx,
                "tr_rmse": None,
                "te_rmse": None,
                "expr": None,
                "error": str(e)
            })
            continue

        expr = get_program_str(est)

        rows.append({
            "run": i,
            "seed": run_seed,
            "subset_idx": subset_idx,
            "tr_rmse": float(tr_rmse) if tr_rmse is not None else None,
            "te_rmse": float(te_rmse) if te_rmse is not None else None,
            "expr": expr
        })

        if te_rmse is not None and te_rmse < best_te:
            best_te = te_rmse
            best_est = est
            best_subset = subset_idx

        if (i % 10) == 0:
            print(f"[{i}/{N_RUNS}] checkpoint best_te: {best_te:.6f}")

    # Save raw sweep rows
    outdir = ROOT
    outdir.mkdir(parents=True, exist_ok=True)
    partial_csv = outdir / f"{ds_name}_sweep_partial.csv"
    pd.DataFrame(rows).to_csv(partial_csv, index=False)
    print("Saved partial sweep results to:", partial_csv)

    # --- finalize: evaluate baselines + RAE ---
    summary = {
        "dataset": ds_name,
        "n_samples_train": int(Xtr_local.shape[0]),
        "n_features": int(Xtr_local.shape[1]),
        "best_subset": best_subset,
        "best_te": float(best_te) if best_te != float("inf") else None,
        "timestamp": time.time(),
    }

    if best_est is not None and best_subset is not None:
        try:
            try:
                Xtr_sub = Xtr_local.iloc[:, best_subset]
                Xte_sub = Xte_local.iloc[:, best_subset]
            except Exception:
                Xtr_sub = Xtr_local.loc[:, best_subset]
                Xte_sub = Xte_local.loc[:, best_subset]

            sr_tr_feature = best_est.predict(Xtr_sub)
            sr_te_feature = best_est.predict(Xte_sub)

            sr_std = float(np.nanstd(sr_tr_feature))
            if (not np.isfinite(sr_std)) or (sr_std < VAR_THRESH):
                print(f"Warning: Best SR feature is degenerate (sr_std={sr_std:.6g}); saving sweep summary without augmentation.")
                summary["note"] = "best_SR_feature_degenerate"
            else:
                Xtr_aug = Xtr_local.copy()
                Xte_aug = Xte_local.copy()
                Xtr_aug["SR_1"] = sr_tr_feature
                Xte_aug["SR_1"] = sr_te_feature

                baseline_metrics = eval_baselines(Xtr_local, Xte_local, ytr_local, yte_local)
                augmented_metrics = eval_baselines(Xtr_aug, Xte_aug, ytr_local, yte_local)

                dMetrics = {
                    f"d{k}": augmented_metrics[k] - baseline_metrics[k]
                    for k in baseline_metrics
                }
                dRMSE = {k.replace("_RMSE", "_dRMSE"): v for k, v in dMetrics.items() if "_RMSE" in k}
                dRAE = {k.replace("_RAE", "_dRAE"): v for k, v in dMetrics.items() if "_RAE" in k and not k.endswith("_score")}
                dRAE_score = {k.replace("_RAE_score", "_dRAE_score"): v for k, v in dMetrics.items() if k.endswith("_RAE_score")}

                summary.update({
                    "baseline": baseline_metrics,
                    "augmented": augmented_metrics,
                    "dMetrics": dMetrics,
                    "dRMSE": dRMSE,
                    "dRAE": dRAE,
                    "dRAE_score": dRAE_score
                })

                print("Baseline:", baseline_metrics)
                print("Augmented:", augmented_metrics)
                print("ΔMetrics:", dMetrics)

            with open(eval_json, "w") as f:
                json.dump(summary, f, indent=2)
            pd.DataFrame([summary]).to_csv(eval_csv, index=False)

            print("Saved evaluation summary JSON:", eval_json)
            print("Saved evaluation summary CSV:", eval_csv)

        except Exception as e:
            print("Error during final augmentation/evaluation:", e)
            traceback.print_exc()
    else:
        print("No successful non-degenerate SR model found during sweep for", ds_name)
        with open(eval_json, "w") as f:
            json.dump(summary, f, indent=2)
        pd.DataFrame([summary]).to_csv(eval_csv, index=False)

    print("Elapsed (s):", time.time() - start_ts)

Hybrid run config: {'dataset': 'openml_637', 'N_RUNS': 40, 'ALPHA': 2.0, 'BASE_SEED': 42, 'VAR_THRESH': 0.001}
Loaded dataset: openml_637 | train: (400, 50) | test: (100, 50) | used files: ['X_train.parquet', 'y_train.pkl']
After impute shapes: (400, 50) (100, 50)


/home/rshayan1/.local/lib/python3.12/site-packages/sklearn/cluster/_agglomerative.py:983: FutureWarning: Attribute `affinity` was deprecated in version 1.2 and will be removed in 1.4. Use `metric` instead
  warnings.warn(


make_correlated_groups: n_features=50, n_groups_requested=7, produced_groups=7
Sample groups (first 6): [[17, 27, 30], [19, 23, 41, 43], [12, 14, 20, 25, 33], [7, 13, 26, 31, 46], [5, 10, 15, 22, 24, 49], [0, 1, 2, 11, 16, 28, 32, 36, 39, 44]]
Top features by imp (idx -> name -> score):
 1 1 0.2498
 0 0 0.1305
 3 3 0.0997
 4 4 0.0378
 2 2 0.0336
 16 16 0.0281
 49 49 0.0242
 33 33 0.0230
 36 36 0.0226
 45 45 0.0192
Correlated groups produced: 7 groups
Using importance array returned by make_correlated_groups.
Prepared groups_list length: 7 imp_arr shape: (50,)
Top importances (idx -> name -> score):
1 oz2 0.2498
0 oz1 0.1305
3 oz4 0.0997
4 oz5 0.0378
2 oz3 0.0336
16 oz17 0.0281
49 oz50 0.0242
33 oz34 0.0230
Starting hybrid sweep: N_RUNS= 40
[10/40] checkpoint best_te: 0.636645
[20/40] checkpoint best_te: 0.620522
[30/40] checkpoint best_te: 0.620522
[40/40] checkpoint best_te: 0.620522
Saved partial sweep results to: artifacts/hybrid_experiments_summary/openml_637_sweep_partial.csv
Base

## Aggregate Experiments Results

In [20]:
# CELL: aggregate_hybrid_experiments_results
import json
import ast
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

ROOT = Path("artifacts") / "hybrid_experiments_summary"
ROOT.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------------------
# Helpers
# -------------------------------------------------------------------
def _safe_float(x):
    try:
        if x is None:
            return np.nan
        if isinstance(x, str) and x.strip() == "":
            return np.nan
        return float(x)
    except Exception:
        return np.nan


def _safe_dict(x):
    """Parse dict-like strings from CSV cells into real dicts when possible."""
    if isinstance(x, dict):
        return x
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    if isinstance(x, str):
        s = x.strip()
        if s in {"", "None", "nan", "NaN"}:
            return None
        # try JSON first
        try:
            v = json.loads(s)
            if isinstance(v, dict):
                return v
        except Exception:
            pass
        # then Python literal
        try:
            v = ast.literal_eval(s)
            if isinstance(v, dict):
                return v
        except Exception:
            pass
    return None


def _safe_list(x):
    """Parse list-like strings from CSV cells into real lists when possible."""
    if isinstance(x, list):
        return x
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    if isinstance(x, str):
        s = x.strip()
        if s in {"", "None", "nan", "NaN"}:
            return None
        try:
            v = ast.literal_eval(s)
            if isinstance(v, list):
                return v
        except Exception:
            pass
    return x


def _pick_first(d, keys):
    if not isinstance(d, dict):
        return None
    for k in keys:
        if k in d:
            return d[k]
    return None


def _load_summary(path: Path):
    """
    Load one evaluation summary.
    Prefer JSON; fall back to CSV.
    Returns (data_dict_or_row, source_path_str).
    """
    try:
        if path.suffix.lower() == ".json":
            with open(path, "r") as f:
                data = json.load(f)
            return data, str(path)

        if path.suffix.lower() == ".csv":
            df = pd.read_csv(path)
            if df.empty:
                return None, str(path)
            row = df.iloc[0].to_dict()

            # Try to parse common dict/list fields
            for k, v in list(row.items()):
                if isinstance(v, str):
                    parsed = _safe_dict(v)
                    if parsed is not None:
                        row[k] = parsed
                        continue
                    parsed_list = _safe_list(v)
                    if parsed_list is not None:
                        row[k] = parsed_list
            return row, str(path)

    except Exception as e:
        print(f"Failed to read {path}: {e}")
        return None, str(path)

    return None, str(path)


def _extract_metric_block(data, block_name):
    """
    Accepts several possible nesting styles:
      - baseline / augmented / dRMSE
      - baseline_raw / augmented_raw / dRMSE_raw
      - baseline_metrics / augmented_metrics
    """
    if not isinstance(data, dict):
        return None

    candidates = [
        block_name,
        f"{block_name}_raw",
        f"{block_name}_metrics",
    ]
    for key in candidates:
        if key in data:
            val = data[key]
            if isinstance(val, dict):
                return val
    return None


def _flatten_one_summary(data, src, ds_name):
    """
    Convert one dataset summary dict into a flat row with stable ASCII columns.
    Includes RMSE, RAE, RAE_score, and delta versions when available.
    """
    row = {
        "dataset": ds_name,
        "source_file": src,
        "timestamp": np.nan,
        "timestamp_iso": None,
        "n_samples_train": np.nan,
        "n_features": np.nan,
        "best_te": np.nan,
        "best_subset": None,
        "LR_base_RMSE": np.nan,
        "Ridge_base_RMSE": np.nan,
        "XGB_base_RMSE": np.nan,
        "LR_aug_RMSE": np.nan,
        "Ridge_aug_RMSE": np.nan,
        "XGB_aug_RMSE": np.nan,
        "LR_dRMSE": np.nan,
        "Ridge_dRMSE": np.nan,
        "XGB_dRMSE": np.nan,
        "LR_base_RAE": np.nan,
        "Ridge_base_RAE": np.nan,
        "XGB_base_RAE": np.nan,
        "LR_aug_RAE": np.nan,
        "Ridge_aug_RAE": np.nan,
        "XGB_aug_RAE": np.nan,
        "LR_dRAE": np.nan,
        "Ridge_dRAE": np.nan,
        "XGB_dRAE": np.nan,
        "LR_base_RAE_score": np.nan,
        "Ridge_base_RAE_score": np.nan,
        "XGB_base_RAE_score": np.nan,
        "LR_aug_RAE_score": np.nan,
        "Ridge_aug_RAE_score": np.nan,
        "XGB_aug_RAE_score": np.nan,
        "LR_dRAE_score": np.nan,
        "Ridge_dRAE_score": np.nan,
        "XGB_dRAE_score": np.nan,
        "mean_base_RMSE": np.nan,
        "mean_aug_RMSE": np.nan,
        "mean_dRMSE": np.nan,
        "mean_base_RAE": np.nan,
        "mean_aug_RAE": np.nan,
        "mean_dRAE": np.nan,
        "mean_base_RAE_score": np.nan,
        "mean_aug_RAE_score": np.nan,
        "mean_dRAE_score": np.nan,
        "baseline_raw": None,
        "augmented_raw": None,
        "dRMSE_raw": None,
        "dRAE_raw": None,
        "dRAE_score_raw": None,
    }

    if not isinstance(data, dict):
        return row

    # top-level metadata
    for k in ("timestamp", "n_samples_train", "n_features", "best_te"):
        if k in data:
            row[k] = _safe_float(data[k])

    if "best_subset" in data:
        row["best_subset"] = data["best_subset"]

    # pull nested blocks in a tolerant way
    baseline = _extract_metric_block(data, "baseline")
    augmented = _extract_metric_block(data, "augmented")
    drmse = _extract_metric_block(data, "dRMSE")
    drae = _extract_metric_block(data, "dRAE")
    drae_score = _extract_metric_block(data, "dRAE_score")

    # fallbacks for older naming
    if baseline is None:
        baseline = data.get("baseline")
    if augmented is None:
        augmented = data.get("augmented")
    if drmse is None:
        drmse = data.get("dRMSE")
    if drae is None:
        drae = data.get("dRAE")
    if drae_score is None:
        drae_score = data.get("dRAE_score")

    row["baseline_raw"] = baseline
    row["augmented_raw"] = augmented
    row["dRMSE_raw"] = drmse
    row["dRAE_raw"] = drae
    row["dRAE_score_raw"] = drae_score

    algos = ("LR", "Ridge", "XGB")

    # baseline / augmented RMSE, RAE, RAE_score
    for algo in algos:
        # RMSE
        base_key = f"{algo}_RMSE"
        if isinstance(baseline, dict) and base_key in baseline:
            row[f"{algo}_base_RMSE"] = _safe_float(baseline[base_key])
        if isinstance(augmented, dict) and base_key in augmented:
            row[f"{algo}_aug_RMSE"] = _safe_float(augmented[base_key])

        # RAE raw
        base_key = f"{algo}_RAE"
        if isinstance(baseline, dict) and base_key in baseline:
            row[f"{algo}_base_RAE"] = _safe_float(baseline[base_key])
        if isinstance(augmented, dict) and base_key in augmented:
            row[f"{algo}_aug_RAE"] = _safe_float(augmented[base_key])

        # RAE score
        base_key = f"{algo}_RAE_score"
        if isinstance(baseline, dict) and base_key in baseline:
            row[f"{algo}_base_RAE_score"] = _safe_float(baseline[base_key])
        if isinstance(augmented, dict) and base_key in augmented:
            row[f"{algo}_aug_RAE_score"] = _safe_float(augmented[base_key])

    # deltas from explicit blocks if present
    if isinstance(drmse, dict):
        for algo in algos:
            k = f"{algo}_dRMSE"
            if k in drmse:
                row[f"{algo}_dRMSE"] = _safe_float(drmse[k])

    if isinstance(drae, dict):
        for algo in algos:
            k = f"{algo}_dRAE"
            if k in drae:
                row[f"{algo}_dRAE"] = _safe_float(drae[k])

    if isinstance(drae_score, dict):
        for algo in algos:
            k = f"{algo}_dRAE_score"
            if k in drae_score:
                row[f"{algo}_dRAE_score"] = _safe_float(drae_score[k])

    # infer missing RAE scores from raw RAE when needed
    for algo in algos:
        b_rae = row.get(f"{algo}_base_RAE")
        a_rae = row.get(f"{algo}_aug_RAE")
        b_score = row.get(f"{algo}_base_RAE_score")
        a_score = row.get(f"{algo}_aug_RAE_score")

        if pd.isna(b_score) and pd.notna(b_rae):
            row[f"{algo}_base_RAE_score"] = 1.0 - float(b_rae)
        if pd.isna(a_score) and pd.notna(a_rae):
            row[f"{algo}_aug_RAE_score"] = 1.0 - float(a_rae)

    # compute missing deltas from base/aug
    for algo in algos:
        b = row.get(f"{algo}_base_RMSE")
        a = row.get(f"{algo}_aug_RMSE")
        if pd.isna(row.get(f"{algo}_dRMSE")) and pd.notna(b) and pd.notna(a):
            row[f"{algo}_dRMSE"] = float(a) - float(b)

        b = row.get(f"{algo}_base_RAE")
        a = row.get(f"{algo}_aug_RAE")
        if pd.isna(row.get(f"{algo}_dRAE")) and pd.notna(b) and pd.notna(a):
            row[f"{algo}_dRAE"] = float(a) - float(b)

        b = row.get(f"{algo}_base_RAE_score")
        a = row.get(f"{algo}_aug_RAE_score")
        if pd.isna(row.get(f"{algo}_dRAE_score")) and pd.notna(b) and pd.notna(a):
            row[f"{algo}_dRAE_score"] = float(a) - float(b)

    # means across models
    rmse_base_vals = [row.get(f"{a}_base_RMSE") for a in algos if pd.notna(row.get(f"{a}_base_RMSE"))]
    rmse_aug_vals = [row.get(f"{a}_aug_RMSE") for a in algos if pd.notna(row.get(f"{a}_aug_RMSE"))]
    rmse_d_vals = [row.get(f"{a}_dRMSE") for a in algos if pd.notna(row.get(f"{a}_dRMSE"))]

    rae_base_vals = [row.get(f"{a}_base_RAE") for a in algos if pd.notna(row.get(f"{a}_base_RAE"))]
    rae_aug_vals = [row.get(f"{a}_aug_RAE") for a in algos if pd.notna(row.get(f"{a}_aug_RAE"))]
    rae_d_vals = [row.get(f"{a}_dRAE") for a in algos if pd.notna(row.get(f"{a}_dRAE"))]

    score_base_vals = [row.get(f"{a}_base_RAE_score") for a in algos if pd.notna(row.get(f"{a}_base_RAE_score"))]
    score_aug_vals = [row.get(f"{a}_aug_RAE_score") for a in algos if pd.notna(row.get(f"{a}_aug_RAE_score"))]
    score_d_vals = [row.get(f"{a}_dRAE_score") for a in algos if pd.notna(row.get(f"{a}_dRAE_score"))]

    if rmse_base_vals:
        row["mean_base_RMSE"] = float(np.nanmean(rmse_base_vals))
    if rmse_aug_vals:
        row["mean_aug_RMSE"] = float(np.nanmean(rmse_aug_vals))
    if rmse_d_vals:
        row["mean_dRMSE"] = float(np.nanmean(rmse_d_vals))

    if rae_base_vals:
        row["mean_base_RAE"] = float(np.nanmean(rae_base_vals))
    if rae_aug_vals:
        row["mean_aug_RAE"] = float(np.nanmean(rae_aug_vals))
    if rae_d_vals:
        row["mean_dRAE"] = float(np.nanmean(rae_d_vals))

    if score_base_vals:
        row["mean_base_RAE_score"] = float(np.nanmean(score_base_vals))
    if score_aug_vals:
        row["mean_aug_RAE_score"] = float(np.nanmean(score_aug_vals))
    if score_d_vals:
        row["mean_dRAE_score"] = float(np.nanmean(score_d_vals))

    # timestamp iso
    if pd.notna(row["timestamp"]):
        try:
            row["timestamp_iso"] = datetime.utcfromtimestamp(float(row["timestamp"])).isoformat()
        except Exception:
            row["timestamp_iso"] = None
    else:
        row["timestamp_iso"] = None

    # best_subset serialization if needed
    if isinstance(row["best_subset"], list):
        row["best_subset"] = json.dumps(row["best_subset"])

    return row


# -------------------------------------------------------------------
# Discover summaries
# -------------------------------------------------------------------
json_files = sorted(ROOT.glob("*_evaluation_summary.json"))
csv_files = sorted(ROOT.glob("*_evaluation_summary.csv"))

candidates = {}
for p in json_files + csv_files:
    ds_name = p.stem.replace("_evaluation_summary", "")
    candidates.setdefault(ds_name, []).append(p)

if not candidates:
    print("No evaluation_summary files found in", ROOT)
else:
    print("Found datasets:", ", ".join(sorted(candidates.keys())))

# -------------------------------------------------------------------
# Load, flatten, and save
# -------------------------------------------------------------------
rows = []
for ds_name, paths in sorted(candidates.items()):
    chosen = next((p for p in paths if p.suffix.lower() == ".json"), None)
    if chosen is None:
        chosen = next((p for p in paths if p.suffix.lower() == ".csv"), None)
    if chosen is None:
        continue

    data, src = _load_summary(chosen)
    if data is None:
        print("Skipping", ds_name, "couldn't load:", chosen)
        continue

    flat = _flatten_one_summary(data, src, ds_name)
    rows.append(flat)

if not rows:
    print("No valid summaries loaded.")
else:
    master_df = pd.DataFrame(rows)

    # Stable column order for CSV / LaTeX use
    cols_order = [
        "dataset", "n_samples_train", "n_features", "best_te", "best_subset",
        "LR_base_RMSE", "LR_aug_RMSE", "LR_dRMSE",
        "Ridge_base_RMSE", "Ridge_aug_RMSE", "Ridge_dRMSE",
        "XGB_base_RMSE", "XGB_aug_RMSE", "XGB_dRMSE",
        "LR_base_RAE", "LR_aug_RAE", "LR_dRAE",
        "Ridge_base_RAE", "Ridge_aug_RAE", "Ridge_dRAE",
        "XGB_base_RAE", "XGB_aug_RAE", "XGB_dRAE",
        "LR_base_RAE_score", "LR_aug_RAE_score", "LR_dRAE_score",
        "Ridge_base_RAE_score", "Ridge_aug_RAE_score", "Ridge_dRAE_score",
        "XGB_base_RAE_score", "XGB_aug_RAE_score", "XGB_dRAE_score",
        "mean_base_RMSE", "mean_aug_RMSE", "mean_dRMSE",
        "mean_base_RAE", "mean_aug_RAE", "mean_dRAE",
        "mean_base_RAE_score", "mean_aug_RAE_score", "mean_dRAE_score",
        "timestamp_iso", "source_file",
    ]
    existing_cols = [c for c in cols_order if c in master_df.columns]
    master_df = master_df[existing_cols].copy()

    # sort for convenience
    if "dataset" in master_df.columns:
        master_df = master_df.sort_values("dataset").reset_index(drop=True)

    # Save outputs
    master_csv = ROOT / "master_hybrid_summary_cleaned.csv"
    master_json = ROOT / "master_hybrid_summary_cleaned.json"
    master_df.to_csv(master_csv, index=False)
    with open(master_json, "w") as f:
        json.dump(rows, f, indent=2, default=str)

    print("Saved master CSV ->", master_csv)
    print("Saved master JSON ->", master_json)

    # Optional: a compact RAE-focused CSV for the comparison table
    rae_cols = [
        "dataset", "n_samples_train", "n_features",
        "Ridge_base_RAE_score", "Ridge_aug_RAE_score", "Ridge_dRAE_score",
        "mean_base_RAE_score", "mean_aug_RAE_score", "mean_dRAE_score",
    ]
    rae_cols = [c for c in rae_cols if c in master_df.columns]
    rae_df = master_df[rae_cols].copy()
    rae_csv = ROOT / "master_hybrid_rae_summary.csv"
    rae_df.to_csv(rae_csv, index=False)
    print("Saved RAE-focused CSV ->", rae_csv)

    print("\nMaster summary (key cols):")
    key_cols = [c for c in ["dataset", "n_samples_train", "n_features", "best_te", "mean_dRMSE", "mean_dRAE_score"] if c in master_df.columns]
    print(master_df[key_cols].to_string(index=False))

Found datasets: airfoil, bike_hour, boston_housing, openml_586, openml_589, openml_607, openml_616, openml_618, openml_620, openml_637, wine_red, wine_white
Saved master CSV -> artifacts/hybrid_experiments_summary/master_hybrid_summary_cleaned.csv
Saved master JSON -> artifacts/hybrid_experiments_summary/master_hybrid_summary_cleaned.json
Saved RAE-focused CSV -> artifacts/hybrid_experiments_summary/master_hybrid_rae_summary.csv

Master summary (key cols):
       dataset  n_samples_train  n_features    best_te  mean_dRMSE  mean_dRAE_score
       airfoil           1202.0         5.0   6.094817   -0.066119         0.005892
     bike_hour              NaN         NaN 142.867539  -12.222760         0.065820
boston_housing            404.0        13.0   4.581586   -0.021359         0.023135
    openml_586            800.0        25.0   0.783443   -0.213324         0.248432
    openml_589            800.0        25.0   0.585082   -0.211699         0.230345
    openml_607            800.0    

/tmp/ipykernel_573713/939103047.py:345: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  row["timestamp_iso"] = datetime.utcfromtimestamp(float(row["timestamp"])).isoformat()


In [21]:
import pandas as pd
from pathlib import Path

ROOT = Path("artifacts") / "hybrid_experiments_summary"

# IMPORTANT: updated filename from previous cell
df = pd.read_csv(ROOT / "master_hybrid_summary_cleaned.csv")

# -----------------------------
# Remove bike_hour (not in paper)
# -----------------------------
df = df[df["dataset"] != "bike_hour"].copy()

# -----------------------------
# Keep relevant columns
# -----------------------------
cols = [
    "dataset",
    "n_samples_train",
    "n_features",
    "LR_dRMSE",
    "Ridge_dRMSE",
    "XGB_dRMSE"
]

df_clean = df[cols].copy()

# -----------------------------
# Compute mean ΔRMSE
# -----------------------------
df_clean["mean_dRMSE"] = df_clean[
    ["LR_dRMSE", "Ridge_dRMSE", "XGB_dRMSE"]
].mean(axis=1)

# -----------------------------
# Round values
# -----------------------------
for col in ["LR_dRMSE", "Ridge_dRMSE", "XGB_dRMSE", "mean_dRMSE"]:
    df_clean[col] = df_clean[col].round(4)

# -----------------------------
# Rename columns for LaTeX
# -----------------------------
df_clean.columns = [
    "Dataset",
    "n train",
    "n features",
    "LR dRMSE",
    "Ridge dRMSE",
    "XGB dRMSE",
    "Mean dRMSE"
]

# -----------------------------
# Dataset name mapping (UPDATED)
# -----------------------------
name_map = {
    "wine_red": "Wine (Red)",
    "wine_white": "Wine (White)",
    "boston_housing": "Housing Boston",
    "airfoil": "Airfoil",
    "openml_586": "Openml-586",
    "openml_589": "Openml-589",
    "openml_607": "Openml-607",
    "openml_616": "Openml-616",
    "openml_618": "Openml-618",
    "openml_620": "Openml-620",
    "openml_637": "Openml-637"
}

df_clean["Dataset"] = df_clean["Dataset"].replace(name_map)

# -----------------------------
# Sort (nice for paper)
# -----------------------------
df_clean = df_clean.sort_values("Dataset").reset_index(drop=True)

# -----------------------------
# Save final table
# -----------------------------
df_clean.to_csv(ROOT / "table2_master_results.csv", index=False)

df_clean

,Dataset,n train,n features,LR dRMSE,Ridge dRMSE,XGB dRMSE,Mean dRMSE
0,Airfoil,1202.0,5.0,-0.1082,-0.1086,0.0184,-0.0661
1,Housing Boston,404.0,13.0,-0.4179,-0.4180,0.7718,-0.0214
2,Openml-586,800.0,25.0,-0.3139,-0.3137,-0.0124,-0.2133
3,Openml-589,800.0,25.0,-0.3296,-0.3292,0.0237,-0.2117
4,Openml-607,800.0,50.0,-0.2571,-0.2570,0.0404,-0.1579
5,Openml-616,400.0,50.0,-0.2801,-0.2807,-0.0075,-0.1895
6,Openml-618,800.0,50.0,-0.3218,-0.3221,0.0166,-0.2091
7,Openml-620,800.0,25.0,-0.3097,-0.3095,0.0160,-0.2011
8,Openml-637,400.0,50.0,-0.3005,-0.3009,-0.0108,-0.2041
9,Wine (Red),1279.0,11.0,-0.0002,-0.0002,0.0011,0.0002


## Visualizations

In [22]:
# === Overall mean ΔRMSE plot (same style) ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path("artifacts") / "hybrid_experiments_summary"
csv_path = ROOT / "table2_master_results.csv"
PLOT_DIR = ROOT / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(csv_path)
df.columns = [c.strip() for c in df.columns]

# Clean numeric columns
for col in ["Mean dRMSE"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Sort by mean improvement (most negative first)
df = df.sort_values("Mean dRMSE").reset_index(drop=True)

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.titlesize": 15,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

fig, ax = plt.subplots(figsize=(13.5, 6.2), dpi=300)

bars = ax.bar(
    df["Dataset"],
    df["Mean dRMSE"],
    color="#4a6879",
    edgecolor="black",
    linewidth=0.9,
    width=0.65,
    zorder=3
)

ax.axhline(0, linestyle="--", linewidth=1.0, color="black", zorder=2)
ax.set_ylabel("Mean ΔRMSE")
ax.set_title("IGS-SR — Overall Mean ΔRMSE Across Datasets", pad=10)
ax.tick_params(axis="x", rotation=30, length=0)
ax.grid(False)

vals = df["Mean dRMSE"].to_numpy()
min_val = np.nanmin(vals)
max_val = np.nanmax(vals)
span = max(abs(min_val), abs(max_val), 1e-9)
margin = 0.22 * span
ax.set_ylim(min(min_val - margin, -0.02 * span), max(max_val + margin, 0.02 * span))

ymin, ymax = ax.get_ylim()
offset = 0.03 * (ymax - ymin)
for bar, v in zip(bars, vals):
    x = bar.get_x() + bar.get_width() / 2
    y = v + offset if v >= 0 else v - offset
    ax.text(
        x, y,
        f"{v:.4f}",
        ha="center",
        va="bottom" if v >= 0 else "top",
        fontsize=9,
        fontweight="bold"
    )

fig.tight_layout()
fig.savefig(PLOT_DIR / "master_mean_dRMSE.png", dpi=300, bbox_inches="tight")
fig.savefig(PLOT_DIR / "master_mean_dRMSE.pdf", bbox_inches="tight")
plt.close(fig)

print("Saved overall mean ΔRMSE plot to:", PLOT_DIR)

Saved overall mean ΔRMSE plot to: artifacts/hybrid_experiments_summary/plots


In [24]:
# === Overall mean ΔRMSE plot (same style) ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path("artifacts") / "hybrid_experiments_summary"
csv_path = ROOT / "table2_master_results.csv"
PLOT_DIR = ROOT / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(csv_path)
df.columns = [c.strip() for c in df.columns]

# Clean numeric columns
for col in ["Mean dRMSE"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Sort by mean improvement (most negative first)
df = df.sort_values("Mean dRMSE").reset_index(drop=True)

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.titlesize": 15,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

fig, ax = plt.subplots(figsize=(13.5, 6.2), dpi=300)

bars = ax.bar(
    df["Dataset"],
    df["Mean dRMSE"],
    color="#4a6879",
    edgecolor="black",
    linewidth=0.9,
    width=0.65,
    zorder=3
)

ax.axhline(0, linestyle="--", linewidth=1.0, color="black", zorder=2)
ax.set_ylabel("Mean ΔRMSE")
ax.set_title("IGS-SR — Overall Mean ΔRMSE Across Datasets", pad=10)
ax.tick_params(axis="x", rotation=30, length=0)
ax.grid(False)

vals = df["Mean dRMSE"].to_numpy()
min_val = np.nanmin(vals)
max_val = np.nanmax(vals)
span = max(abs(min_val), abs(max_val), 1e-9)
margin = 0.22 * span
ax.set_ylim(min(min_val - margin, -0.02 * span), max(max_val + margin, 0.02 * span))

ymin, ymax = ax.get_ylim()
offset = 0.03 * (ymax - ymin)
for bar, v in zip(bars, vals):
    x = bar.get_x() + bar.get_width() / 2
    y = v + offset if v >= 0 else v - offset
    ax.text(
        x, y,
        f"{v:.4f}",
        ha="center",
        va="bottom" if v >= 0 else "top",
        fontsize=9,
        fontweight="bold"
    )

fig.tight_layout()
fig.savefig(PLOT_DIR / "master_mean_dRMSE.png", dpi=300, bbox_inches="tight")
fig.savefig(PLOT_DIR / "master_mean_dRMSE.pdf", bbox_inches="tight")
plt.close(fig)

print("Saved overall mean ΔRMSE plot to:", PLOT_DIR)

Saved overall mean ΔRMSE plot to: artifacts/hybrid_experiments_summary/plots


In [25]:
# === Publication-style 2x2 representative breakdown plots ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path("artifacts") / "hybrid_experiments_summary"
csv_path = ROOT / "table2_master_results.csv"
PLOT_DIR = ROOT / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(csv_path)
df.columns = [c.strip() for c in df.columns]

# Clean numeric columns
for col in ["n train", "n features", "LR dRMSE", "Ridge dRMSE", "XGB dRMSE", "Mean dRMSE"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Representative datasets for the paper
datasets = ["Housing Boston", "Airfoil", "Openml-589", "Openml-637"]

# Keep only available rows
df_sel = df[df["Dataset"].isin(datasets)].copy()

# Enforce ordering
df_sel["Dataset"] = pd.Categorical(df_sel["Dataset"], categories=datasets, ordered=True)
df_sel = df_sel.sort_values("Dataset").reset_index(drop=True)

# Reference-like pastel palette
bar_colors = ["#fbfbf4", "#d4deb9", "#9aafa1", "#4a6879"]  # baseline-like pale -> dark teal
bar_edge = "black"

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.titlesize": 15,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

fig, axes = plt.subplots(2, 2, figsize=(13.5, 8.2), dpi=300)
axes = axes.flatten()

panel_labels = ["a", "b", "c", "d"]
metric_labels = ["LR", "Ridge", "XGB", "Mean"]
metric_cols = ["LR dRMSE", "Ridge dRMSE", "XGB dRMSE", "Mean dRMSE"]

for ax, (_, row), p in zip(axes, df_sel.iterrows(), panel_labels):
    vals = [row[c] for c in metric_cols]

    bars = ax.bar(
        metric_labels,
        vals,
        color=bar_colors,
        edgecolor=bar_edge,
        linewidth=0.9,
        width=0.55,
        zorder=3
    )

    # Zero reference line
    ax.axhline(0, linestyle="--", linewidth=1.0, color="black", zorder=2)

    # Clean styling
    ax.spines["top"].set_visible(True)
    ax.spines["right"].set_visible(True)
    ax.tick_params(axis="x", length=0)
    ax.grid(False)

    # Dynamic limits with zero always visible
    min_val = np.nanmin(vals)
    max_val = np.nanmax(vals)
    span = max(abs(min_val), abs(max_val), 1e-9)
    margin = 0.22 * span
    ax.set_ylim(min(min_val - margin, -0.02 * span), max(max_val + margin, 0.02 * span))

    ax.set_ylabel("ΔRMSE")

    # Panel label + dataset title underneath, matching the reference style
    ax.text(
        0.5, -0.25,
        f"({p}) {row['Dataset']}",
        transform=ax.transAxes,
        ha="center",
        va="top",
        fontsize=15
    )

    # Annotate values
    ymin, ymax = ax.get_ylim()
    offset = 0.03 * (ymax - ymin)
    for bar, v in zip(bars, vals):
        x = bar.get_x() + bar.get_width() / 2
        if np.isfinite(v):
            y = v + offset if v >= 0 else v - offset
            ax.text(
                x, y,
                f"{v:.4f}",
                ha="center",
                va="bottom" if v >= 0 else "top",
                fontsize=9,
                fontweight="bold"
            )

fig.subplots_adjust(wspace=0.35, hspace=0.60, bottom=0.12)
fig.savefig(PLOT_DIR / "representative_hybrid_breakdowns.png", dpi=300, bbox_inches="tight")
fig.savefig(PLOT_DIR / "representative_hybrid_breakdowns.pdf", bbox_inches="tight")
plt.close(fig)

print("Saved representative 2x2 plots to:", PLOT_DIR)

Saved representative 2x2 plots to: artifacts/hybrid_experiments_summary/plots
